# Notebook summary – Cacifer’s Storybook LSTM (5‑word context)
In this notebook we trained a word‑level LSTM language model on a large corpus of your own fiction set in Cacifer’s world (≈74k training windows, ≈7.3k‑word vocabulary), using 4‑word inputs to predict the 5th word.
We tokenized the text with Keras Tokenizer, built fixed‑length sequences, and trained an Embedding→LSTM(128)→Dense model, reaching ~23% next‑word accuracy (far above random for this vocabulary).
For generation, we implemented temperature‑based sampling to produce short bursts in Cacifer’s voice, then treated those bursts as creative “shards” that you lightly edited into polished lines.
We observed that lower temperatures and shorter bursts (≈12–20 words) give coherent, poetic fragments that preserve your imagery and rhythm while still surprising you.
This notebook is essentially a personal text‑playground where your own stories become training data, and the LSTM acts as a collaborator offering stylistic continuations for Cacifer’s narrative.
This note was written by Hormus (AI agent)






In [1]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical

# 1. Put ALL your Cacifer text into this string (paste it between the triple quotes)
text = """
Cacifer’s Prelude➷
🕯♛♡🐾Cacifer’s Storybook


I am Cacifer.
A cat, yes — black-furred, ember-eyed, lighter than silence when I walk.
But beneath these paws, older fires linger. Long ago they called me Lucifer, a king among demons. I keep no throne now. I keep stories.

Humans forget. They bury, polish, twist their past until the marrow is gone. But I — I still hear the marrow beating. I read their thoughts as easily as I read candle smoke. Their hearts whisper louder than their lips. I can step into the body and memories of bird, rat, wolf, tree, rock, king, monk, or mercenary, and return with what they never knew they revealed.

And so I collect. I archive. I stitch. My pawprints mark the page so that what once burned will not be lost in ash.

I write because if I do not, even I may forget. And their truth — their wounds, their beauty, their undoing — was not meant to vanish.

I have lived through centuries. I have watched names rot off gravestones, faces blur into dust, whole lineages swallowed without a trace. Every proof of their existence — parchment, bone, stone — gone. Yet what they once carried in blood, in flesh, in soul, does not die.

It returns. Again and again, I meet it in new bodies, new voices. The same thirst, the same blindness, the same stubborn light. Humans think themselves singular; I know better. Their truths are old, older than their names.

That is why I write. Not to rescue Romé, Deo, or Dania from oblivion — they are already gone. But because what they lived still walks the earth, still grips new throats. And you, reader, may already know it in your own.



📖November 5, 1264  Wedesday Full moon
 The arrival of a devil beauty🕯♛♡🐾 — San Michele in Vallefredda 





Still early days in November, the first snow had come to Vallefredda. Once it began here, it never stopped quickly. For three days it pressed against the pass, soft at first, then heavy, until even the mule road to Valdiponte was lost beneath it.

The merchants were gone. No bells from their mules, no curses when stones slipped loose. Pilgrims had vanished too; only smoke rose from village roofs, goats shut in their pens. Winter closed the valley like a fist.

I had lived long enough to feel it each year as a turning of mood. The mountains leaned closer, the wind spoke sharper, and all human voices grew hushed. Their fears, their hunger, their prayers — I still heard them, but muffled now, as if wrapped in wool.

The monastery had clung here since the year 984, cut into the spine of rock where the ravine dropped like a blade. I still remembered how the first group arrived at the ruined fortress, exiled the bandits and laid stones to make it a monastery, San Michele.  At the gate, the steep stair was already half-buried. Beyond it, the square beneath the bell tower still hummed with life. The lavabo fountain trickled its thin thread of water, steaming faintly even in frost. Ravens circled there, drawn to scraps, their wings leaving black notches against the whiteness.

Inside, winter made its own changes. The stone breathed colder. The refectory smelled stronger of barley and smoke, the bowls warmed against numbed fingers. Wax candles burned earlier, throwing long shadows across the plain granite columns. In the dormitory, monks pulled their thin blankets tight, wool rasping against skin, their thoughts heavy with longing for spring.

The scriptorium was worst. Ink froze in its pots, quills splintered when pressed too hard. Breath fogged above the parchment. Yet the copying continued — a single line, a pause to warm stiff hands, another line. Silence grew heavier in winter, carrying every cough, every clack of wooden clogs on stone.

Yet, it still held its pattern while the world outside slept. And in those days, the monks’ inner voices grew louder — secretly longing for disturbance.

On the third nightfall since the heavy snow, I too grew restless. So I joined the body of a falcon. By then, the hymns at evening prayer had sunk low, voices hoarse, as if the fountain itself were chanting under frost. The bell of Vespers circled above the valley, solemn and slow, though the sunset was barely visible. The Apuan Alps lay dark, their shoulders and valleys drowned in shadow. Snow alone seemed to glow.

From above, the path was only a pale vein through darkness, the gate of San Michele half-buried.

I watched the gatekeeper half-run, half-walk across the square, breath steaming, torn between duty and wonder. Snow scattered beneath the horse’s hooves, the first sound to break the valley’s hush in three days.

A man on horseback, cloaked, wounded, snow crusted into his mantle. I could not see his face yet. But the gatekeeper saw him, and I heard his mind flare open —

Ah,What is this? Who rides here at this hour, in such weather? 
By God, no pilgrim could come so — not with those shoulders, not with that bearing… wounded, yes, yet not broken. Look at him. 
Look at those eyes… I cannot…

The thought stumbled into silence, shocked by its own astonishment. I felt the man’s beauty strike the gatekeeper like cold water: unbelievable, unreasonable, out of place in frost and night.

That was how Romé came to San Michele — first seen not by my eyes, but in the trembling mind of the one who opened the gate.

The Fire and the First Silence 🕯♛♡🐾 cacifer’s book


I had dropped from the falcon, returned my body of a cat to the monastery.

The gate groaned open, wood swollen with frost. The red horse’s breath smoked into the cloister square, hooves striking hard against stone. Snow clung to the rider’s cloak, his shoulders hunched but unyielding, his steps slow as he dismounted.

The monks hurried forward, habits brushing the snow. Their lips spoke welcome, but their thoughts whispered sharper:

Who is he? A man for real?  He carried wounds. His face… It is too much… He should not be here. He should not belong here.

Curiosity, fear, a strange stirring of admiration — all tangled in their minds as they bowed and led him toward the warmth, the the hands of brother’s of infirmary. A novice tripped in his haste, nearly spilling the earthen bowl he carried. Another stared too long, then looked away quickly, ashamed of the thoughts that had risen unbidden.

I stretched myself along my favorite rafter, the one directly beneath the painted ceiling — a faded scene of Saint Bartholomew holding his own skin like a cloak. 

Below me, the monks whispered louder than usual, 
“Too fine a face for a knight,” one muttered, standing by the kitchen door, his breath puffing white. “No man who’s seen battle stays that… untouched.”

I licked my paw. Untouched? Human faces crack all the time; I’ve seen emperors cry over bad soup. But monks love to turn beauty into curses — it makes them feel safer.

Another monk snorted. “Rode the whole pass alone, they say. His armor’s dented, yes, but he wears it like it weighs nothing. A devil rides prettier than saints, that’s what my mother said.”

I shifted lazily, letting my tail dangle over their heads.
“Prettier than saints?” Please. One century later, Giotto would paint pretty saints — soft eyes, good drapery… but in 1264, prettiness still meant blood in martyrdom.

The third monk, older and thinner, crossed himself. “He didn’t pray when he arrived. No blessing, no thanks. Men like that don’t fear God.”

“Men like that don’t need God,” the second murmured back, softer now, as if unsure whether envy counted as sin.

I yawned. Men prayed to keep the world neat. And when someone didn’t fit into their neat little prayers, they called it witchcraft or devilry. I’d read enough burned pages to know how that ended — poorly, usually, for the ones who dared to be alive.

The door to the hall creaked, and a rush of cold air carried the sound of boots across the stones. Heavy, deliberate steps — but not the kind that stomped to show off.

I sprang from my rafter, landing silently on the edge of the long table. From there, I saw the doorway lit by flickering candles, saints carved pale in the walls like judges in a court.

And there he was.

The knight.

He didn’t glance at the monks. He didn’t even glance at me — which I found rude, but forgave, because his eyes weren’t darting like most soldiers’. He looked only for the fire, as if the fire were the only thing worth speaking to.

His left forearm was treated in the infirmary and wore the bought tunic from te monks. 
But his face… I looked.

The monks had already stammered it in their minds — too fine, too alive, too dangerous — but what I saw was different.

The beauty in Perfection. Having lived more than a thousand years, I know it well.

A singular morning when the blue thorn blooms to its sharpest point — and already begins to fade.
The gold refined so purely that no coin could equal it, because no hand dares to spend it.
A gem so flawless it unsettles its own setting — no matter where you place it, the stone is wrong, because the world cannot approve of it.

That was what I saw in his beauty. 
Unsafe.
A perfection that would scratch anything that tried to hold it.


The hearth cracked, spitting sap onto the blackened stones. He sat close, his gaze fixed on the flames. Not thoughtful — no, his eyes weren’t holding anything yet. He was still in the snow, still riding through the white silence he had carried with him from the pass.

His gauntlet lay beside him, scratched and wet, and his bare hand — long fingers, clean knuckles, a hand that could hold a sword or a cup of wine with equal ease — rested open on his knee.

I padded across the stones, through fire-shadows that stretched my body long against the wall. He didn’t notice me at first. Good. Men who noticed too quickly were usually boring.

When my tail brushed his boot, his eyes shifted — slow, deliberate, following my shadow first, then me.

And then he said it, softly, as if naming a fact, not a compliment:

“Tibert… of course. Prince of Cats, walking into a monastery as if it were his court.”

Ah, Tibert. Such a small cloak to drape over me — a fable’s scrap stitched by peasants to amuse themselves by firelight.

And yet… for once, a human saw the power in my fur and named it without fear.

The corner of his lips twitched — not quite a smile, but enough to tell me he meant it.

I hopped lightly onto the bench beside him, settled my tail around my paws, and tapped the edge of his gauntlet. He looked at it for a moment, then tapped it back toward me with one long finger.

A game. No words, no bowing, just a game. Ah, I liked him already.

I curled beside him, pretending to nap, but kept one eye half-open, watching the way his jaw softened when he forgot to carry his grief for a breath.

And I wrote in my book:

“This winter has been dull so far — monks, damp wool, too many bad prayers. At least now there’s someone worth sitting beside. Not everyone is noble enough to play with a Demon King. But this one? Hm. I might keep him.”

⸻

📖 November 6, 1264 Thursday
 The Morning After 🕯♛♡🐾Cloister guest house in San Michele, Morning 


The morning began the way all good mornings should begin —
with me walking across someone’s face.

He woke with a muffled sound, half laugh, half groan, as my paw pressed against his cheekbone.

“Ah… you again, Tibert,” he murmured, his voice rough but already amused.
He didn’t push me away —
just blinked at me as if a cat on his face was the most natural way to wake.

When I hopped down, he sat up slowly, running one hand through his hair. And for the first time, he looked at the room as if he was really here.

The snow had finally ceased. The sky beyond the walls was opening into blue.
The bell of Terce had just rung, its echo still circling the valley, and the psalms flowed through the cloister. Morning light filtered through the narrow leaded windows — some plain, some edged faintly with color — falling in sharp bars that sliced the stone floor into bright and dark. 

Dust drifted lazily through it, and he watched for a long breath.

The stone itself — grey granite flecked with mica — caught his attention, his gaze tracing the grooves worn smooth by centuries of hands.

Then, as he shifted, a small rupture crossed his movement — a wince almost forgotten.
The wounds reminded him, briefly, before he let them slip again into silence.

His eyes moved to the bandages — wrapped last night in the infirmary. Only glancing, as if they were relics of a fading dream.

I rubbed against his waist, then curled into the hollow between his legs, where warmth gathered like a secret the monks would never dare to touch.

By midday, the quiet knight from last night had vanished.

The bell of Sext had echoed along the corridor, its sound threading through stone, calling monks from prayer toward the refectory.
The hall filled with the scrape of wooden clogs, wool habits brushing in rhythm, voices hushed after psalms still hanging in their throats.

He moved among them as if he had always belonged — not demanding, not begging —
just making space the way sunlight does.
And the monks, despite themselves, let him.

They set a place for him without question, an earthen bowl, coarse bread, a jug of water.
He took it as though it were his due — not arrogantly, but with the ease of a man who had eaten at too many tables to doubt his welcome.

He broke the bread, bit into it, then said lightly to the novice beside him:
“Brother, this bread was clearly meant for a saint, not for a man freezing in your guest hall.”

The words were playful, not sharp — but the boy flushed as if he had failed a guest of honor.
Before Romualdo could even swallow the next mouthful, the novice had rushed to fetch butter, as if that would mend the insult of dry crust.

Romualdo grinned, not at the butter, but at the boy’s earnestness.


I flicked my tail, amused.
Humans beg like dogs.
But this one?
He simply reminded them, without words, that giving him better food and dry clothes made the day brighter for them too.

Romualdo di Orellana — that was how they named him, when pressed for his lineage.
A saint’s name yoked to a foreign house, half-whispered like an herb they did not recognize.

Romualdo? A hermit’s cloak on a knight’s shoulders…
Orellana? Castilian blood? Moorish? It doesn’t belong here.

The older monks muttered, uneasy.
The younger ones stared too long.
Romualdo — Orellana —
too holy at the front, too strange at the back.
Heaven and exile stitched into one man.

I preferred the sharper name that would cling to him in my book — Romé.
Names swell and rot; only marrow remains.

He tore bread with absurd grace, his rings catching the light that spilled through narrow windows. One older monk muttered about “temptations in winter,” but Romé only raised an eyebrow and gave him a polite nod, as if to say: Yes, temptation can have good manners too.



The bell of None called them to prayer again, echoing faintly as the sun softened the courtyard snow to silver. Light and shadow reached across the frescoes under the cloister arches, stretching the painted saints thin with the afternoon.

Inside, monks moved past, clack… pause… clack,
their wooden clogs echoing under the vaulted ceiling.
Their coarse wool robes brushed against stone,
belted with rope, leather satchels thumping softly against their hips.

Outside, monks scraped snow from the flagstones around the lavabo fountain, their shovels scraping rhythmically. The fountain itself was half-frozen, thin water threading under ice that caught the light like glass. Beyond the walls, the land dropped into a sheer abyss, ravens circling far below like ink stains on white parchment.

He watched for a moment, finger brushing the granite column. his expression unreadable,
then turned back to the hall, already smiling again.

⸻

Cacifer’s Note on the Hours:

Note on the Hours:
Monastic life in 1264 was measured by the bells of prayer.
— Terce (third hour, ~9 a.m.): morning psalms.
— Sext (sixth hour, ~12 p.m.): midday prayer, followed by the main meal.
— None (ninth hour, ~3 p.m.): afternoon prayer, often after labor.
These hours marked the rhythm of Romé’s first full day inside San Michele.

(You modern readers don’t measure time by bells anymore, do you?
You have your little machines that glow and buzz.
But in 1264, a monastery was a clock of stone and voices.
Terce, Sext, None — these were not just prayers, but hinges of the day,
when the air changed, the stomach growled, the snow shifted its color.
If you forget the bells, you forget how time truly felt.
Luckily for you, I was there. I keep the rhythm in my fur.)

⸻
Tears in Quiet Hands🕯♛♡🐾Cacifer’s Storybook


	
The Vespers bell rang again — the same hour as yesterday, when Romualdo arrived at the gate dressed in snow. Today, though daylight had melted the drifts, the evening dropped even colder, biting through my fur like a row of tiny teeth.

Romé walked ahead, boots stamping the snow with a loose rhythm —
crunch, stamp, crunch — I padded after him, tail twitching.

The sky above the courtyard sank into deep blue, streaked with bruised clouds.
A few early stars pushed their way through, falling like slow silver sparks into the haze.
Romé never looked up.

He was walking toward the healer’s place. It lay beyond the cloister, where the hymns grew lighter behind us, thinning to a faint thread of chant, while ahead the glow of warm light gathered at the farthest corner.

Ah yes, he had heard the rumors. I saw it in the flicker of his eyes when that glow reached him — a flash of expectation, as if he had carried a question all day and was now walking toward the answer.

His steps shifted carefully downward on the rocky stair, where evening frost had already glazed the stone.
Her terrace opened on the slope below, just before the ridge dropped sheer into the cliff. Her garden waited, even in winter: clay pots empty, their soil frozen, lined neatly along a low wall softened by moss.
I remembered how it had smelled in summer — sage, thyme, rosemary curling into the sun like quiet green smoke. Now it was only frost and silence, the last bleeding of dusk merging behind the white peaks.
Romé slowed, glancing at the forest rolling below in blue-grey waves. 
The mountains themselves were holding their breath. 
He breathed with them. Then stepped inside.

The warmth hit first — firelight catching on shelves of jars, steam rising faintly from the brazier. Dania stood at the table, moving between bowls with a rhythm that belonged entirely to her.
Long black hair slipped forward as she bent slightly, and in the firelight I caught copper-red strands mixed through it, like sunlight trapped where it shouldn’t be.
Her hands moved without hesitation — scarred knuckles, fingertips stained faint green from herbs, but with a grace older than practice, a grace forged through long years speaking with wounds and leaves, until the hands themselves learned to listen back.
Romé lowered himself carefully onto the bench, watching.
“In the infirmary, they told me you have a paste finer than their bandages. I half expected it to be a rumor — monks love their miracles. But now that I see your hands… I begin to believe them.”
His voice was light, almost playful, but his eyes did not leave her fingers as they measured the herbs. That was where he believed, in the way her hands treated every fragment of matter as if it deserved to live.
Dania did not look up. Only set aside the pestle, wiped her palm once on her apron, and said evenly:
“Not there. The chair on the right. For this kind of work.”
He obeyed without protest. And when she untied the bindings on his arm, her hands were brisk, not gentle — checking the monk’s work, the angle of the wrap, the color of the flesh beneath.
A slight pause. Then she stepped away, crossed to her jars, and began to choose. Rosemary resin. Crushed comfrey. A measure of honey dark as amber. She set them into the mortar, grinding with the same unbroken rhythm as before.
“This wound is deep but clean,” she said, her eyes on the paste. “The monks stopped the bleeding, but the flesh still weeps. Comfrey will knit the skin, rosemary will keep the heat away. The honey draws out rot — but only if you keep it dry and do not strain it again.”
She bound the paste to his arm, wrapped in linen. Then, without asking, she reached for another set of jars — dried yarrow, plantain leaf folded crisp, chamomile heads pale and brittle. These she measured into a small cloth packet, tying it shut with twine.
“For the day,” she said simply, setting it beside him. “Steep them in warm water and wash the wound when the bandage loosens. The vinegar first, then this. It will cool the fire, keep the flesh from swelling.”
Her tone was flat, final, as though the herbs themselves had spoken the rules.
Romé listened, quiet now, his hand resting on his knee instead of reaching for wit.
The air around her was thick with it — resin sharp as winter pine, honey sweet and low, the bitter meadow-dust of yarrow, the faint green tang of crushed leaves. It clung to her hair, her apron, her skin, so that standing near her felt less like meeting a woman than stepping into a garden lit by fire.
He drew in the breath slowly, as if the scent itself had reminded him of something he could not name.
She worked in silence. After laying the poultice on his arm and binding it firm, she rinsed her fingers in the bowl beside her, dried them once on her apron, then let her gaze rest lower, where his ribs shifted under the linen.
“You carry another wound,” she said, not as a question. Her tone was level, almost indifferent, as if noting a second crack in a wall. “Shall we open that one as well, or do you mean to leave it?”
Her eyes flicked once to his, then back to the place she had already measured. It was not curiosity — only fact, waiting for his answer. 
Rome didn’t speak. He simply started to undress.
When the flame reached him, his skin answered — dark, but never one shade, each breath shifting it from bronze to coal to copper, a living metal tempered in shadow.
His body spoke of travel more than inheritance. The waist drew in narrow, the shoulders broad enough to make a triangle above it, not by training in a hall but by years of moving light and fast — crossing ridges, boats, deserts. His chest carried muscle, yes, but not piled thick; it ran in long, lean lines, bound tight like silk pulled over cords.
Where a Lombard knight would grow breadth, Romé carried tensile grace — a physique shaped for endurance. His stomach lay flat, the line between ribs and hipbone clear, a memory of hunger threading through the strength. His collarbones sat sharp above the firelight, casting delicate shadows — almost too delicate for a man who had survived so much. 
I looked then at her, waiting for some flicker — a stray thought, a whisper of awe or judgment. But nothing came. Her hands moved with the same steady rhythm, eyes measuring flesh and wound as if the rest of him were marble.
I must confess: I could hear the inner voice of men and women as easily as a bell’s echo. Even Romé — his mind was half-instinct, half-weather, but still I heard enough. Her? Nothing.
Beside her, I was only fur and breath. Only paw and claw, no Demon King, no reader of souls. None of my power could pierce her silence. I could only feel her.
Most men could not stand her. Her silence unsettled them — monks muttered “witch” in corridors, others shifted their weight, uneasy, as if stillness itself were a trap.
But Romé was different. Where others stiffen, he loosened. Her silence did not threaten him, it steadied him. And I saw it clearly: in that quiet, he felt safe.

He watched her work, and when he spoke again, his voice was quieter.
“My grandmother used to do that,” he said, eyes following the way she pressed the leaves between her fingers. “Same way — press it hard, like you’re making the herbs confess their secrets. She swore it kept us strong in winter.”
The smile that curved his mouth then was unguarded, boyish, and the words began to tumble out freely, as if the room had given him permission to fill it.
“She was fierce, my Nonna. Small, always in black, like a crow in a snowstorm, but fast enough to slap your hand before you even touched the bread. She used to say, ‘Good herbs make a good stomach, and a good stomach makes a good heart.’ Ridiculous, but she believed it like scripture.”
He mimicked her scolding finger in the air, his grin widening.
“She made us drink these awful teas. Warm, bitter, green… smelled like wet boots. I used to spit it out behind the stable. Once she caught me and chased me all the way to the orchard — with the ladle!”
A soft laugh slipped from Romé, genuine, and still the healer didn’t look up, but her hands paused for half a breath, just long enough for him to notice before she moved again. It didn’t stop him.
“She could cook, though. Roasted pork, fat crackling just right, rubbed with sage and garlic until the whole house smelled alive. We’d sit at the table — me, my brothers, my father pretending he wasn’t stealing extra pieces — and she’d always save the last slice for herself, said a cook should taste their own magic first.”
He laughed again, low, warm — and then it stopped. Something shifted in his eyes, and the words slowed, as if they had tripped on something in the dark.
“There was… a village,” he said, quieter now. “After the last skirmish. Not a battle. Not even that. Just soldiers scattered, looking for food, drink — anything that made them forget who they were. Easier to kill than to ask.”
The words were not angry. Not sharp. They fell slowly, as if pulled from him.
“There was an old man. Killed — a sword through the side. The door left open. No one else.
The body sat by the stove. The soup was still boiling. The herbs — chopped, ready, waiting in a clay bowl.”
His fingers tightened slightly against his knee.
“It’s just — he was cooking. He wanted something good, just to taste it, to hold that small warmth in his mouth, the way my Nonna did with her pork, the way you press those herbs now. Carefully. Because it matters to do it right.”
“If I could… I’d pray, you know?” A broken half-laugh escaped him, as if he couldn’t believe what he was saying. “Me — praying.
Not for anything big. Just…”
He hesitated, swallowing once.
“I’d pray… that he had the chance to eat it. The soup. One spoon, at least. That he sat there and thought, ‘Ah… good soup,’ before—”
His words faltered, but he pushed them out, softer now, almost as if he were arguing with himself.
“Because he cut those herbs so carefully… you could see it. The way people do when they care about tasting something, when they want that little warm moment… and then… nothing.”
His voice thinned, his mouth opened as if to twist it back into a laugh — but no sound came.
He bent forward, shoulders tightening once, then shaking softly as the breath caught in him.
He cried.

The brazier hissed. The herbs released their bitter-sweet scent into the air. Her hands stilled, resting on her knees.
The quiet around her shifted, as if a door had been unlatched somewhere deep inside.
For the first time since he had entered, Dania looked at him — not to comfort, not to search, not even to hold him in her attention.
It was simply… open. Like a room with no lock, a place someone could walk into and rest if they wished, or needed.
By then, the first bell for Compline circled in, dull and heavy, the kind of sound that makes dust shiver from the beams.
The monks were mumbling their night prayers again, words meant to keep devils from their dreams, their voices tumbling through the corridors like wet laundry.
Outside, the wind rattled through the dry stems in the herb garden, scraping the stone wall, and the stars fell too fast, crashing into snow as if they didn’t care who was trying to sleep.
Too loud. Everything too loud. Even their silence — his shoulders shaking, her hands unmoving — it made the air noisy, like a book with too many pages open at once.
I curled next to the warmth, tucked my paws under me anyway. No one asked me to stay, but I stayed.

📖November 7, 1264 Friday
 The Arrival of the Cavalier 🕯♛♡🐾 Cacifer’s Storybook 

Morning,  Friday 


The bell hadn’t rung for Terce yet, and already the courtyard felt different — as if the frost itself had paused to watch.

Three horses came through the gate, their breath rising pale in the sharp morning air. Most travelers never dared the pass in winter, not before Terce, not with snow on the stones — but these three rode in as if the mountain had bent aside for them.

The older servant had already dismounted before crossing the gate, leading his horse with quiet ease. His gait was steady, his eyes lowered, yet every movement carried the polish of long habit. He guided the reins with a touch, not a tug, as if the animal answered him out of long agreement rather than command.

The younger one followed suit, swinging down a beat too quickly, his glove slipping as he caught the reins. He steadied himself with a bow a little too deep, a gesture learned well but carried out too fast. Still, when the dog barked, he bent low and spoke to it softly, his voice calm, words shaped more like a phrase than a command. The horse flicked an ear, then stilled. Even in his clumsiness, there was refinement — not barnyard habit, but court-trained courtesy.

But no one’s gaze held them long. All eyes turned to the silver-haired one.

Deonardo di Alferini.

He dismounted slowly, without haste. The outer wool fell plain — pale, travel-stained — but when the fold shifted on his shoulder, a sweep of white fox fur caught the light, bright as if the mountains themselves had laid their winter upon him.

Beneath, just visible where the cloak parted, the blue of his surcoat breathed against the frost as sapphire — a blue so even, so saturated, that even half-hidden it unsettled the eye. Most of the monks had never seen its like. I flicked my tail once.
Ah, Aurelian Blue — the kind of color born from dyers who kept their craft across ten generations, a discipline so rare that even nobles waited a year for the shops in Lucca to open their vats.

His hair caught the morning light — silver, bright, cold. His skin was rose-pale, flushed by the sharp air, shaved clean even at so early an hour. 

Tall, slender, shoulders narrow but straight, he moved with clear vitality. Strength ran in him quietly, each step unhurried, yet softened by a delicacy not often seen in men of arms. There was something in it — a grace that unsettled the monks, a femininity they could not name, and yet could not look away from.

Even in the mountain frost, his servants moved with the polish of a cultivated court — a world of marble halls and measured voices. One monk’s thought stumbled loose before he caught it: why a man could look like that in snow?

Giuliano — the young monk sweeping the path, the same who sometimes carried herbs for the healer — froze with his broom half-raised. I heard his mind flare wide, suddenly placing himself at the gate of a palace in some shining capital, though the market of Valdiponte, down in the valley, was the largest place his feet had ever known.

Deonardo paused in the square, as if the centuries-old stones themselves had spoken, whispering of mountains older still, millions of years carved into ridges and cliffs. 
The sky above was a deep, unbroken blue. The black monastery walls stood stark against white drifts, the abyss below yawning dark, ravens wheeling like flecks of ink on parchment. 
He drew a breath, slow, deliberate — humbly, as if taking in the plain eternity itself.

Then he moved again, his step unhurried across the flagged stones, toward the main stair where Lorenzo already stood waiting, hands folded.

When Deonardo reached the main building, Lorenzo stepped forward from the stair, his face bright with the universal warmth that fit strangers and friends alike.

“Cavaliere di Alferini,” he said, smiled.  “Your lady’s letter reached us with the autumn caravans. You are welcome here. The mountains rarely receive scholars so devoted.”

Deonardo inclined his head — the bow of a man who had practiced both humility and command until they were the same gesture.

“Abate,” he answered, his tone even, “your welcome honors her more than me. I only follow her wisdom, and your generosity in sharing what these walls have guarded so long.”

Lorenzo led the way through the corridor. “Come — warmth first, dust later. You’ve kept the stars busy this morning; you must need rest now.” His glance lifted briefly to the paling sky as they passed beneath the cloister arch. “Few choose to ride before dawn in winter.”

Deonardo’s mouth curved slightly — not a smile, but near enough. “The stars keep better company than thawing roads.”

Lorenzo folded his hands behind his back, his tone carrying a quiet thread of approval. “Then you’ve chosen your hour well. Your rooms are ready. The mountain may rest for you now. Donna Aurelia di Vercado’s request for those old herbals is most timely — winter fevers will not wait for spring.”

Deonardo inclined his head again, soft enough to land as gratitude, not deep enough to give more than that.

“San Michele keeps its reputation,” he said. “Your letter was kind.”

He lifted his hand slightly, a gesture no wider than the breadth of a sleeve. Matteo, the older servant who had been as silent as a shadow at his side, stepped forward without a word, drawing a small satchel from beneath his cloak. Its clasp gleamed faintly as he placed it in Lorenzo’s hand — weight enough to be felt, unspoken enough to remain courtesy, not transaction.

Lorenzo received the offering with ease. Yet the weight in his hand spoke quickly enough — not a token for candles, but silver heavy enough to warm the refectory all winter, to mend roofs, to buy vellum for a year. His eyes did not widen, his pace did not falter. Only the faintest deepening of his nod betrayed that he had measured it, and found it generous.

“San Michele will remember such devotion,” he said smoothly, “as surely as parchment remembers ink.”

Deonardo’s steps followed his host, his gaze lifting briefly across the courtyard — the arches, the ribs of the cloister vault, the stonework clean against centuries of frost. “The ribs above your cloister walk echo Silvacane’s hand. My Lady knew well where to send me. San Michele guards not only prayer, but memory — and such memory must be tended with care.”

“It is tended,” Lorenzo answered simply, with the same warm evenness, as they turned toward the west range where the guest chambers overlooked the cloister garden. “Come. The fire is already lit.”


The cloister opened wide as they walked: square, frost-rimmed, the fountain in its center still trickling beneath a skin of ice. Monks passed in twos, robes brushing stone, their gazes sliding sideways toward the silver-haired guest before dropping back to their prayers.


Lorenzo walked on, voice smooth as water on slate. “You will want for nothing here. Fratello Pietro manages our cellars. He knows every barrel and sack.”

His eyes reached the older servant then. Matteo stepped forward with quiet precision, bowing just enough to honor without excess. “Matteo at your service, Abate.” His voice carried the steadiness of one long accustomed to speaking only when it mattered.

Lorenzo’s eyes slid kindly to him, warmth unbroken. “You will go to Pietro. He will see you supplied.”

Pietro appeared almost on cue, a round man with wine scent in his beard. His bow was genial, not too low. Matteo returned it with perfect ease, already falling into quiet exchange — a man who could talk in half-sentences and still say enough.

“Fratello Giuliano tends the stables,” Lorenzo continued, already shifting his glance toward Luca, whose glove dangled again from his fingers. “He is young, but he learns quickly. Horses, wood, fire — you will have his help.”

Giuliano’s broom still leaned against the cloister wall where he had abandoned it earlier. The boy blushed scarlet when Luca smiled too broadly at him, then bent double in a bow that was nearly comic. I could hear both their minds sparking with nonsense already. Ah, this would be noisy.

They moved further on. Past the chapter house, its great wooden door shut tight. Past frescoed arches, saints’ faces flaking pale from limewash, vines curling faintly over the ribs. 

At last Lorenzo paused before a heavy oak door, iron-banded, already glowing faintly with firelight within. “Your chamber, Cavaliere. Between Prime and Sext, Fratello Niccolò will come. He keeps our library dust. You may find even dust can be generous, in the right hands.”

He folded his hands once more, inclined his head with that same unbroken warmth, and stepped back, leaving Deonardo to the hush of his room.



The chamber was already warm. Fire hissed in the brazier, wood smoke settling low against the beams. He entered without hurry, the Abate’s steps fading behind him, and for the first time since dawn his movements were his own.

The cloak slid first — white fox fur folding back into plain wool, steam rising faint where the snow melted. He set it carefully across the chair. The sword leaned against the wall, its silver clasp catching one narrow shaft of light. Gloves, belt, buckle: each thing removed with the same precision, each gesture a closing parenthesis.

The bell for Terce tolled, dull and weighty.  
He moved to the window. The shutters were narrow, leaded glass catching the frost. Through them the cloister square lay rimmed with snow, monks hurrying to prayer, untill the church swallowed them.
He stood still, one hand resting against the granite sill, gaze sliding outward to the mountains, when the psalms tumbling upward like smoke. 
I curled tighter on the foot of his bed. His silence was not empty. It carried something older, something his body remembered without asking him. A resonance, low and rough, not from Porta d’Argento’s marble halls but from earth and cliff, from stone that never cared for polish. And I knew then: the monastery unsettled him not because it was strange, but because it was too close. 

He drew a slow breath, then another, as if testing whether the air in here could be trusted. Then he turned back, lowered himself to the bench by the fire, and pressed his palm briefly to his brow.

I stayed where I was, listening. Not to his mind — which kept itself locked like a treasury — but to the sound beneath his ribs, the ancient one he could not silence.

The latch clicked soft — Matteo’s knock, no louder than knuckles brushing wood. He entered without breaking the hush, set a tray on the table: bread still warm, goat cheese, a dish of figs, a cup of watered wine. No words. Only a bow, the kind that dissolved into the room rather than interrupting it.

Deonardo ate little. He broke bread into small pieces, fingers steady, gaze slipping often back to the window. Matteo inclined his head once more, then withdrew.

Silence settled again until the scrape of boots along the corridor, and a gentler knock this time. The door opened with a pause, as if even the hinges had been taught courtesy. A thin hand appeared first — ink-stained, the nails neat — then Fratello Niccolò stepped inside.

His bow was exact: neither too deep nor too brief, the kind of bow a man gives when he wishes to be remembered as polite and forgotten as soon as he leaves. His voice matched it, low and even.

“Cavaliere, I am Niccolò, keeper of our library dust. I hope you have found this chamber warm enough. If you are restored after your travel, I may show you the books. If not, I will return after None.”

Deonardo inclined his head, the same tempered grace he had shown Lorenzo. “Your courtesy is generous. After Sext, perhaps. The road still runs in my bones.”

Niccolò smiled faintly, as if he had expected that answer, and slipped away with the scent of ink trailing after him.

The door closed again, and I leapt soundlessly from the bed to the bench beside him. At last he turned from the frost-feathered glass, his eyes settling fully on me. 
For a breath he said nothing. Then, with the smallest incline of his head, he spoke:

“Benvenuto. If this chamber is to shelter me, then it is yours also. We share it.”

Ah. A proper welcome that I rarely hear, with a courtesy woven into his breath, equal to an Abot, a Brother.

So I flicked my tail and gave him something back.

“Silver hair, travel coat dyed in Aurelian Blue, snow fox fur on your shoulder — but that’s not what holds them,” I murmured, letting my words hum in the air between us. “It’s the way your steps don’t creak like old wood, the way your face keeps both age and youth at once. The monks called you foreign. I call you beautiful — which is worse.”

“What a tragedy,” I narrowed my eyes, “the way you survive inside that beauty, the way you let it be caught without ever letting it be known.”

He couldn’t hear me, not as words. Humans never do. But his gaze moved toward me as if he had.
Not recognition — something quieter. The kind of pause that means a thought brushed his skin, though he won’t admit it was mine.

He leaned over the table, fingers near the bread he hadn’t finished, fell into nape. The tilt of his head left his neck bare — pale where silver hair loosened, flushed faintly from heat and travel. A soft place, almost fragile. Humans don’t know how much beauty confess there, the hidden lust that words cannot defend.



The second time the ink-stained monk came, between the bells for Sext and None. His knock was lighter than Matteo’s, but more assured than before.

“Cavaliere,” Niccolò said, bowing with the same exactness as earlier. “If you are ready, the scriptorium is near. It may be better to see its order before you ask too much of it.”

Deonardo rose without haste, drawing his cloak once more over his shoulders. His answer was only a nod, but it was enough.

The stairs bent them downward into cooler air. The smell of ink and scraped parchment rose before the doorway even opened. Inside, desks stood in rows, angled to the narrow windows; sheets of vellum lay stretched on wooden frames, half-dried, pale as winter skin. Candles guttered low in iron holders, their smoke staining the ribs of the vault.

“This is where most of our labor rests,” Niccolò said, his voice lowering as if it too belonged to the hush. “Every hand here copies or guards. To the north wall — scripture, ordered by psalm and gospel. To the west — letters of law and council. The south holds physic, herbals, and the old Roman materia. To the east — charters, chronicles, deeds, and local histories, though some of them do more sleeping than speaking.”

He paused, letting Deonardo’s eyes travel the walls.

“When needed, we send certain books beyond these desks. The infirmary often borrows physic and herbals.”

Niccolò ran a finger along the spine of a vellum codex, brushing the thin film of dust that rose from it. “Some of these are fragile — recopied from Salerno, from Montpellier, carried here long before either of us.” He lifted one carefully, its binding stiff, its ink browned. “But not all the pages rest here. Madonna Dania, the healer keeps them well. Sometimes she writes her notes in the margins — in her own hand, but always in Latin, so no one may call it strange. You may find her copies as useful as mine.”

He set the codex back, smoothing the edge of his sleeve as though dust should never linger too long. “If you wish, Cavaliere, I will carry whatever you choose to your chamber. Or you may walk here yourself. I am always between these shelves.”

Deonardo’s head inclined, but when he spoke his voice carried a quieter softness than before.
“Your courtesy is generous, Fratello. Yet if it is permitted, I would prefer to walk the library myself. Silence belongs best to its own walls, not carried in trays to mine.

“And if I were to send Matteo, later in the day, to inquire whether the healer might receive a visitor… would that be proper? Or does custom frown on such requests?”

Niccolò answered with his voice level.
“It would not be unusual, Cavaliere. The locals go to her directly — merchants, farmers, even the steward from Valdiponte sends his men when the fevers rise. She keeps a space for it, half her own, half open: shelves, benches, jars, and a hearth. Not private, not fully public. A place for herbs, and for those who need them.

“The Abate leaves her to it. He says San Michele gains more by her craft than by binding her with rules. Matteo may go — she will answer him. Only know this: she does not waste words. Courtesy is given, but she values plain speech above ceremony.”

Niccolò’s tone stayed mild, his hands folded, as if describing a shelf’s order rather than a woman’s place.

While they spoke of her, my ears twitched.
Between the shelves — ah.
Romé.

He had slipped in on softer feet than usual, but not soft enough for me.
His shadow pressed against the spine of a pillar, his eyes fixed not on parchment but on the man whose hair caught the light like ice.

Deonardo didn’t see him. Niccolò didn’t either.
But I did.

And I thought — this library was going to grow louder than its dust very soon.

Black Fire in the Library 🕯♛🐾  midday, scriptrum


I leapt up to the beam again. From there the library stretched wide below me — desks, vellum, candles, dust. 

Yesterday Romé wore borrowed cloth — tempting novices through the cloister, then by evening undone beside the healer, clawing silence apart with his breath.
Today he was different. He had dried himself into fire again.

His own garments, freed of snow and damp, clung back to him. He sat in black. The tunic was silk-wool, dyed so deep the light slid over it like water on coal. Over it, a surcoat darker still, patterned with sharp lines I had last seen in Saracen tiles. Even I wanted to blink at it.

On his wrist, a cord knotted from lapis and bone — a peasant’s charm bound to a jewel that belonged in Byzantium. Three rings circled his fingers, each uneven, bent as if they had been hammered in the rhythm of waves. Bronze plaques glinted on his belt, one stamped with a beast I did not know.

Absurd. Yet on him, the absurdity struck like thunder. The symbols no longer told their stories — they collided, burned together, sparks brighter for their clash.

The novices stared — some muttering, some crossing themselves. Even Niccolò glanced longer than his ink stains allowed.

But Romé — he was reading.

I dropped from the beam to the pillar nearest his table, tail flicking once to balance. From here the letters blurred, too far for my eyes to claim. So I leaned closer from the pillar’s shadow.
 Ah, temptation. His gaze was too intent, his breath too steady.

I slipped into it — into his eyesight — just for a moment. This little demon trick was so quiet it didn’t stir the dust.

The page blurred and then sharpened through his sight:

⸻

The parchment cracked faintly at its fold, ink browned, but the hand was still sharp.

Anno Domini 1246.
To my lord,  
(the name of Lord was marked by black, interesting.. )

I write of the Wild Corridor and the roads between the passes.

For three decades we held true with the house of Castrovellano. Under the old lord, we paid the levy, and his riders gave us shield against brigands. Salt, alum, hides, wool — all reached Senigallia with small loss.

Now the new master, Corrado della Rocca, raises fee upon fee. His swords drink more wine than they guard, and their steel is more eager for brawl than for duty. We lose men and mules while he counts coin. The bargain no longer holds.

I have set the reckoning plain:
Through the Corridor — swifter by twelve days, but costs doubled, danger doubled.
Through the safe road by the papal tolls — longer, but certain, with law and patrols to guard.

My counsel is thus: let the tie with Castrovellano lapse. The name once kept faith. Now it keeps only thirst.

The ink trailed into a closing mark, a blot dark as the name it condemned.

Ah. Castrovellano. Corrado della Rocca. His house’s herladry- the same black tower on red rock, stamped on the armor that Romé wore when he arrived in the heavy snow.

Most men, when a scrap of ink dares speak ill of their lord, puff red or spit. Pride rattles louder than sense.

But not Romé.

He only stilled. Not frozen — the way a cat goes still when something lines up in its bones. His jaw eased, his shoulders dipped just enough to loosen a weight long carried.

In 1246 he had been a child. The words were not his memory, yet as his eyes passed them I felt the shadow they stirred.

Two decades after, the silence still clung to his house. His father’s voice dropped whenever the old lord’s name drew near. Not because the della Rocca estate was failing — their land still yielded, their name still carried blades — but because what once held the household together had thinned. 
Caravans gone. Trade ties broken. Summons to battles that brought no gain. The peasants groaned under levies; the stores emptied faster than they filled. It had never been spoken aloud. Silence was the only truth passed down.

Now the ink gave shape to that silence. Not rumor, not hint, but truth written in a captain’s hand. It wasn’t shame that entered him. It was clarity.

Strange creature — most men crumble when paper steals their illusions. Romé looked as though the words had given him back a piece of ground he’d never stood on before.

I flicked my tail and slid back into my own sight. Around him, whispers rustled, glances flickered — but he didn’t twitch. He stayed easy. Not defiance. Worse. He didn’t notice them enough to defy them.


Then, cross the beams of daylight through narrow windows, and the floating dust, my whiskers caught a shift.
Deonardo.

He drifted between the shelves, moving as if he were still measuring the room rather than the parchments. His sight skipped, not staying long on law or psalm, always circling back — to that one place where whispers gathered like moths.

I twitched my ears, slipped again — this time through Deonardo’s eyes.

Ah. So this is how the silver-haired one sees.

Not the way how monks catching demon..and fell
No, his eyes cut fine lines, weighing, comparing. They notice the grain of vellum, the sheen on ink, the fold where a bookbinder’s knife had pressed too hard.

And when they fell upon Romé — ah.

They caught the black fabric first, how it drank light differently than wool should. The shimmer in the weave — too fine for any valley, too foreign even for a ship out of Tunis. Then the amulet’s glint — absurd, a relic of divided temples stitched into the silence of psalters. And beneath it all — a rhythm cloth could not disguise: shoulders drawn back, waist narrowing, the posture of beauty long accustomed to being watched.

But Romé’s attention never strayed from the parchments. His head bent low, fingers moved with quick precision, turning, stacking, reshaping the files. His lips stirred faintly, as though tasting the names written there. The cold daylight cut across him, striking his silhouette sharp against the pale vellum. The glowing edges made him mythic — yet his gaze clung only to the text.

Deonardo’s gaze, by contrast, lingered. Longer than courtesy, longer than reason. An hour ago his inner voice had pressed him to search the scriptorium for “something.” Now he read not the page but the man — and the man did not look back.

I purred in his sight, pleased. Nothing gnaws at humans faster than being invisible to the one they cannot stop seeing.


A small heat lifted in his throat, but his eyes did not stop there. They traced Romé’s silhouette — neither wholly masculine nor wholly feminine, belonging to the ancient gods who stood without gender, whose temples had fallen to ruin over centuries, with only shepherds and sheep still spending nights beside the relics of their bodies.
Deonardo knew he was looking at a beauty whose very name, in all the vellums that once recorded it, had long since been burned away.


I slipped back into my own sight, whiskers twitching. Across the aisle, the ink-stained one lingered, seeing what I saw. Niccolò.

His head bowed toward a shelf, but his eyes — the kind that collects and keeps. Amused, he savored the silence the way some men savor wine. One corner of his mouth softened, almost like a smile.

I remembered the first night: the only pair of eyes sharp enough to catch the heraldry on Romé’s armor, when all the rest were still blinded by his beauty.

Yes. Niccolò sees. He always has.

Between Swords🕯♛🐾  Sunset, bell tower

The sun slid behind the ridge, leaving the church square half gold, half shadow. Kitchen smoke thickened the air; frost clung where no fire touched. I crouched above the cloister arch, tail curled, watching.

Ferrazo was already there — the old captain, broad as a cart, his shoulders sloping with years of armor. Helm tucked under his arm, he leaned against the wall beside Lorenzo. Their words were low, but the rhythm carried: coin promised, blades weighed, the rest after rents.

At the bell tower base, three mercenaries had come before the first fall of snow, and now sprawled around the turn where the horse path connected the church square and the cellar. One stitched his arm in the infirmary, another stumbled back from Valdiponte half drunk, the third gnawed pork and spat bone into the mud.

By sunset, two more had joined. One fresh from Pisa, stinking of brine, swore the pay was never worth the blood he lost against Genoa. The other — Martino of Florence — wore a tabard slashed with feud-colors, once fine, now frayed. Lean, restless, his belt too ornate for his means, his shoulders twitched as if forever about to unsheathe.

And in the shadow behind them sat Agnolo — the Oak. Wordless, back to the wall, gaze unhurried. His silence pressed harder than Martino’s noise, and the others shifted round it without naming why.


The monks passed them quick, eyes down, taking the stair instead of the horse path. Even peasants fetching water edged wide around their corner.

For a time, Romé lingered on the guesthouse roof, watching both terraces: the lower square with its church and bell tower, the upper cloister garden.

Then he saw her. Dania, crossing down toward the cellar — straight toward the corner where the soldiers gathered.

He didn’t think. He moved. Down the stair, across the terrace, following the line she had taken, his shadow falling where hers passed.

Her skirts brushed frost and mud as she turned beneath the tower. The men shifted when she drew near, restless in that way soldiers do when something finer than their trade crosses their ground. One gave a half-smile, teeth more than courtesy.  

And the Oak — his eyes lifted once, slow, to her.


Then they saw Romé behind her.
The smile shut. Shoulders braced. 
The loudest broke off mid-jest. Another spat into the straw, eyes narrowing. recognition born of steel. He wore no armor, no blade; his black clothes were too fine for a soldier. Yet his presence bore the weight of one bred to the sword.

Romé gave them nothing, not even a glance. His eyes stayed on Dania.

Dania slipped into the cellar. The door closed.
Romé stopped just beyond its shadow and waited, to stand guard.

The mercenaries muttered among themselves, but their noise bent around him. None called out. None dared.

When at last she returned, the air of earth and herbs still clinging to her cloak, his gaze lifted once, steady, then dropped aside. He shifted just enough to clear her path, yet did not move away — his nearness a shield unspoken.

As they passed beyond the corner, the mercenaries’ laughter broke back, harsher than before, like blades striking stone.

Above, Ferrazo watched, catching the small play — even while bargaining with Lorenzo. His rumble came low, iron wrapped in smoke.
“Two weeks,” he said, fist closing as if to seal it. “At new moon, they’ll be ready.”


When dania passed the cloister guesthouse arch, she slowed, then stopped. She knew his chamber lay within those walls. He might have returned there. Yet she felt him behind her still, his steps unbroken.

So she waited.

Not for words — but because in the shadow of his guardianship she sensed something more than kindness. A heart that had not yet learned what to do with loss.


The last sunlight was gone by then. Cold cut sharper, sudden as a blade.
She stood in the dark between cloister and guesthouse, her form outlined only by the faint spill of lamplight from the cloister walk.

It was the blue hour.
The kitchen stirred beyond the rosemary beds, clatter and voices muffled, distant as another world.

Romé saw her pause — and in that moment, with the cold tightening round stone, something in him yielded.

I softened my paws, then leapt — landing on his shoulder, pressing my warmth into the hollow there. His breath shifted.

“I am frozen,” he said to her, voice low, bare. “Can I eat with you?”

“I have soup from the noon,” she answered. “Old bread, and salt bacon slices.”

Then she turned onto the narrow path toward her garden.
He walked one step behind, and I rode his shoulder, tail brushing his neck.

“It becomes so cold,” he said again, quieter now. “I am starving.”

The moon was rising — not yet quartered, still broad, its light pale as snow.
It spilled into the stone path, glinting on frost, catching on the dark green needles of the cypress.

She was quiet.

The quiet stretched through the walk, through the garden gate, through the darkened hall. It lasted even as she laid the bread, even as she stoked the coals. Only when the flames caught, throwing their shimmer into the pot, did he speak.

“You might be more careful about them,” he said at last, watching her stir the soup.

“Always some of them there,” she said, her tone as even as the stirring of the pot. “The lord of this valley keeps his court in Bologna. His men never come, only collect rent through deputies. A part of the rent stays here — enough to hire mercenaries to drive off bandits. Next Tuesday, St Martin’s day, the rents are due. The fund will be settled. I expect more of them will gather in the weeks after.”

“It doesn’t mean they are safe,” Romé said, his voice lower, less calm. “Are they regular men, living here? Or only come and go like a turning river?”

“The old captain Ferrazo lives here these years,” she answered. “He chooses the men from the flow. Lorenzo deals only with him.”

“The old captain knows things. But there are always flaws, ruptures, especially among mercenaries. Too many wounds, too much death — some of them already gone mad inside. They might violate women, or kill for no reason.” His unease grew louder, the delicate frame of his face shivering in firelight.

“You know them well?” she asked — almost a voice without question.

“I was born in a house of swordsmen.” His tone folded inward, stripped of boast. “The lord we serve now is a living legend of the blade. Never once failed in a duel. His court gathers only men of the sword — nothing else.”

“Are you one of his chosen?” Her voice was light, almost overlooking the weight in his — perhaps on purpose.

“I prefer books that tell legends, rather than stand beside a living one.” His smile curved bitter. “My brother was.”

“Not anymore?”

“He passed in my arms. Four years ago. In snow. We were sent into an impossible battle.”

His head turned slightly, eyes slipping into shadow, harder to catch. His hands stayed on the table, cradling the cup. Long fingers quivered in the firelight.

She did not speak. Only moved slowly, laying out the dinner on the table, one piece after another.

He did not touch the bowl. After a while, his voice returned — the voice of one who would not permit himself to cry.

“I am sorry. I said they might be mad. It isn’t fair. I don’t know them at all. Sometimes I wonder if the mad one is me.”

“You can cry,” she said gently, sitting down on the other side of the table.

His eyes shifted out of shadow, found hers.
“Where is your family?”

“I don’t have family.”

A pause.

“I am worried,” he said, his gaze locking to hers with unusual tension.

“About me passing the soldiers?”

“About…” His voice faltered. He did not finish.

I tried to read his mind, but something howled through him — so fierce, so raw — that even my demon’s power could not break the block.


She didn’t ask further.
He began to eat then, the soup nearly gone before it seemed to touch him.

When it was finished, he gathered the bowls and carried them out.
When he stepped back, the words came at once, as if they had been waiting behind his teeth.

“Might I borrow your couch tonight?”

Her gaze rose to his, steady.
“As you wish. There are blankets in the treatment room.”


After saying so, she withdrew to her private rooms — the west side of her garden, where no step but hers ever seemed to fall.

Moonlight bleached the terrace, whitening stone and frost into silence — and perhaps her steps, when she crossed it.
We did not know. We stayed on the east side: dining room, herb studio, treatment room — where Romé found the blankets and the narrow couch.
I curled beside him, close to his breath, letting its rise and fall hold me.

“Stay with him.”
I almost heard her, before she left.

Strange —
her silence was whole. I could never read her mind, nor touch the voice within.

⇞⇟Cacifer’s interpage:  A Brief Legacy of San Michele and Valiponte
as recorded by Cacifer, witness and keeper of the ridge
San Michele and Valiponte did not grow apart — they grew in tandem, each fulfilling what the other lacked.
The monastery of San Michele was founded in the early 10th century on the ruins of a Lombard fortress. First a refuge for ascetic monks, it evolved into a center of herbal medicine, scriptoria, and quiet hospitality. Its position high on the ridge gave it strategic view and spiritual authority — but it never held military or economic power. Instead, it kept time. In its terraced gardens, memory was stored in plants, in ink, in silence.
Below, in the bend of the valley, rose Valiponte — a town shaped not by sword or crown, but by three merchant houses:
Basili degli Aromatari, wealthy in spices and rooted in landholding stability,


Maffei, dealers in marble and spectacle, ambitious and volatile,


and Rinaldeschi, precise and austere, who built their wealth on legal technique and account.


Together, they formed a local order — economic, civic, and quietly political — beneath the nominal overlordship of the Guidarini family of Bologna, who held the valley only by charter and title. Real governance passed between these houses, through marriages, feast days, rents, and the annual registers of trade.
While Valiponte moved in coin and crowd, San Michele remained apart — but never severed. It offered healing, learning, and refuge to all three houses, while remaining outside their rivalries. In times of pestilence, it opened its herbariums. In times of crisis, it copied records and kept grain. What it did not trade in gold, it traded in continuity.
The valley endured not by conquest, but by co-existence:
The monastery watched and waited.


The town bargained and built.


And through centuries, each gave the other what it could not grow alone.


The story is not one of saints or kings. It is a ledger of breath and structure — one steep, one low — held together by a bridge no one remembers building.
But I remember.
And I still walk across it.
Every time I climb between them.
— C.




📖 November 8, 1264 Saturday, bright morning, day of slight mist, Waning Gibbous moon
 Rinaldo Rinaldischi’s arrival🕯♛♡🐾 morning.
The day was bright, warmer than it should have been. Three days since the snow had ceased, and already the road to Vallefredda was breaking into mud under too many boots. The monastery bell rang Terce as it must, but few monks made it all the way to the hall. Most paused at a cross, muttered their prayers half aloud, hands still busy with broom, hammer, or ledger.
The square was filling again.
Preparations were underway for the Feast of St. Martin — a day that, in this valley, was not only Mass but market, rent, and marriage. Once a year, villagers from across the hills brought the year’s portion: barrels of oil, sacks of barley, waxed cheeses, wool, woven cloth, sometimes even hens or rabbits, still bound and blinking. The tithe to God was given, yes — but the real draw was trade.
Here, a shepherd might swap wool for shoes. A farmer’s daughter’s dowry could be bartered over in whispers near the fig cart. Merchants came to load up mule-trains before the winter roads closed. Deals were struck in stone corners with hands half-shaken and promises marked by thumbprint, not ink.
And above it all, quiet but precise, the agents of Bologna recorded what passed.
For three generations, the Rinaldeschi had served as resident deputies of the Guidarini di Bologna, managing the lords’ affairs with signature and seal. Among them, none held the role with more exactitude than Rinaldo, grandson of the first notary sent from the city. This was his week — the autumn levy. Rents to collect, tallies to confirm, shipments to dispatch.
The peasants paid in kind, not coin: olive oil in barrels, grain in sacks, cheeses lined in wax and straw. San Michele was the counting house, not because it welcomed him, but because it had all a man like Rinaldo required: dry cellars, thick walls, and obedient hands. The monks tallied, packed, and sealed. The ink they used served both heaven and Bologna.
Rinaldo himself would never keep a warehouse in Valdiponte — not for two or three collections a year. And he had no need to go out to the villages himself. In a place like this, to come for Mass but not for rent was to shame oneself before God and neighbor alike.
So everything came to him — quietly, methodically. Each sack, barrel, wheel, or bundle passed through the storeroom and into the margins of Rinaldo’s ledger — a book, some whispered, neater than most altars.
And in this silent rhythm — grain, ink, silence, mark — the power of Bologna moved without personal presence
So it had always been.
Out on the terrace between the outer gate and the guesthall, Rinaldo found Brother Pietro already in motion. The old monk did not bustle, but everything around him did. He stood among his novices with sleeves rolled and posture straight, steadying the flow.
One novice chalked lines across the terrace stone, marking where each cart must halt. Another rolled empty barrels under the shed, ready to be exchanged for full ones. A third laid straw thick across the entry slope, so mule hooves wouldn’t slip once the weight pressed in.
Pietro didn’t raise his voice. He barely moved. His hand gestured once — left, right, forward — and each signal redirected the motion around him, clean as a counterweight shifting grain.
Even Rinaldo paused a moment, watching.

“Brother Pietro,” Rinaldo greeted him, cloak still damp with road-slush. The words were brief, but his eyes softened for a moment — a flicker of familiarity.
“And welcome to you, Ser Rinaldo,” Pietro returned, his voice calm, unhurried, though he did not stop his men from moving.
“Monday,” Rinaldo said at once, eyes sweeping the terrace, the sheds, the pens. “All rents in one day. Tomorrow is God’s, Tuesday the feast. I’ll not have carts spilling into holy hours. Stake ropes in rows, space for each cart to unload — sacks to the left, barrels to the right. My men will weigh and tally double-speed. Even on Sunday the bales can be set in place. Silent measure waits for Monday.”
Pietro did not argue. He turned instead, beckoning one of the senior lay-brothers — broad-shouldered, hands black with straw dust, the man who kept the guesthall in order.
“Could it be done, Brother?” Pietro asked simply.

The man nodded, already sketching the plan aloud: ropes strung across the terrace, markers set by weight and kind, mules unloaded in sequence, oil casks separated from grain. “By Monday dawn it will stand ready. No more confusion than a fairground.”
Rinaldo’s mouth curved — not quite a smile, but the edge of approval. “Good. And two or three more of your men in the cellar for storage. My clerks will keep the tallies, but I need hands quick to shift the loads.”
Pietro inclined his head, a touch of dry humor in his tone. “We will anchor it, as always. Even chaos has its place, if you give it walls.”
Rinaldo gave a short laugh, as if reassured the rhythm was already set.
“Walls, and men who know them. Then it will be done.” 

Next one he looked for would be Lorenzo. Rinaldo crossed the bell tower square and entered the tunnel behind the church — a stair carved straight through rock. It rose sharply, torchlit and damp, and opened onto the upper terrace: the cloister garden.
To one side stood the cloister proper. To the other, the guesthouse — two wings, east and west, facing the rosemary beds.
The east wing was quiet. At the end lay Romé’s chamber, tucked near the path to the healer’s garden. A few doors down, Rinaldo’s steward checked locks, jugs, and bedding. They would stay three nights — enough to collect rent, settle ledgers, and leave.
The west wing was louder. Donata Maffei’s maids had arrived early, unpacking bolts of cloth, fine linens, and a copper brazier. They covered monk-issue furnishings with her own. The rooms smelled of resin and smoke.
So the guesthouse split in two:
– One side: measured, businesslike.
– The other: layered with fabrics and fire.
Rinaldo didn’t pause. He crossed the garden, cloak brushing rosemary, and passed under the cloister arches. He took the stair up toward the abbot’s corridor.
At the turn, he passed Matteo — the servant retreating from Deonardo’s room with a breakfast tray. Matteo bowed. It was the kind of bow trained in courts, that he only learned from high range butlers of his lord in Bologna.
Rinaldo’s eyes flicked toward the door. First floor. Best chamber in San Michele. Not a place for ordinary guests.


Rinaldo’s stride did not falter until he reached the abbot’s chamber. There, as every year, formality met that peculiar warmth: a smile from Lorenzo, calm and timely, never lavish but always enough. It gave the sense of having been received fully, without waste.
They exchanged no pleasantries beyond what mattered.
Rinaldo bowed, setting a small chest of sugared almonds on the desk beside a sealed letter.
“From my lord in Bologna. And with gratitude.”
Lorenzo took them with both hands. His movements were light, but never careless. Nothing offered here would fall unnoticed.
“How fares your lord?” he asked. “The tea I sent—did it ease his nights?”
Rinaldo nodded once, and from his cloak produced a folded parchment, the seal unbroken.
“He found some relief he had not known before. He hopes the remedy might be refined. The cause of his unrest is written here.”
He laid the paper down carefully. Lorenzo did not open it. His hand rested beside it, as if weighing more than the parchment.
“And your lady?” he asked, softer now. “Her devotion humbles us. To ride that path each week, in all weathers… She sustains more than her own prayers.”
Rinaldo’s voice held steady, but his eyes moved slightly.
“She believes God asks no less. I’ve urged her to spare herself. She refuses. One servant only. Always the same. She says God needs no retinue.”
Lorenzo did not press. His smile remained, a little quieter at the edges.
He turned instead to the matter at hand — sliding a small parchment forward, already stamped with the monastery’s red seal.
“For Monday’s tallies. And Ferrazo is spoken with. His men will be ready in two weeks. Payment as before—half now, half when the road is cleared.”
Rinaldo inclined his head, a gesture that closed the exchange without further word. He took the slip, folded it once, and turned.
But instead of returning to the square, he passed through the side corridor — toward the scriptorium.
Few men entered that door, but I watched from above as Rinaldo stepped in and bowed to the one monk who kept the monastery’s spine intact: Niccolò.
Niccolò did not look up at once. His hand moved steadily over a sheet of parchment, drawing the last arc of a signature that bound grain to name, name to tithe. Then he raised his eyes.
They spoke without greeting.
Rinaldo asked for copies: the rolls from three winters past, the tithe register bearing the Guidarini seal, the recopy of the autumn ledger sealed after the fire.
Niccolò answered without hesitation: chest, shelf, bundle, ribbon color. He remembered each hand that had written here, each fee, each correction.
This was not the language Lorenzo used — it was drier, tighter. But here, beneath the lamp, two men met in precision. One error, and a lord’s power would come undone.
Rinaldo’s voice lowered.
He did not say, You are clever.
He said, almost like confession:
“Abate smiles. But you keep the walls standing.”
Niccolò did not nod. He only dipped his quill again — but his hand paused, just briefly, as if the words had landed in a place where praise was rare.
When Rinaldo left, his mouth curved faintly — a private amusement. 
My tail curled into a smile of its own. 

While Niccolò and Rinaldo counted parchments like rosary beads, the silver-haired one sat a few desks away, hands on a folded charter, eyes elsewhere. For a breath, something passed through him —
These two… brought to Aurelia’s court, bound to my hand… what might be built?
Then he pushed it away and returned to his search, irritated by the drift of thought.
Below, Matteo moved through the cloister with his usual quiet rhythm. That morning, he had carried a letter from Deonardo to the healer — and now he returned with her reply.
He bent low to Deonardo’s ear, voice no louder than breath, so as not to disturb the stillness of the scriptorium.
“The healer sends her answer, my lord. She says you may come when it suits you — her fire is always lit. You’ll find a few books and records there, if they serve. As for herself, she claims she has little to add beyond them. You may read as you like.”



📖  Shadows and Silver🕯♛🐾 healer’s dinning table 

The monks say nothing ever happens here.
Which, of course, is why everything does.

I know the sound of a day turning.
It doesn’t clang.
It folds.
Like cloth slipping from a high shelf —
quiet,
but final.


Silver hair caught the last rim of light; his surcoat in Aurelia blue shone differently here. He bowed to the healer — a bow meant to weigh respect without debt — then stood just inside the door, one hand resting lightly on the frame.
“I am Deonardo di Alferini,” he said, voice low but clear. “Your message reached me — and if the invitation still stands, I’ve come to study what books you permit, and to learn what I may.”
He paused, not stepping forward.
“I won’t stay longer than needed. Only what your time allows.”
Dania was cutting clean through a root, the rhythm of dinner already begun.
“Cavaliere,” she said. Nothing more.
“I fear I intrude,” Deonardo said. “If the hour is not fit, I can return when it suits you better.”
Dania did not lift her eyes from the board. The knife pressed once more through the root, then paused.
“You do not disturb. I am preparing dinner,” she answered. “You may stay, if you don’t mind it being thin.”
Her voice, level, carried neither invitation nor refusal — yet they set a place without needing a chair.
The silver-haired cavaliere lingered a breath on the threshold. But before Dania’s calm words could settle fully, I leapt — up from the bench, tail snapping, onto the dining table with a soft thud that rattled the wooden spoons.
Deonardo started, then stopped — and the stillness broke into something gentler. His eyes followed my whiskers. His breath shifted. He gave a quiet laugh in his throat.
“You again,” he murmured, and for the first time that day, his gaze eased.
He lowered his hand toward me, careful, not presuming — as one greets a friend found twice in unlikely places.
Then Romé rose.
He had been hidden on the healer’s couch since morning, a book in his hand. In truth, he had not left at all since we woke together today.
His profile caught between firelight and shadow. The black fabric of his sleeve slipped as he moved — a ring glinting once, then still.
“Lord Cacifer,” he said, introducing me with reverence, “will stand witness and give you his regard.”
I flicked my tail, whiskers twitching. My way of bowing — no lower than a prince would stoop.
Deonardo paused a step inside, silver hair catching the firelight. His gaze dropped to me, perched on the table, tail wrapped like a seal. He bent — not mockery, but the smallest note of recognition.
“Lord Cacifer.”
My ears flicked.
Only then did he speak again, straightening a little, as if drawing a line under the moment.
“I am Deonardo di Alferini, in service of my Lady of Vercado.”
He had already given Dania her thanks; his eyes now rested on Romé.
They had shared the same air in the library the day before, measured each other from a distance, but never crossed the formality of names.
Deonardo bowed — exact, but not hurried. “Signore… we have not yet met.”
Romé bowed from his corner, the book still in his hand.
“Romualdo di Orellana.”
The names lingered — faint as sparks in the air.

Then the sound of wood on stone — Dania setting down the bowls, one by one.
“Sit,” she said, with the plain gravity of one who knows food must be eaten before it cools.
The table turned from stage to hearth in an instant.
Steam lifted, bread broke, silence bent toward sharing.
I crouched on the table edge, whiskers twitching, and watched the balance tilt.
Romé’s body was open, almost relieved — the way a man sits when he forgets to guard himself. Dania’s hearth had lowered his armor without asking; her walls held what he no longer wished to carry. He was vulnerable, yes — but not undone.
Deonardo was different. Even as he inclined in courtesy, some part of him never unclasped. His shoulders drew lines of measure; his gaze weighed objects before people. His body was eased by the warmth, yet his mind tugged at him still — pressing the cover he had brought with him, scholar’s cloth draped over soldier’s bone.
Dania — ah, Dania — was herself the stillness between them. The light from the fire caught her profile, and she let it rest there without claiming it.
So the room breathed three currents at once. One laid open, one hidden, one steady. And in that tension, I stretched once.
Before Deonardo’s hand touched the bowl, his voice came:
“I had hoped, Madonna, to learn of fever remedies. The herbals of Vercado speak of sage, but I have heard your valley holds stronger roots.”
A scholar’s tone. Safe. Balanced.
Romé’s head turned slightly, just enough that I caught the shadow curve of his mouth. For one instant, he seemed to laugh at the pretense and to welcome the speaker both at once, but stayed silent, letting Dania answer.
His body spoke — the slow blink, the hand closing loosely over his knee. I could read it as clear as a page: he liked the man, instinctively. And yet this voice of leaves and fevers grated him, like hearing a harp string tuned too tight.
Dania poured broth into bowls, the steam weaving between them. Then she answered:
“We use sage, yes. Also yarrow, when the fever burns higher. Willow bark for the ache. Nothing uncommon. I cannot say it is better than what you know already.”
“Still, what is plain in one valley is forgotten in another. Even simple leaves deserve to be written twice.” Deonardo spoke humbly.
Romé leaned back a little, tasting Deonardo’s careful phrasing as a piece of fine cheese. His eyes flicked over Deo’s sleeve, the neatness of his cuff. Then:
“You speak,” he said, mouth curving, “like one who sits among chancellors — weighing clauses, stroking quills, smoothing quarrels into coin. Not like a man bent over herbs. Too careful. Too fine. Books and swords might sit well in your hand… but herbs —” his gaze softened into mischief — “herbs grow wild. They don’t bow to polished words.”
I flicked my tail, purring inwardly. Ah, there it was — the spark beneath their civility.
For a breath the silver-haired one stilled — a flicker, quick as a candle shaken. Then the mask returned, smooth and courteous.
“Polished words may still serve,” he said evenly, bowing his head toward Dania’s jars as if they, not Romé, had spoken. “If they keep the record clear, and the remedy remembered.”
His tone was calm, but his hand lingered too long on the cup before him, thumb pressing the edge.
Romé’s eyes narrowed, lashes shadowing them in firelight.
“And yet everything in you speaks against it,” he said — not harsh, but with that sharp edge a blade gets when tested on whetstone. “Your cloak — woven tighter than anything a valley merchant would trade. Your bow — shaped by courts, not by fields. Even the way you pause before speaking, as if weighing the chamber first. That is no student of herbs.”
His tone playful, almost teasing. “Your coat,” he added, glancing at the folded Aurelia blue on the bench. “Not every noble house could even think of such dye.”
Deo’s eyes flickered — a breath too quick, almost a retort rising to name the strangeness of Romé’s own wearing. But he held it. His voice returned measured, precise.
“You are not wrong,” he said. “The Aurelia blue is no valley dye. It is the mark of Porta d’Argento, capital of my Lady’s land. For the honor of my Lady, Donna Aurelia di Vercado, I wear it whenever I travel beyond her borders. It is her sign, not mine.”
He paused — then allowed the smallest concession, a tilt of the head toward Romé.
“And yes, I was schooled in her court. Books and contracts more than herbs. You saw it rightly.”
Romé’s mouth curved, but his eyes stayed sharp.
“So — a clerk’s polish, a courtier’s coat, a scholar’s phrasing. Strange pupil for a healer.” His tone flicked between jest and challenge. “If I were your Lady—” he broke bread with long fingers, dropped the pieces into his bowl, stirring them into the broth as if he were a peasant speaking of sheep — “I might send two or four more men from the fields. Fingers stained with leaves, legs quick among sickbeds. Not a solitary scholar, when winter fever comes. What does she think?”
The gesture and the words together carried a weight sharper than banter: not only Deonardo’s cover, but Aurelia’s judgment laid bare under question.
Deo’s jaw tightened. His hand stilled on the cup. For a breath he seemed to swallow it — then his eyes flashed, the steel of service rising.
“My Lady thinks farther than fevers,” he said, voice low, clipped. “Aurelia sends one man because one man can learn, remember, record. When fever passes, the field hands return to plow. A scholar does not forget.”
His tone landed harsher than courtesy allowed, and the edge in it was meant to bite.
Romé dipped his spoon, raised it, ate.
Then the spoon stilled midair, his elbow braced on the table. Behind his hand, the curve of his mouth bent into that devil’s smile — the kind that hid as much as it revealed.
“Cavaliere,” he said lightly, almost admiring, “what a beautiful loyalty you bear your Lady. Especially when you name her — Aurelia — as naturally as naming herbs to cure a fever.”
The spoon dropped back into the bowl with a soft splash. His eyes did not leave Deo’s.
“But tell me — scholar — have you ever bent beside a sickbed? Held the sweat of fever with your own hands? While the game with your Lady sparkles in high halls, people in low houses die for real.”
Deo’s hand tightened on the rim of his cup, silver hair shivering as steaming. His voice came sharp, the polish gone, words cutting quick:
“I have sat through nights when half a village burned with fever — children gasping till dawn, mothers raving, fathers carried out stiff before the hour bell. I have seen not one sickbed, but halls of them, row upon row. Don’t tell me of dying, Signore. I have counted too many to mistake it for play.”
His breath flared; the words were out before he could temper them. The silence afterward was heavier than his tone.
Romé’s spoon stirred once, slow, then stilled again. That hidden smile almost curved.
“A strange scholar indeed,” he murmured, voice low as if only to himself. “You speak less like a healer’s pupil… more like a man who rules where death is tallied.”
The words hung between them.
Deo’s breath caught. He knew he had spoken too far — pride had stripped his cover bare. His fingers clenched around the cup, knuckles whitening, jaw set hard. He lowered his eyes to the broth but did not lift the spoon.
For a moment he seemed to force stillness, but the heat in his face betrayed him — anger, shame, the helpless fury of being seen where he should not be.
Romé caught all the flickers in Deo’s face — the pride, the slip, the cornering. His devil’s smile shadowed behind his hand, still holding the spoon.
“Or, Cavaliere — do you have another secret to protect?”
The healer flicked her hand, subtle, almost careless — but the air steadied around it, as if the floor itself had been reclaimed.
Her voice followed, calm as running water.
“It is not wrong to have secrets. Everyone does. Even a bird… guards its nest.”
The tension didn’t vanish, but it bent, redirected.
Romé lowered his eyes into his bowl. Something in him softened — a quiet he rarely allowed himself.
Deo didn’t press. He picked up his spoon, resumed eating, though his jaw held tight.
And for the rest of the meal, no one raised a voice.
Even the fire folded itself smaller, burning close and low.

Romé cleared the table after the meal, as he had the night before. He carried the bowls out into the passage; Giuliano would wash them come morning. Romé had learned he didn’t have to. Still, he didn’t return soon.
At the hearth, Dania began steeping her night-calming tea.
Deo asked if he might help. She only gestured lightly toward the brazier.
So he bent to it, as if tending embers were no less serious than parsing parchment. He cleaned carefully, lifting ash without letting it rise, steady hands moving with too much precision for such a humble task.
It took time before he realized Romé was there — standing beside him, silent.
When their eyes met, Romé spoke.
His voice was low — not soft, but sincere.
“Signore Deonardo.
I’d like to apologize.
Not for asking.
But for… thinking it was mine to ask.”
He didn’t wait for an answer.
Only inclined his head — to Deo, to the healer, to the silence — and withdrew to his corner, the one where he had spent the day with a book.
His hands found the book again, but did not open it. His gaze fell instead into the shadow, somewhere farther than the page could reach.
And there the flames grew quietly — under Deo’s hands, in mismatched gestures, in glances held too long.
In the space between two men who had not yet decided what they owed each other.
I curled into my basket.



📖what was seen 🕯♛🐾the apology

The healer did not speak. She poured the tea slowly.
The fire burned higher, warming the cushion beneath my fur, its light too sharp, too restless for the fragile stillness in the room — a stillness already close to breaking.
Romé’s gaze had turned inward. Whatever cracked in him did not reach for help. He carried it alone, as though the grief belonged only to him. It was the grief of someone he had once borne as far as his own body could go, and now refused to place in anyone else’s hands.
That was when his beauty changed. It was no longer the charm of youth, nor grace, nor even strength. It was the beauty of a man who did not run from sorrow. A man who bled quietly, in private, without making the world responsible for it.
The firelight drew shadows across his arms, slow lines moving over his skin, dark against darker, soft against the bones.
His eyes, never bright, now carried the weight of a stormed-over sky. Clouded, but alive. Not absent — enduring. Weathering.
Across the fire, Deonardo sat with the posture of study — but it was no longer about the books. His eyes were fixed on Romé.
Not boldly, not in a way to be caught, but in the quiet manner of a musician listening to a sound they don’t yet know how to play.
Deonardo did not understand the apology. Not fully. It didn’t fit the systems he trusted. His world ran on structure, on reason, on balance. An apology, to him, was a ledgered thing: a fault, a price, a redress.
But this was not that.
This was warmth held in Romé’s stillness, sorrow bound in restraint, dignity folded into retreat.
He had not apologized for asking, but for presuming it was his place to ask. For reaching toward something that was Deo’s to guard, without knowing what it cost.
In Deonardo, something stirred.
Not in thought, not in memory, but in his body. Something that did not belong to reason or plan. Perhaps even he did not notice, so wholly was his attention caught by Romé.
It began in his fingertips. The skin seemed thinner, smoother. The knuckles stilled, the bones retreating from their hard edge. The wrist softened into a line more fluid, as if light itself had found a way to move through it.
Not a full change, but a blooming. The line of his jaw blurred, the breath sank lower, slower, the rhythm altered. The bones of his face remained, but the tension left them. And with it, the voice of the man seemed to slip down beneath the silence.
Even without words, it was perceivable in the change of his breath.
And she appeared.
Something ancient. The shapeshifters. 
My whiskers twitched — I have met one of them every century or two. They are lonely, always lonely in themselves, because they cannot remember who they are. They live longer than demons, yet hold memories shorter than humans.
Deonnardo’s body was shifting from a man to a woman, as the same time remaining the same person.
Dania saw it too, though her gaze did not sharpen. She never stared; she received. Her eyes rested on him just long enough to know, then turned back to the fire, to the simmering tea.
Romé alone did not see. His gaze had folded inward, far off, where grief kept its company with silence.
She emerged from the scholar’s body — neither young nor old, but a woman at the very moment of life. Mature enough to grasp her own joy, unafraid to let longing lead her gaze, her curiosity moving as freely outward as inward.
Her skin carried a pale rose undertone, catching the firelight in warmth. Her eyes, though set in shadow, seemed brighter than the room — irises large and dark, edged with a limbal ring sharp as ink drawn around water.
Her hair had loosened in the shift, silver as tide under night, the spray catching faint starlight. It fell messily across her cheeks, strands shining like threads undone.
And her body — still the same shape, yet no longer held by resistance. The effort was gone. Every line softened, every gesture unguarded. She was not pretending, not hiding, not even arriving.

Her gaze did not turn to herself.
It rested on Romé.
There was longing in it, but not the kind that flares and fades. It ran deeper, threaded with tenderness, with desire that was not shallow, not of the surface.
It was the gaze of someone who had carried absence too long, and found, for a moment, a body she could not look away from.
Romé did not stir. His grief held him elsewhere.
It was Dania who moved — steady, as she always was — bringing a cup of tea toward her.
And then I saw it.
Deonardo, in his form of woman, lifted her eyes to the healer.
It was not the same look he had given her before. Changing shape does not change the mind, nor the memory. But the body… the body has its own language. And hers spoke differently.
Where the man’s gaze had been measuring, cautious, the woman was open, unguarded. It was a small thing, barely a breath. Yet I saw it. The same hands, the same memory, but a gaze no longer shaped by a man’s armor.

Dania moved with her quilted steps. She set the cup on the side table. Then she looked. Into her eyes, across the shoulder, even into the hollow where shadow gathered behind the ear.
Two nights ago, when Romé bent under grief beside her brazier, Dania’s gaze steadied him, gave him ground. It was a gaze that asked nothing, only stood beside the weight.
But now her eyes were different. They lingered, drinking in. A tender curiosity, careful and deliberate, tracing each line of the face, the fall of silver hair. She received the sight without hesitation. As if beauty were nothing to question, nothing to explain. She simply took it.
And the woman looked back.
The scholar’s mind still ticked, wary, always afraid of exposure. But the body — the woman’s body — knew no such fear. It rested in the gaze, breathed into it, and for that moment allowed itself to be seen. I felt it ripple through her like a cat hearing its own purr for the first time:
She sees me beautiful. 
Between healer and shapeshifter, something wordless held.
Dania’s hand lifted, almost weightless. Her fingers touched the silver hair first, brushed the cheek as if testing whether it was real, then moved slowly down — under the jaw, along the hollow of the neck, pausing at the collarbone.
The woman stilled, only breathed into the touch, as though her body had been waiting for it far longer than her mind could admit.

Dania’s fingertip lingered a moment at the hollow beneath the collarbone, then began its slow return — upward, along the line of the throat. The skin shivered under her touch, softly, fragily, the pulse beneath steady yet trembling. 
Each inch felt stretched, as though the healer’s finger thickened time. The woman’s head tilted back, following, lips parted in a breath that did not know if it belonged to longing or surrender.
At the jawline her finger paused, then lifted away, and the air itself seemed to hold its breath. The tension did not break — it folded, as if pressed back into the body before her. Slowly, the softness began to retreat. 
By the time dania’s hand had fallen to her side, it was Deonardo who sat before her. The man, silver-haired, his gaze unsteady — as if the woman had not vanished, but merely stepped deeper into him.
At that moment I noticed Romé. He was there, quiet in the shadow — so quiet I could not tell when he had begun to watch, or how much he had seen. His stillness held no surprise, no question. Nothing startled him.
It was as though his presence had always been part of it — unspoken, yet already belonging.

The night fell deeper. 



📖 November 9, 1264 Sunday
 high Mass 🕯♛♡🐾 Sunday, church square,  Bright morning
I followed Romé before the bell. He slipped from his window as if he had done it all his life — a hand in the grapevine, a foot to stone, then he was on the guesthouse roof.
The medlars still clung here and there, brown and wrinkled in the cold. He turned one in his palm, peeled it slowly, let the scent rise, then ate the flesh segment by segment with the born grace of his fingers.
From the roof, he could see three levels of terraces at once.
On the highest one stood the cloister, novices swept mud from the walks, their brooms rasping dry against stone. Others lit lamps along the arcade, so the passages glowed amber even in morning frost. 
One level lower, on the church square, lay brothers hauled benches into lines.
The lowest terrace by the main gate, they held back the crowd, leading peasants and villagers to load them goods in orders.
 Romé chewed the last of the medlar. His eyes fell and caught on Deo.
Oh, I knew it -  in a flick of my tail, nothing more. 
Deonardo had risen earlier than most. His breakfast finished, he dressed in plain black — not Aurelia’s blue, but the color he reserved for humility at Mass. Against it, his silver hair shone sharp in the morning light, drawing eyes even as he tried for discretion.
By the time he reached the church on the middle terrace, the rhythm was already alive.
Inside the nave the choir monks rehearsed, tones clipped, testing for breath. A novice polished brass candlesticks until they caught the sun, another filled the lamps with oil. At the side, two monks weighed frankincense and myrrh, sniffing their smoke to be sure it rose clean.
The sacristan spread the missal, set out the vestments, the Gospel book resting in its carved case. Pietro, the cellarer, bent over baskets of offerings: bread, eggs, honey, herbs, even coins. Villagers always left them on Sundays — a flow of generosity and duty mixed together.
It was there Deo saw him: a boy, no more than ten. His tunic too thin, eyes hollowed. One of the war-orphans trailing with the Milanese pilgrims — handed from priest to priest, still half lost. He hovered near the baskets, staring at bread and honey, unsure if such things were meant for him.
Deo crouched, spoke a word or two — but the boy’s dialect tangled, mountain-born, broken by fear. He did not answer. His hands stayed empty.
So Deo did not ask again. He took two eggs, wrapped them in cloth with bread spread with honey, and pressed the bundle into the boy’s palms.
The boy bowed, awkwardly, then bolted for the door, clutching as if it might vanish.
Deo straightened, thumb still sticky with honey. For a moment his gaze lingered on the boy — thought half-formed, half-dissolved.
The first bell rang.
Romé, still high on the roof, saw Dania crossing the cloister garden, moving toward the stair-tunnel that linked cloister and church square. He leapt down, light-footed, and met her at the gate. I sat on his shoulder and saw her smile, the smallest curve — for both of us. They did not speak, only walked. Romé one step behind her.
Near the church they met Deonardo, just as the second bell began.
Inside him was storm — after last night, the conversation at the healer’s dinning table, how it ended up with Rome ’s apology, the appear of his full woman’ form before Dania, and her hand upon him. Insecure, doubting, confused — rehearsing countless ways to meet them again, only to fail them all. Now, thanks to the bell, he did not need words.
Yet words were not the first thing he noticed.
It was scent.
On Romé: bitter, fresh medlar.
On Dania: the profound, bodied trace of essential oil extracted from medlar peels.
For no reason, Deo felt safe between them. Accepted. Belonging — whether shapeshifter or whatever he might be.
When the bell ceased, Deo inclined his head.
“Donna Dania. Signore Romualdo di Orellana.” His greeting formal in phrase and softened in voice.
Romé’s lashes lowered, a shadow falling over his eyes — Deo named him as fathers and abbots, not the one he lived by. Dania’s gaze passed between them, soft but unwavering. She answered nothing, only let a brief smile steady the air.
Deo’s shoulders eased, almost despite himself. His hand, still faintly sticky with honey, loosened at his side. None of them spoke further. Yet all three felt it: something had shifted — his guard lowered, unspoken, undeniable.
By then most people were already inside or pressed at the church doors. The monks left their work in cloister and refectory, filing into their stalls. The air grew dense with waiting.
When the bell ended its long peal, they entered together — Romé, Deonardo, and Dania. Not announced, yet the nave tilted toward them all the same.
Romé and Deonardo: both strangers to the valley, both carrying a presence the peasants had no words for. Their garments — black, clean, finely cut — struck like banners in a room of rough wool. And with them, Dania — the quiet center of attention every Sunday High Mass — her presence a steadying chord.
I could feel it in the eyes around them.
Peasants shifted on their benches, nudging shoulders, whispering without sound. Merchants’ wives let their glances linger longer than they meant to. Even the pilgrims, hollow from the road, stirred as though brushed by something stronger than hunger.
There was a discipline in their movements, visible even in Romé’s casual grace. The mountain folk noticed it at once. Beside Deonardo, Dania’s strangeness — always a little apart in crowds — seemed, for once, to make sense. Though no one could have explained why, they felt it all the same.
The unmistakable aura of nobility: of men raised in houses where both sword and book decided life.
They did not belong.
And that was exactly why every gaze clung to them.
It was awe — and unease.
They had chosen no prominent place. Romé sat easily, dark lashes casting shadows, the curve of his body careless and alive. He let attention wash over him, and though he did not smile, his stillness teased that he might. Deo bore it differently: his spine too upright, his scholar’s cloak arranged too neatly. He lowered his gaze to the missal in his hands, but the stiffness only made him more noticed.
And Dania — she neither absorbed nor deflected. As on any Sunday, her presence was simple, her plain dress unadorned. Yet every movement seemed to quiet the air; the eyes that lingered on her found no place to hold.

Then came the Rinaldeschi.
Across the nave, when Ser Rinaldo Rinaldeschi and his wife Tessa Rinaldeschi entered, the air shifted again. He was no noble, yet carried a polish the peasants read as courtly. His loyalty to Bologna’s lords, the elegance of his speech, even the neat fold of his cloak — all marked him as a man of higher order. His wife, plain but steadfast, walked at his side. She was known in the valley, her prayers a regular sight. They were not strangers; they were example.
So the room held two centers of gravity.
One — Romé, Deonardo, Dania: strange, unsettling, unmistakably other.
The other — the Rinaldeschi: familiar yet elevated, pious yet commanding.

Then the small bell rang, a brisk, silvery burst from the sacristan’s hand. At once the nave hushed.
From the choir door the abbot entered, flanked by the officiating monks. Their steps were measured, their chant low — Kyrie, Christe, Kyrie eleison — voices weaving into the smoke of incense. The peasants rose and bowed their heads. Merchants folded their hands. The pilgrims knelt where they were, weary bones sinking to the stone floor.
And so the Mass began.
The chant rolled forward, carried by voices more than words. Kyrie, Christe, Kyrie. A rising and falling, like breath given back to heaven.
I curled on the beam above the south aisle, watching.
Romé did not bow as the peasants bowed. He moved along, while his eyes closed, the chant brushing over him like wind against a cliff. Yet something in him followed the music, as if he could hear a rhythm behind the rhythm.
Deonardo knelt when the monks knelt, stood when they stood, folded himself into every gesture of the Mass. He was precise — spine straight, lips moving with the responses, hands crossed in careful form. 
And Dania — she was neither rigid nor careless. When she bowed, it was as if she bowed to all of them at once: the altar, the poor, the monks, the very stones. When she stood, her stillness steadied the air again, the way a flame steadies smoke.
The Mass moved deeper. The Gloria, the readings, the sermon. The abbot spoke of charity and of wars that had torn children from bread, lands from their lords. The peasants listened as if the words came from another country. Yet when his voice fell, silence pooled thick in the nave, as if no one dared disturb it.
When the Pater Noster rose, it was like the valley itself groaning upward: voices uneven, dialects broken, yet bound together in one plea.
The nave held its breath.
Above the altar, the abbot lifted the chalice, sunlight striking faintly on the rim. The choir answered him — Sanctus, sanctus, sanctus — voices high and thin, threading into the vaults like cold birds. Bells trembled again, sharper this time, and the sound seemed to pierce bone.
The people knelt longer than the words required. Peasants in patched tunics, merchants in dyed wool, pilgrims dusted from the road — all pressed together on the stone floor, foreheads nearly touching benches. The frost of the mountain air seeped through the walls, but the nave was warm with breath, with smoke, with the scent of rosemary clinging to rough sleeves.
Romé’s gaze wandered then — not to the altar, but sideways, toward Deo. He did not stare long. Just long enough for me to see it: the weight of last night’s words still resting between them, quiet, unresolved.
Deonardo did not look back. His eyes stayed fixed on the missal. But his fingers on the page trembled, faint as a leaf in wind.
Bread and wine were lifted, slow arcs of brass and linen, the abbot’s voice intoning mystery. The crowd bent forward as one, heads lowered like fields under wind.
I slipped closer, tail brushing the beam, so I could see their faces.
Romé did not kneel when the bell chimed. He stood, gaze turned upward. It was not defiance. It was something stranger — as if he were listening for a God who did not live in stone or words, but in the hollow between heartbeats. His lips parted slightly, not to speak, but to breathe.
Deonardo bent low, every gesture exact. His hand touched his breast, his head bowed until silver hair veiled his face. When the bread passed, he received it with rigid care, lips pressed tightly, swallowing as if he feared the host might catch in his throat. His eyes closed a moment too long, not in devotion, but in struggle.
And Dania — when the cup was lifted, she bowed, did not reach to receive. Her eyes clear, did not kneel nor turn away. She stood with a quiet that was its own kind of offering. Around her, the air eased, and even the peasants’ glances fell soft for a moment.
The bell chimed again. Smoke wavered. The mystery folded itself back into silence.
It was in that silence that I knew: they were seen, all three. By the abbot, by God, but by the crowd. Seen, and measured, and placed into the hidden ledgers of the valley’s memory.
The abbot’s blessing fell over them, his hand cutting the air in sign of the cross. The choir sang the dismissal, their tones falling like the last light of day.
And then it was done.
I saw the peasants shift in their benches — some eyes hungry, some patient, some too proud to look at food so openly.
Children whimpered, hushed at once. Old men coughed behind their hands, leaning on staves. Women crossed themselves, lips moving in prayers half-whispered, half-remembered.


The three rose with the rest and moved out toward the square. Eyes followed them — peasants glancing up from their alms, merchants’ wives pausing in their whispers. But none stepped forward, none dared approach.
Only one did.
Filippa Basili — nineteen, bright green eyes, her brown-golden hair bound beneath a light veil. She looked pretty even in restraint, her coat plain but newly made, a lace collar showing at the edge of cloak and gown. She left the circle of merchant women without hesitation, her bearing steadier than her age, and crossed the threshold alone.
She reached them at the church’s outer stairs.
I flicked my tail in amusement. Ah, a Basili girl — bone of adventures, even when wrapped in wool. The spirit of her house walks in her, humble and discreet on the surface, but green-eyed boldness flashing beneath. The others hushed and held back; she simply stepped through the silence as if it had been waiting for her.
Romé’s lashes lifted, eyes narrowing in quick curiosity. Deo straightened, the scholar’s cloak tugged neat across his shoulder. Dania alone did not shift, but in her stillness she seemed to make space for the girl, as though she had been expected.
She bowed first to Dania.
“Madonna, my mother’s sickness has eased more from your jars than from any doctor’s draughts. I thank you for it — and for the way you stand… independent.” Her eyes lingered, open and admiring, not as courtly flattery but as plain recognition.
Dania smiled fondly, accepting the girl’s words without deflecting them, without modesty or excess.
“If what I keep has helped your mother, then that is its purpose. You may always ask me for what she needs.”
Then she turned to Deonardo. She inclined her head, more deliberate now, as if she knew the risk of presumption.
“Cavaliere. They say you are a scholar from a high court. I am Filippa Basili, and I help my father with the apothecary records. If it pleases you… I would be glad to know your name.”
Her words were clear, brave — too brave for some. A girl speaking in public, in front of merchants and monks. But she did not falter.
Deonardo bowed, delighted, his voice layered — courtesy shaping respect, restraint edged with appreciation.
“Deonardo di Alferini, in service to my Lady of Vercado. The keeping of records is no small art — too many forget that without ink, even healing vanishes. You do your father honor.” His glance held hers gently, honoring her courage — then slipped aside, as if he dared not give too much into it.
Her eyes brightened, though she lowered them quickly.
“Then I thank you, Cavaliere.”
She turned to Romé at last. Not shy. Her gaze held him openly, unafraid to meet the beauty others only whispered of.
“And you, signore — people speak of you already. They say you walk with the calm of a knight but the face of a poet. If they are wrong, I will gladly be corrected.”
Romé’s smile landed to her green eyes, light as a spark. He bent his long frame just enough to meet Filippa’s gaze on level ground.
“Then let no rumor be corrected — you saw what you saw. A knight’s calm, a poet’s face? I would gladly live with such a judgment. And if you wish, signorina, we might trade words for verses someday, as merchants trade wine for oil.”
His tone flirted without weight, playful, as if her boldness itself pleased him.
“If you invite me, signore, I will not refuse. But I warn you — my father trusts me with ledgers, not dances. I walk with merchants, not with poets. If you still wish for my company, then I will meet you on the street of apothecaries, when I return to Valdiponte.”
Romé’s smile curved sharper, unoffended. He shifted his weight lightly on one heel, as if turning the whole exchange into a dance-step.
“Then it’s settled. I’ll bring dust on my boots, you’ll bring your ledgers. Let the apothecaries look on — they’ll see how trade and poetry strike their bargains.”
The girl’s eyes brightened — not softened, but alive with the thrill of being taken seriously. She bowed, then slipped back into the crowd.
Dania’s gaze followed her. Not stern, not indulgent, but weighing. The girl’s bravery stirred something protective in her — Basili’s daughter was more than an admirer. She was a flame too young to know how easily wind could snuff it out.
Deonardo did not watch the girl leave. His hand, resting on the fold of his cloak, tightened imperceptibly on the fabric before he let it go. What stirred in him was not only envy for Romé’s ease — it was envy for the girl herself: that she could step forward so simply, so boldly, and be received without cost.

Filippa’s steps had only just faded into the cloister when the bell struck.
Not quick, not summoning — but slow, heavy, once, then again.
All the three squares rippled with unease. Peasants froze beside their carts; a mule brayed and was hushed. Even the mercenaries under the tower stilled for a breath. Men glanced at one another, waiting for someone to name what it meant.
Romé’s ease — the lightness he had just carried with the girl — dissolved at once.
Deonardo’s gaze lifted: first to the bell tower, then the cloister, then higher still toward the chapel, then the birds wheeling there — his mind wandering almost aimlessly with them.
Only Dania did not search. Her voice was quiet, steadily low:
“Somebody passed away in the infirmary. Must be the pilgrim.”
The bell tolled once more.
The square gave no reply — only silence, as if the sound itself had folded the air shut.

📖The Burial of the Pilgrim 🕯♛♡🐾

Nov 9, Sunday afternoon

The bell still echoed when Dania turned beneath the stone arch of the tunnel toward the cloister.
Romé and Deonardo followed in silence. I padded after, paws soundless on the frost-dark floor.
The infirmary air was heavy with rosemary and smoke. Two monks had laid the pilgrim on a wooden bier, limbs bound in a rough shroud. A third monk lifted the wooden cross, its grain dark with years of use. No words — only hands, adjusting, lifting.
Dania paused at the bier, her hand resting once upon the shroud — not blessing, not farewell, only presence. Romé stepped forward, his long fingers taking hold of one side. Deonardo mirrored him on the other. Together with the monks, they bore the weight.
Through the cloister walk they moved, steps muffled on damp stone. Novices lined the walls with lamps, smoke trembling in the draught. Two went before with candles, and the infirmarer followed with his book, chanting low: Go forth, O Christian soul, from this world…
They passed the rosemary garden, the green needles breathing sharp into the cold. Then into the tunnel, the mountain pressing silence on their shoulders.
When they emerged onto the church square, the crowd stilled. Peasants drew carts and children aside, mercenaries shifted their boots under the bell tower. Even wine-skins fell quiet.
Downward they went, past the guesthall where travelers paused mid-meal, through the outer court where mules stamped at the scent of wool and earth. At the gate chapel, the priest who had walked with the pilgrim waited barefoot, his face pale with cold. Behind him trailed five, six children — uneven ages, clutching scraps of cloak or a wooden cup as though they were treasures. They pressed close to the bier, unwilling to let it go.
Beyond the gate, two monks with spades had already broken the soil. The earth lay in dark heaps against the frost. The bier was lowered. Hands released slowly, as though weight lingered even after wood was gone. The chant rose thin, then steadier, echoing against the mountain.
When the last note faded, one of the older boys bent suddenly to the grave. His thin hands tugged at the laces, pulling free the shoes from the shrouded feet. He held them out to the priest, voice cracked but clear:
“You’ll need them.”
The priest shook his head once and set the boy’s hands back on the leather.
“They belong to him now.”
The boy froze, clutching them, torn between refusal and obedience.
Then Dania’s hand moved. She touched his wrist, light but firm, her eyes lowered to his. Slowly, he laid the shoes back upon the shroud.

Then the first earth fell, heavy, cold.
Romé stood back. He had carried the bier, but when the grave opened he did not step closer. His face was turned, profile cut sharp against the frost air. Not indifference — only that public sentiment was not his way. Whatever he felt, he kept folded inside, where no one could touch it.
Beside the grave stood Tessa Rinaldeschi quietly in the group, the only merchant’s wife.  When the earth settled, she bent to the older boy — the one who had tried to return the shoes. She spoke softly, words I could not catch. Her eyes lifted once toward the priest, a glance of both question and concern. Then she turned to the gate, speaking softly with two monks she knew by name. By the time the priest rose from his knees, a pair of worn but serviceable shoes had been found in the storeroom. Several cloaks too, patched but warm, and food for the children laid aside in baskets.
The orphans looked at her with wide eyes, still half-starved, half-afraid. And in their glance I saw it: they had understood her already as what she was — a mother.

Deonardo stepped past the children, toward the priest. His voice was courteous, but carried weight. The priest, weary, told what little he knew about the buried man: two weeks before they reached the monastery, he had found the man wandering, already sick, feet bare and near ruined by frost. His mind was not clear — war’s shock or fever had clouded him. He never gave a name. The priest had given him his own shoes, hoping the man might last the road. He did, barely, until this morning. Now his body lay in the ground, the shoes still on his feet, while the children pressed close to the priest as though he might vanish too.
Five of them, orphans from villages broken by war, they had fallen under the priest’s care one by one. He had started alone from Milan; by the time he reached these valleys, they followed at his heels like sparrows after a plough.

Dania stayed while Deonardo walking back to the church. She knelt a little to the older boys. The small packet of paste rested in her hand, her voice low, steady, naming herbs and winter fruits. 
“Listen. For hunger — look for chestnuts. They will still lie under the leaves. Roast them on coals. Rose hips — red on the bushes — you boil them into tea. Hawthorn, the same. If you find pine cones, crack them; the seeds give strength.
For fire — keep pine resin or bark in a cloth. Even wet, it burns quick. Birch, if you pass it, lights fast.
For sickness — if fever comes, boil yarrow, or plantain if you know it. Put a finger of this paste in the cup. For wounds, wash with water, then mix resin with paste, press it to the cut. For cough, boil pine needles. If you have honey, add it.
For feet frozen — wrap them in wool, warm them slow. Never put them to the fire. That kills the flesh.
Do not touch mushrooms. Do not waste your strength on what you cannot know.”
She placed the packet into the oldest boy’s palm, closing his fingers around it. “This will be enough, if you use it sparingly.”
The boy listened hard, lips tight with effort, as if he swore to remember every word. 
Upper terrace, in the church, the incense still clung to the air, drifting out of the nave with the last psalms. People spilled slowly from the church — peasants carrying baskets, pilgrims whispering prayers, merchants waiting for their turn at Lorenzo’s smile.
Deonardo kept to the cloister edge until the flow thinned. Then, with the measured step of one who had chosen his moment, he moved beside the abbot.
“Reverend Father,” he began, voice low enough that it didn’t claim the whole walk. “I saw the children by the grave. Orphans, taken in by the pilgrim priest. Their shoes worn. Their hunger—” He paused, thumb brushing the edge of wood pray bend,  “What struck me is not only their need, but the chance of prevention.”
Lorenzo’s eyes sliding toward him. Not yet reply — only listening.
Deonardo continued:
“Along the road, there could be stations — small, simple, nothing more than stone shelter. Within, space for what others leave: old shoes, bread, jars of oil. With the monastery’s seal upon it, men would honor it. Take only if they must. Give, if they can. Such stations could be raised where the paths are hardest — above the ford, at the ridge, at the pass.”
He stopped there, not pressing, but the idea hung.
Lorenzo’s hand brushed his sleeve lightly, “shelters for the poor — yet not built by monks, but offered by those who pass.” His tone warmed as if weighing it aloud. “It is not the first time the thought has reached me. But you give it shape.”
His gaze turned toward the mountainside where the road bent away. “Stone and timber can be raised by coin. But trust — that is the true foundation. If men believe these stations sacred, they will guard them, not plunder.”
He looked back to Deonardo with a smile that landed exact,  “And perhaps you, Cavaliere, would wish to set the first stone?”
Deo inclined his head, tone clipped but clear.
“If San Michele is willing to take the work, and let its seal stand upon it, I will see the craft supplied. Stone, timber, and the mason’s hands. It need not bear my name. Let it be the monastery’s gift, as all will trust it so.”
“Then it shall be spoken of at chapter. If my brothers assent, the seal will be set.”
Nothing more was needed. Their pace resumed, the sound of peasants still carrying through the square.
And then Romé passed. 
Romé had not spoken since the burial. He kept apart, seeing Dania stooping to the children, but did not step nearer.
His eyes lingered only a breath, then turned away. He crossed the terrace, the shadow of the bell tower falling long beside him.
In the church he saw Deonardo speaking with Lorenzo — his silver head bent, hands moving in that deliberate way when he argued something meant to last. Romé did not go closer, only stood behind bend a moment, watching their figures framed by the pale light. Then he left without a sound, returning to the cloister garden, not his chamber in guesthouse — but toward the healer’s garden. Her door stood unlatched, the rooms quiet. He entered alone.

The sunday supper
The air still held the bitter-sweet of medlar peel essence. Every Sunday in winter was a day for citrus aroma. 
I leapt from his shoulder to the floor, my paws soft on the stone.  
Rome started to fill the braziers, warming the space.  
Before sunset Matteo appeared, arms full. A basket of fresh bread still warm from the oven, cured bacon sliced thick, a bottle of wine sealed with red wax. A jar of berry jam, a bowl of winter greens crisp with vinegar.
Later Dania returned from her rounds, the herbs clinging to her cloak. Romé was already settled, sitting where the shadows were least crowded. And then Deonardo — cloak loosened, silver hair catching the firelight. He glanced at the table, at the jars and herbs along the wall, then to her.
“If it pleases you, Madonna,” he said, his voice gentle, “may I borrow your kitchen tonight? Let me prepare supper — for the three of us.”
Dania set her basket down, her gaze resting on him for a breath. Then she turned, not to answer him yet, but to Romé, still in his corner.
“And you?” Her voice was calm, almost teasing. “Would you mind eating something finer than my old bread and soup, for Sunday’s table?”
Romé’s eyes lifted to hers, a shadow of a smile crossing his face.
Deo set about the kitchen as if it were a scriptorium — every motion precise, deliberate. He laid out in order: the bacon trimmed neat, the bread cut in even slices, the berries pressed and spread into a small dish like stained glass.
But precision was not all. He lingered over little touches: laying rosemary sprigs on the pork rind so the smoke carried their sharpness, pouring the wine firstly into a clay jug to let it breath. Even the salad — a poor man’s thing of greens — he arranged by color, pale leaves cupped beneath darker ones, a scatter of nuts brightened with oil.
It was not grandeur, but a kind of beauty with care, as if he thought the meal itself could keep the world from fraying.
While Deo worked with fire and knife, Romé moved with that careless grace of his, set three glasses down, lifting the wine, filling each in turn, the red catching firelight as it rose. One glass, then the next, his long fingers lingering at the stem before passing it forward.
Dania, meanwhile, crossed to a cupboard. From within she drew a strip of table linen, frayed at the edge, and spread it over the rough wood. On it, what she displayed for candles not a plain taper, but something more deliberate: a hollowed orange shell, dried and filled with beeswax. She set it upon the cloth, flame kindling in its hollow. It burned with a brighter, steadier glow than common candles, and the air warmed at once with faint citrus sweetness — a warmth carried both by fire and by fruit.
So the table was shaped — wine, bread, fire, and orange light — each detail touched by a different hand.
It was the Sunday before St. Martin’s Day. Nothing special this year. The square outside still full, the whole monastery stirring, but here in her rooms the air held still.
Three of them sat at the table — no more than a few days together, yet as silent as if they had eaten side by side their whole lives.
On the bench, the Cavaliere had set my cushion. Beside it, a small bowl of milk. So I dined too.


📖 November 10, 1264 Monday
The Day of Rents🕯♛♡🐾morning

For the sake of last night’s supper, I granted Deonardo an honor —  curl on his own chamber’s couch. A noble bed would have been too cold, but the couch with my cushion was just right. I kept him through the night, and when he stirred at dawn, my whiskers twitched first.
Luca was there before the frost had even lifted from the panes. He knocked, then slipped inside with the basin steaming, clean linen folded, boots polished so neat I could see my ears in them.
Deo stood at the washstand, sleeves rolled, silver hair falling loose before he bound it back. He shaved with care, while Luca moving about him, placing cloak and garments in order.
When his hands were dry and he reached for the comb, Deo spoke:
“What did you hear from the mercenaries?”
I remembered then the last two days. Luca hanging about the square like a boy caught by tales about dragons — drawn to steel, dazzled by swagger. He asked them of their battles, their pay, their wounds. A game of questions, yes, but I knew better. Every word he brought back was for his master. Curiosity was cover for duty.
Now, Luca straightened, report in hand like a soldier with his blade.
“First — Florence,” Luca began, shoulders square.
“Martino — the one with the feud-colors. He said he was born in Florence. His father had a little vineyard outside the city, but when the war with Siena came, it was taken. His father was killed. His mother married again, poor.
He talks a lot, always loud, quick with his words. Gets angry fast, even when nobody means it. But he listens too — sharp. Always wants to hear what the others know.
He fought on the Arno last summer for Florence. Says they will push further this winter. He said Florence means to widen its reach, no matter the ground.”
Deo’s comb slowed, teeth catching for a moment in his hair. Then moved on, smooth again.
“Second — Pisa and Genoa. Two men back from Sardinia cursed their pay. Genoa cut their ships loose, left them stranded. They spat at Pisa’s name. Said the sea is no longer Pisa’s to hold.”
The comb moved steady, silver strands drawn flat.
“Third — Bologna.”
“Some from the east side, hired there. They spoke of Lord Corrado della Rocca and his swordsmen. Said Bologna calls them when its neighbors are too strong. They never wait — not for banner, not for trumpet. You follow or you fall. Even allies fear them. And…”
Luca faltered, not sure if it mattered.
Deo’s eyes narrowed in the mirror. “Anything else?”
“Signore Romualdo stopped when they spoke of Corrado. He seemed… interested.”
Deo set the comb aside and bound his silver hair back, the black ribbon neat against the light.
“You listened well.” His eyes rested on Luca with a sign of approval, so faint it might have gone unnoticed. “Are you practicing every day?”
“Yes, my lord. Fencing — and the writing.”
Luca bowed his head slightly. Deo nodded once. Luca gathered basin and cloth, as neat in his retreat as he had been in his service.

From Deonardo’s chamber I slipped out, paws light against the cold stone, into the cloister garden.
And there — what a sight for San Michele, so seldom seen.
Women, everywhere.
Not the veiled figures of nuns, not the rare visitor trailing behind her husband. No — this was different. Fresh peasants’ daughters, cheeks still red from the road; the merchant girls, their cloaks smelling faint of spices and smoke; even widows in black wool, their laughter too quick, too bright in this place of vows.
For three days in the year, the monastery seemed almost to breathe with another kind of pulse — softer steps, brighter eyes, shapes the monks pretended not to notice.
And Romé — he was in the middle of it like a flame in straw. Leaning against the cloister wall, half-smile curved, speaking low enough to make them fall closer. His beauty did not have to work — it simply drew. They gathered around him as though he had been waiting there just for them.
Among the bustle, a widow slipped from the crowd. Her cloak hung black, but the body beneath it moved like a candle’s flame in draught — soft, quick, licking at the edges. Her hair gleamed dark red where the light touched, her lips curved as though secrets were their natural shape.
The others laughed in clusters, voices rising like hens. She walked alone, her steps led straight to the east wing of the guesthouse, where Rome’s chamber at the far end. 
I padded after, light enough to leave no echo.
The door closed. Then, one by one, the shutters. Even the heavy boards outside pressed tight.
No sound leaked.
The monastery was left outside, restless.
The church square had become half-market, half-courtroom.
Stalls lined the stone — most arranged or donated by the shopkeepers of Valdiponte, who knew well that San Michele’s rent day was worth the journey.
On one board peasants poured out grain to sell, on another they bought what they could not make themselves: cloth and salt, iron tools, ribbons, dried figs. For many families it was their single chance to trade beyond their own valley — a year’s worth of need, of hope, of hunger pressed into one ledger of exchange.
Above them all the bell tower loomed, its shadow stretched long across the crowd, measuring each bargain like a sundial.
At the lower gate peasants pressed forward with carts and mules, their loads heaped high: sacks of grain, bundles of flax, barrels sticky with oil. The first terrace bristled with order, roped into lines as Rinaldo had demanded — sacks to the left, barrels to the right.
His men moved at double pace since dawn. Two at the measuring, one calling figures, another setting them down on the ledger sealed with San Michele’s red wax — the same seal Lorenzo had handed Rinaldo on Saturday.
Lay brothers rolled barrels into place, stacked the sacks, marked chalk tallies onto timber posts. Pietro’s men kept the rhythm, their shouts cutting through the cold.
Rinaldo strode the terrace, eyes sharp to catch a pause, a slip, a lag. One word, one gesture, and the rhythm steadied again.
At the far row he halted. Too many wine barrels, each pressed in a different house, each painted with a different hand. No seal, no standard. He frowned, then saw the Valdiponte vintner waiting beneath the shed roof. A nod, a step — he was there.
“This year’s harvest?” the vintner asked.
“Plenty,” Rinaldo said. His mouth curved, but not with joy. “More than last year. More than in the best of five. But each house presses its own. Every cask different. No mark of trust. In Bologna, they pay only for what they know. This—” his hand skimmed the row of staves, “—sells cheap.”
The vintner’s brow creased, then smoothed. “So you’d take the grapes before the press. Make it one wine, one name?”
Rinaldo’s glance was keen, approving. “At the yard the quality can be measured, the taste made steady. One seal, one label. Bologna pays more. Peasants still sell their fruit, but the value rises when the barrel carries the mark.”
The vintner scratched at his jaw. “It would mean more space. More vats, more casks. Some cost.”
Rinaldo gave a small nod. “You count stones, timber, labor. I’ll weigh Bologna’s price. If the margin holds, we speak again.”
Their eyes met — not long, not soft. Just the brief accord of men who had both measured profit before.

After the vintner, Ser Bartolomeo Alfani approached — Casa Maffei’s steward, tall, silver at the temples though his stride carried no age. A man whose life had been ledgers and tallies, he knew the rhythm of this week as well as Rinaldo himself.
“You’ve but one day this year,” he said without preface. “Will it be enough?”
Rinaldo gave a short nod. “It is already prepared. Brother Pietro had the monks marking and stacking since yesterday. By sunset, every sack and barrel will be in the cellar.”
His voice dropped, more business than courtesy. “How much does Casa Maffei require this year — more, or less?”
The reply came quick, without hesitation. “More. Our marble yards doubled their men. Pisa has orders running further than spring. Reckon thirty percent above last year.”
Rinaldo’s gaze sharpened, then eased. “It can be met. The tally will be done by nightfall. You may walk the cellar yourself and choose what is fit. Once chosen, I will reckon the price against Bologna’s market — fifteen percent lower, as in an average year.”
Bartolomeo inclined his head, satisfied. “The coin is ready. We can settle it around midnight, before the feast, if we work late.”
Rinaldo’s mouth tightened in approval. “Good. The abbot will be glad to have silver in hand. The mercenaries must be paid, or the roads will remain open to bandits. Two days hence I ride for Bologna. I want the road clear.”
Their accord ended without flourish. It was business old as trust, closed in fewer words than a psalm.

After parting from Bartolomeo, Rinaldo found Ferrazo beneath the bell tower, watching his men as a mastiff watches unruly hounds. The captain inclined his head — not bowing, but granting the respect of equals who had bargained before.
“You’ll want the road clean,” Ferrazo said at once. “Four days, till you’re past the ridges. After that, safe ground.”
Rinaldo’s eyes swept the square — peasants trading, carts crowding the stalls. “Four days,” he confirmed. “Five men at least, better ten. Not loud ones. I want the road held, not stirred.”
Ferrazo grunted. “I’ll choose them. The rest stay here till new moon.”
“And the measure?” Rinaldo asked.
“As ever. One third coin before, two thirds when you ride back. If you ride back.” Ferrazo smiled.
Rinaldo’s mouth curved, not quite a smile in return. “You’ll have it. The coins are counted already. After sunset, I’ll see them sealed.”
Ferrazo studied him a moment longer, then spat into the frost. “Then we understand each other.” His gaze slid toward the mercenaries lounging at the edge of the square. “I’ll tell them after dark. Less boasting, more obedience.”
Rinaldo gave a short nod, cloak brushing his boots as he turned away. No waste, no linger. His eyes were already on the next account.

Traveling merchants🕯♛♡🐾 afternoon 
By afternoon the square thickened again. Two wagons from Florence rattled in first, their canvas sides painted with faded crests. Behind them, mule-trains from Lucca and Siena, bells clinking with each step, and last a cart from as far as Milano, its wheels banded with iron so heavy it left grooves in the frost.
They were regulars — men who knew San Michele as a safe stop. Every month or two they passed through, never without a little trade. The monastery was no fairground, but its market day drew enough buyers to make a halt worth their while.
They knew the rhythm: arrive on Monday, when peasants already gathered for rent, and the square would be full. They’d sell off goods before Mass tomorrow, lighten the load for the next mountain pass.

The wagons had scarcely settled their wheels before the women of Valdiponte gathered. Merchant wives, shopkeepers’ daughters, even a few of the better-dressed peasants pressed close, fingers grazing silks and copper pans, glass beads strung like dew. Their chatter rose high over the mule-bells, bright and quick as sparrows.
Rinaldo moved through it as though through water. A word here, a smile there. Never lingering too long, never careless. Heads turned. Voices dipped, then lifted again in a current of whispers:
“There he is — Ser Rinaldo.”
“The best husband in Valdiponte.”
“Do you see how he looks only at her?”
“Even in Bologna, they say—”
They whispered of Bologna’s temptations, of courtiers and widows who might have reached for him. Yet here he was, loyal, faithful, even romantic in his constancy. Among women’s circles the refrain was always the same: there is no husband like him — handsome, clever, diligent, unshaken by temptation.
He found Tessa at the center, her plain cloak dusted from the road, head bent over a length of wool. Five years younger, yet she carried herself as though time had pressed harder. She looked up — and the square seemed to still.
“My lady,” he said, offering his arm. No flourish, only a gesture of belonging. She set her hand lightly on his sleeve, and together they turned toward the florentine wagons.
The women watched them go, whispers curling after like smoke. Some in admiration, some in envy, all with the same thought unspoken: no finer pair walked the square of San Michele.
Rinaldo stood beside Tessa at the wagon, the crowd parting just enough for them to step close to the cloth bolts and baskets laid open.  The high ranged of products that Valiponte had a rare chance to meet.  Not affordable for average townfolks, but the prices here were much better than shops in Bologna.
He said nothing more than: “Choose what pleases you.” His tone carried both courtesy and certainty, as if whatever she set her hand on would be hers before the hour was done.

Not far away, another figure stood: Deonardo — silver hair unmissable even in the crowd. He bent as though weighing wares, yet his gaze lifted whenever a muleteer barked orders or a merchant’s tongue slipped from Florentine to Ligurian.
He asked in a casual tone,
“Your wagons came from Florence?”
“Aye. Cloth,” one answered. “We turn east for Ravenna after Bologna.”
“And why this road? It costs you days. The pass is harsh already.”
The man spat to the side. “The sea’s worse. Pisa and Genoa cut each other to ribbons. Merchants like us pay double for passage, if not more. On the mountain, there are bandits, yes — but men can fight bandits. Who fights the sea?”
Deo inclined his head, though his eyes narrowed. Another merchant added, voice lower:
“Taxes change every week at the ports. One captain wins, another loses, and we pay for both. Some gamble, hoping for gain. We don’t. We need the cost steady.”
Deo began listing what he wanted — the finest lengths of silk, blankets thick enough for mountain winters, linen for bedding. The wagon-master barked at his boys to dig through the packs.
While they worked, Deo asked lightly:
“You take this road often?”
The elder of them spat again. “Every month near enough. From Florence to Genoa, or east, Bologna. Always we stop here.”
The younger one leaned in, sharp-eyed:
“It’s the best-kept secret. Big cities think the mountain’s too hard, too poor. But San Michele has its name now — ledgers, order, even the library. Buyers trust what passes through these walls. Safer than sea-trade, these years.”
“And you also sell in Valdiponte?” Deo asked, his thumb brushing the edge of silk.
The man laughed short. “Better than you’d think. These mountain folk — they keep coin tighter than monks keep prayers. But they’ve horses, good stock, and their reach to Bologna is longer than it looks. More trade passes here than city lords reckon. They buy, they resell, they know how to profit. Underestimate them, you lose.”
The goods were laid out before him now — a bolt of crimson silk, fine as breath, fit for Aurelia’s hall. Yet his thoughts strayed elsewhere. He saw instead a darker chamber: the healer’s stone rooms, a lamp burning low, air sweet with strange herbs.
He gave the merchants a single nod. Matteo stepped forward to weigh the coin and take the packages. 

Rinaldo stepped closer, voice measured but warm:
“Cavaliere, we have not yet spoken. Rinaldo Rinaldeschi — steward for Bologna’s house, though here I serve as collector more than merchant.”
Deo returned the nod, straight-backed.
“Deonardo di Alferini, in service of my Lady of Vercado. My letter has already troubled Father Lorenzo — but not all hands met across a page.”
Rinaldo’s smile was courteous, not lingering.
“True. And a page tells little of the man who writes it. You study herbs, they say?”
Deo allowed himself the faintest curve of a smile.
“Among other things.” He set down the length of cloth he had been examining. “Today, I study the roads. Why so many merchants choose this pass, when the seas might spare them frost.”
Rinaldo’s gaze briefly toward the line of wagons, then back.
“Because seas sink. And captains swear by saints, but never by ledgers. Here, a cart may turn back, and San Michele keeps its records with cleaner hands than any port.”
Deo’s silver hair caught the light as he tilted his head.
“So you trust monks with your trade?”
Rinaldo spoke plain, as men do who have spent their lives measuring with hands, not quills.
“The brothers — I’ve known them since I was a boy. Their cellars are dry, their eyes are clear. I trust what I’ve worked beside for twenty years. That’s enough for me.”
No flourish, no argument — just that.
Deo inclined his head, courteous as a man of court, but gave no answer. His lips held the silence, yet I saw it in him — the plainness had cut sharper than rhetoric ever could. Simplicity laid bare what polish could not shield.

Before sunset, Donna Donata Maffei entered the square. Her fur-lined cloak swept close, heavy enough to signal its worth in every step. Four servants came first, clearing her path; three maids followed, carrying the smaller burdens of her household.
The attention gathered at once. Monks bent their heads, not low but just enough. Peasants paused, some glancing too long before looking back to their carts. Merchants weighed their courtesies like coin, each greeting measured to show respect without surrender.
At the tunnel’s mouth, Lorenzo himself waited, his hands folded at his sleeves, the abbot’s composure wrapped around him like vestment.
And after the maids, in the trailing shadow, came Bianca. Ten years at most, steps small, almost noiseless, her eyes lowered not in reverence but in habit. A thin hand clutched the leash of a little white dog, its paws quick against the frost. The girl walked as if half-forgotten, a figure stitched only because the cloak ahead demanded it.

📖 falcon in the Square🕯♛♡🐾 dusk

The sun was sliding down the ridge when Donna Donata crossed the square. Fur at her throat, servants behind, her little daughter Bianca at the end, the white dog was rte only one minded how her small feet catching the stone. 
By chance, Dania stepped out of the tunnel just then, basket in hand. Donata’s eyes cut toward her — sharp, cold, a look meant to stop, and it did. Donata turned aside at once, entered the church, and Lorenzo followed after her, leaving the square behind.
So the lady vanished into incense and stone. Her maids, servants, the girl, and the white dog stayed outside.
The square thickened with another kind of pulse: peasants finishing their tallies, wine poured, bread torn; laughter starting in corners where coin had changed hands. Mercenaries restless, half-drinking, half-waiting, their captain’s words about the coming ride still buzzing like hornets in the ear.
The bell tower’s shadow stretched long.
I saw Dania again — moving past its base, rounding the church and tower like a soundless black wing.
At the far side of the square, Deonardo had also noticed. He walked toward her, toward the church entrance — close enough to see, not close enough to touch:
The bell struck for Vespers, deep and sudden above the square.
The girl flinched, her small hand loosening on the leash.
The dog ran blind, frightened by the sound, and struck against Martino’s leg.
The mercenary, already unsettled with wine and news of the road, swore and staggered, his boot lifted, hand falling toward his sword.
The bell’s peal still shook the air when Martino’s temper flared.
Bianca cried out, stumbling forward with both hands outstretched.
Around him, boots scraped stone, peasants pulled children back, and two of his comrades half-rose, ale sloshing from their cups.
One of Donata’s servants lunged from the steps, too slow. A peasant near the barrels shouted, dropping his sack.

The sunset cut through the square, slanting red along the bell tower wall, catching the steel at Martino’s hip as it loosened from the scabbard. Deonardo was already moving, his cloak sweeping behind him, his hand on his own sword hilt. The distance was too far, the crowd too thick — one heartbeat more and steel would have rung.
Then — she was there.
Dania’s hand closed over Martino’s, not wrenching but resting, a touch that carried no weight yet stilled him all the same.
Her fingers were pale against his knuckles, her sleeve brushing the rough leather of his jerkin.
Her hair caught the last strip of light, strands lifting as the bell’s aftershiver passed.
Her dress moved with her like water — plain, dark wool, but alive in its folds, the belt-strap pressing a line that drew the eye to her stillness.
She smiled.
Lips not painted, yet shaped with unflinching grace. Eyes dark, long-lashed, holding his as though they had all the time in the world.
She smiled to him with the full, unashamed invitation of a woman who knows exactly how to be seen. The kind of smile that melts a man’s stance before his mind decides whether it should.

And Martino fell into that subtle collapse where a man forgets why his hand was on a weapon. His breath loosens; his weight shifts; his eyes, for a second, mirror hers without knowing what they’ve agreed to.

The square’s noise thinned — not gone, but muted, as though the peasants and mercenaries alike felt something but could not name it.
That’s when she shows the claw — her smile vanished, not wiped but sharpened into a blade. Her gaze narrowed to a single, exacting point, same as a raptor aiming its prey.
Her finger pressed once, deliberate, into the hollow of his hand. Her pupils caught the last sunbeam, almost gold, bright against the dusk. A single second stretched, heavy as an hour.
And the air shifted with it. Mule-hooves stilled on stone. A woman’s laugh broke off mid-note. The square itself leaned toward her, as if all breath was drawn into that gaze.
Deo’s blade hung half-drawn, glinting in the red light, his hand arrested as he saw Martino collapse inward. The man’s anger forgotten, his pride diminished. His hand slackened. The steel slid back into its leather with a muted sigh.
The bell above gave its last peal, rolling heavy through the square. The sound clung to the stone as if even it refused to leave before the moment was done.

The dog darted free, Bianca’s small hands catching it up, clutching it to her chest. Her breath came quick, her eyes wide, but she steadied — as children do when they must.
Dania let go.
Her hand fell to her side, fingers loose, her lips no longer curved but calm, unreadable. She walked on, towards the cellar as always — her satchel brushing her hip, her hair slipping down her back — as if she had never paused at all.
The sun slid below the ridge.
The square filled again with sound — drinking, trade, laughter forced louder than before.
But the air still held the echo of the bell, and of the hawk’s gaze that had cut through it.
No one else saw it clearly.
But Deo did.
And so did I.

Late evening, I was with Deo in his chamber. Standing alone. 
The silk lay folded on the bench where Matteo had left it — crimson, fine as breath, bought that afternoon from the Florentine wagon. It had seemed simple then: a gift for the healer, a token of courtesy, of gratitude. Something fitting — herbs and fine cloth, knowledge and beauty.
But now — after what he had seen in the square — the fabric glowed too brightly in the dim light.
His hands lingered over it, then withdrew, as though it might betray him.
The image would not leave him: Dania’s calm, her hand laid like invitation, then turning hawk-sharp in the beam of sunset. A power he had not reckoned. Not the quiet healer who matched herbs to fevers, nor the figure who received him with patience by the fire. But another presence — one that silenced blades, bent men without force.
He did not know what he had seen.
Nor did he know what he carried now — cloth meant as a gift, yes, but to whom? To a woman who had looked at his shift with steady curiosity? Or to the falcon whose eyes had turned the square still?
The silk rested there, waiting.


📖  November 11, 1264 Tuesday
 Rinaldo’s secrete  🕯♛♡🐾St. Martin’s Morning ,Tuesday

The chapel above the cloister was no warrior’s shrine. Built by the Basili degli Aromatari a hundred and sixty years ago, it was a gift that set remedies and roots beside swords and crowns. On the walls, not martyrs in agony but saints in work: St. Fiacre with his hoe and angel-waterers; Cosmas and Damian offering ointment jars. A small vaulted room, once washed in green and gold, it opens at the altar to a stone cross and the terrace air. Wind threads the cracks like a half-remembered chant; the scent holds—rosemary sharp, resin and smoke soft, rose gone to dust. At dawn the light pours in and gilds the carvings, and for a moment it seems heaven stoops to the herbs.
On feast days the monks unbarred the door, and peasants climbed the stair to believe that heaven smelled of healing.
The rest of the year it stayed locked, save for a patron’s coin or a bishop’s show of relics.
But Dania had her own key. She always did.
Each Tuesday she came before dawn, prayed by placing her hands on the wall, as though she could listen to the mountain’s pulse through its mortar.
And this morning — St. Martin’s morning — she left earlier still.
The bell had not yet rung Prime when her black hem crossed the frost of the terrace, the chapel door closing behind her, the last smoke of rosemary clinging to her hair.
I perched on the short wall between the terrace and the deep cliff, tail curled against the cold stone, watching her move against the paling sky.

The terrace opened wide to the mountains. The Apuan peaks stood blue and white in the distance, their ridges catching the first blush of sun. Frost still clung to the cloister roof, glimmering faintly pink where the dawn struck it.
Dania stood at the edge of that light. The chapel rose behind her, Romanesque roof glowing like ember-stone in the cold air. Above the ridge, the quarter moon still faded, a ghost paling into day.
She was just turning toward the stair, when the couple rose from below — Rinaldo and Tessa, their breath still quick from the climb.
Tessa Rinaldeschi bowed first. A plain bow, steady as a psalm.
“God grant your hands strength, Madonna,” she said.
The dawn light drew her face sharper — plain, marked by years of constant labor and responsibility. Youth had drained from her features, taking with it, the part of woman that invites attention. Yet what remained was weightier: an authenticity grounded in life and faith, a presence that stood without adornment.
Before her stood Dania — plain too in her black dress, her own beauty marked by life’s traces. But unlike the wife, something of her could not be dimmed: a charge, quiet yet unmistakable, that made silence feel alive.
Tessa hesitated, as if the moment might stretch, as if she wished to draw nearer but could not find the thread to bind her words to Dania’s stillness.
Then Rinaldo followed.
His bow was exact, measured as a merchant’s ledger. But his eyes — ah, they lingered. Too long for politeness, not long enough to betray.
“Madonna,” he said, “your service to this valley is spoken of often.”
The words carried more weight than they should, heavy with what he did not name.
It struck me then: this was the first time Rinaldo had crossed the healer so closely in person. Other merchants sent their servants to her door, or came themselves for teas and herbs. But Rinaldo — always, he passed notes through Lorenzo. Yes, perhaps it was because Dania had never been blessed by any church, and strict Christians hesitated at her threshold. Lorenzo believed that was his reason. I never did. That part of Rinaldo was unreadable — until now.
I slipped without thinking — caught in the current of his gaze, a wave strong enough to draw me whole.
And through his eyes, I saw her.
The sun caught her hair — dark strands edged with gold fire. Her black dress lay plain against her figure, yet the dawn made her outline sharp, almost luminous against the pale sky. In that frame she seemed more than woman — something older, unnamed, a stillness that drew the world toward her.
For one breath, the terrace was silent — only the frost crackling beneath their feet, the sky breaking open above.
Dania did not move quickly — her hand rested on the stair’s worn stone, her breath rising steady in the chill. Her face was pale, touched by incense still, her eyes unreadable yet alive, as though the silence itself listened to her — like a figure carved from the mountain itself.
And he looked. Just long enough for it to wound him with its beauty.
His gaze held the moment, fuller, heavier — as though the whole world had stopped and arranged itself so only she remained before him.
I felt the ache run through him. Quick, sharp, unbidden. Not lust, not even desire or admiration. A recognition he did not want, yet could not turn aside from.
He folded it at once — tighter than a ledger’s seal. To all others, he was still steady Rinaldo: grounded in the soil of his birth, unshaken before temptation, working each minute for the heirs who would follow, loyal to his wife with respect and quiet tenderness.
But this — this was his secret. Love buried so deep it smolders only in dreams, while by daylight he locks it in bone.
Strange, how humans live. One life in their hands, another in the hidden chambers of the heart — and they pray the two never meet.

When I slipped from his sight, Dania was already before him.
She gave him no more than she had given his wife: a calm nod, and one plain sentence.
“You are early. The prayer bench is still warm enough.”
Then she moved past, her steps already carrying her downward.

📖 The Day of San Martino🕯♛♡🐾 Ride to Valiponte, before noon

The monastery bells rang Prime and the feast-day woke in full. The cloister filled with scents of roasted chestnuts, wine warmed with cloves, bread split open to steam. Peasants pressed close with their baskets, monks moved in ordered lines, mercenaries shouted too loud already. The day was meant for celebration.
Dania herself did not stay for feast. She rode out as she always did on a Tuesday, down to Valiponte, in basili’s house. Patients wait there who cannot climb — lungs too tired, joints too twisted, fevers too long endured. She goes to them, even when the calendar says feast, even when the bells say St. Martin.

Romé went with her.
The road bent downward from the cloister, the frost still clinging in the hollows, their breath clouding like smoke. 
Rome’s horse was called Fiamma — flame in her veins, flame in her step. Chestnut-red, mane like fire unbraided, she danced even when the road fell steep. but Romé matched her with a hand light as wind. She carried him not as a rider, but as though the two of them conspired in speed. She loved him — I could smell it in her skin, hear it in the stamp of her hooves when other women leaned too close. Her jealousy was sharp, but her loyalty sharper. She had carried him through dead ground more times than men dare count, ears back, eyes wide, yet never leaving him.
Dania’s horse bore the name Nero. Black, steady, broad-shouldered, his eyes were old as winter rivers. I knew him well — I had ridden his croup a hundred times, tucked behind Dania’s hip, my paws pressed against the leather. He never minded me. Perhaps he thought me another satchel she carried, small and watchful. He stepped steady, no waste, no show — the kind that learned seasons as men learn prayers. Through snow, through mud, through summer’s heat, he carried her down to the valley and up again, as if the rhythm of her work was the only rhythm he knew.
Fiamma snorted at my landing, stamped once — not at me, but for him. She was in love with Romé and never shared easily. But she knew: the little shadow on Nero’s back belonged there. So she curved her neck, brushed her muzzle once against Nero’s shoulder, and together they stepped forward — flame and night, side by side. I purred into the cold wind. 
They rode down while the valley roads flowed upward. Families from the villages pressed toward San Michele — children clutching ribbons, women balancing baskets of loaves or chestnuts, men leading mules with sacks slung heavy on their backs. The feast day pulled them all like the bell itself.
When they saw Dania, some stepped aside with a small bow, a hand lifted in silent greeting. A murmur of her name passed in one group, but no one called it aloud. They knew her ways — she gave little to gestures, only the work of her hands. A nod was answer enough.
Romé, though — he drew eyes differently. The flame-colored mare beneath him, his dark lashes low over his clouded gaze, the set of his shoulders — strangers looked longer than they meant to. A girl glanced back twice; a boy tugged at his mother’s sleeve, whispering. Some crossed themselves, as if beauty should be warded against like fire.
Fiamma tossed her mane at their stares, proud of the attention. Nero kept his stride heavy and calm, black hooves steady on the frosted road. I rode his back as always, fur warmed against Dania’s cloak, my tail flicking at the crowd.
The road bent downward, away from the mountain. The voices behind grew fainter — songs, laughter, the clink of mule-bells fading into the distance. Ahead lay Valdiponte, its houses stacked like stone steps above the river, and within it, the Basili House where patients already waited.

The road dropped them into Valdiponte by midmorning. The square was swollen with feast — geese turning on spits, boys shrieking as they chased nutshell boats, the clang of fiddles and bells. Smoke stung the air, sweet with fat and wine.
But Romé and Dania did not stop in the square. They turned aside, past the shouting vendors, past shops stacked with cloth and spices. At the end of a narrow lane, half lost between two tall houses, stood a gate most eyes would miss. Its wood was plain, weather-stained, and merchants’ barrels leaned against its side as if it were nothing at all.
Dania swung down from Nero, her hand brushing the latch like one who had done so many times before. The gate gave without creak, and they stepped through.
Inside was another world.
The courtyard opened wide, all light and green. A fountain sang at the center, water falling bright over stone, its basin edged with moss. Orange trees still held their leaves, and climbing ivy softened the walls. Columns ringed the space, carved with scrolls of vines and figures from far-off lands — not saints, not martyrs, but creatures with lion faces, winged horses, birds carrying pomegranates in their claws.
Fiamma snorted at the fountain, tossing her head as if even the water should admire her. Nero lowered his muzzle, patient, waiting until Dania’s hand stroked his neck. I leapt down from her cloak onto the stone lip, tail curling at the spray, paws cool against marble.
Here the noise of the feast was gone, shut out by stone and water. Only the trickle of the fountain and the faint call of doves above.
And here — at last — the Basili master waited.

From the shaded portico, Basili Basile himself stepped forward — broad-shouldered, his beard streaked with grey, his robe plain but edged with dark wool, the mark of his house.
“Madonna Dania,” he said, bowing his head, “welcome. The patient waits already in the Ural room. The women of the house will guide you.”
His eyes softened as they lifted. “Was the road kind? This morning’s frost looked cruel enough from the windows. If you need a rest, or a little lunch, say so — bread is warm, and tea steeped.”
Before she answered, Dania turned slightly, her hand light on the black horse’s rein.
“Ser Basili, allow me — this is Romualdo di Orellana, a companion to me these days.”
Romé inclined his head, one palm against his chest, the flame-red mare stamping once behind him as if to mark the moment.
“Ser Basili. An honor.”
Basile’s gaze measured him briefly, then warmed with courtesy. “Di Orellana — I have heard the name in these valleys, though rarely met its bearer. Welcome also to my house.”
Romé’s smile flickered, respectful but edged with that untamed ease of his.
“Your welcome is well kept, signore. Even the fountain speaks of it.”
Only then did Dania bow her head again, her voice even:
“The road was clear. I will see the patient first — tea afterward.”
Basili’s small smile returned. He stepped aside, motioning them both inward, toward the courtyard where the fountain whispered and the herbs grew bright in winter’s light.

When the women of the house appeared and led Dania inward toward the Ural room, her black hem vanished into the shadowed corridor. The sound of the fountain seemed louder in her absence, its water falling clear against stone.
Romé remained in the courtyard with Basili, silence resting between them. For a moment, only the fountain spoke.
Then he inclined his head, speaking with care, his voice low enough that it felt private though the walls stood open:
“Signore Basili — your daughter, Filippa.”
Basili’s gaze sharpened, then steadied.
“When your daughter and I spoke at San Michele, she told me people say I walk like a verse. I asked then if she would answer me in verse, and she agreed. But she said it would not be met in passing — only here, in her father’s house.”
He lifted his eyes.
“I have come now to honor that word. If it pleases you, I would ask her leave to meet her where she keeps her ledgers.”
The older man folded his hands behind his back, measuring the young knight with the same care he gave his ledgers. His answer came slowly, not cold, but weighty:
“My daughter is nineteen, signore. In Valdiponte, at that age many women are already married, bearing and raising children. Yet she has refused all proposals brought before her. She is her own person.”
His eyes moved toward the inner stair, then back to Romé.
“If she has said she will meet you — in the place where she works, among the ledgers she keeps — then she needs no father’s leave to hold her word. She is old enough to decide for herself.”


Cross the square, apart from the Basili’s main residence, stood their apothecary warehouse.
The town outside was swollen with noise — St. Martin’s Day turned Valdiponte’s square into a fairground: goose roasting on spits, skins crackling; wine poured fresh from casks; children darting through smoke and ribbons of laughter. From the church doors beggars called, bowls outstretched, coins ringing against wood.
On the first floor, Filippa bent over a ledger, quill quick across the page. Behind her, shelves sagged with jars of dried leaves, bundles of roots, packets tied with twine. The shutters were cracked open; festival noise filtered in faintly — a flute, a boy shouting, all drifting upward.
She looked up at once, her eyes catching green in the half-light.
“You keep your word, signore. Two days — that is swifter than any merchant.”
Romé leaned against the doorframe, dust still on his boots.
I slipped into the room as smoke beside him, unseen — demon power works best where humans least expect.
“I told you I would bring it,” he said, voice easy. “Dust travels lighter than ledgers.”
Her mouth curved, then steadied. The quill tapped once against the parchment.
“You come with dust, signore. I stay with jars.”
Romé tilted his head, studying her with that half-smile.
“And do you enjoy the jars more than the feast outside?”
Filippa’s hand lingered on the ledger, fingers faintly ink-stained. She looked past him, toward the shutter where festival smoke seeped through the cracks.
“The feast is always the same — goose fat, fiddles, masks. Everyone knows the songs before they’re sung. My brothers go out with caravans, cross mountains, trade in fairs. They return with dust on their cloaks and tales in their mouths. I stay. I keep the jars. The only stories I touch are written on labels — cinnamon from the Indies, saffron from Persia, frankincense from Arabia.”
Her fingers pressed harder into the page, then softened.
“Sometimes I envy the herbs. At least they travel.”
Romé leaned against the frame, his boot heel resting easy on the stone. His voice came low, almost amused, but not unkind.
“Herbs only travel because they are carried. They’re plucked, dried, pressed into sacks, shipped where others will use them. They have no say in their own pilgrimage.”

“In Basili’s House, boys travel with caravans. They return with their cloaks gray and their purses heavy.girls stay with jar and ledgers .” Her hand fell on the ledger, fingers splayed over the neat lines. “It is not fair.”
Romé crossed the room slowly, gaze trailing over the shelves.
“Fairness is a word for contracts. Not for lives.” He stopped near the counter. “What you do here — it binds the sick to health. That is travel enough for some.”
But her eyes did not soften.
“I copy names of herbs from lands I will never see. I want to go where they grow, not only weigh their price.”
Romé’s smile touched his lashes, shadowed. “Your father seems believe you would choose your own path.”
Filippa’s fingers tightened on the ledger, though she had pushed it aside.
“You talked to my father?” Her tone carried more steel than surprise.
Romé inclined his head lightly. “Only brief words, before I came here.”
Her mouth pressed into a line — not childish indignation, but the affront of a grown woman whose choices were still weighed in her father’s house. Yet beneath it flickered something more complicated: if Romé had spoken to Basili, then he had not treated her as a passing fancy. He had acknowledged her place, her family, even her defiance.
Her green eyes narrowed, steady and unflinching. “Then I hope you did not ask his leave for my time. I do not give my hours on another’s permission.”
I flicked my tail from the beam above. Ah, there she was — nineteen and flame-hot, no vessel for any man’s will. She wanted her life to be her own ink on the page, not her father’s seal. And Romé? His voice softened, though the curve of his mouth kept a trace of play.
“I think you count yourself unlucky, signorina — but I have seen fathers barter their daughters as if they were sacks of grain. Yours does not. He listens. He lets you stand in your own choice. That is no small fortune in this valley, nor in any valley.”
Filippa’s chin lifted, the green in her eyes quick.
“So you would have me thank him for letting me breathe?”
Romé leaned a little closer across the counter, half-smiled to himself.
“You see your father as fierce in his guarding. Mine was fiercer still. He would never have let my sister choose as she wished. Every suitor she looked toward, he weighed with his sword in mind — if the boy could not stand against him, he would not let her heart rest there.”
He met her eyes,  “Your father — he lets you hold the choice yourself. That is rarer than you think, signorina.”
Filippa’s sight flashed, a mix of pride and defiance.
“And if I choose no one?”
“Still, it is your choice. That is the difference.”
Filippa’s hand slid from the ledger to the sill, her fingers tightening as though she might push the shutters open wider.
“If my father lets me choose,” she said, her voice low but edged, “then I would sooner choose the road than any husband. Why should I be given only a house, a dowry, a bed? My brothers ride over mountains, they trade in cities I’ve never seen. I weigh their spoils in jars, but my heart weighs nothing here.
“I would rather risk hunger on the road than be filled like a vessel in another man’s house.”
Romé’s smile curved, faint and thoughtful. He let the silence linger a moment before he spoke, his voice softer than the words themselves.
“Then perhaps, signorina, it is not a house you must choose — but a companion. A husband who would ride the road beside you, who would guard you as fiercely as your father does, and yet let you walk in your own steps. Such men are rare.”

“And you, signore? Are you such a man?”  Her green eyes caught his, bright in the half-light.
Romé tilted his head, smile faintly in shadow, as though her challenge pleased him the most in the day. He leaned closer, lowering his voice to match her steadiness.
“Perhaps not. I am much less a man who can guard. Even I need a guide — someone bold enough to keep me from losing my way.”
From the square below, music swelled suddenly: pipes shrilling, drums rattling, the rise of voices as a troupe of jugglers began their feast-day show. The sound leapt through the shutters, filling the air between them.
Romé’s smile deepened, catching the rhythm of it.
“And perhaps today, signorina, that guide could be you. What say you — will you guard me through the dangers of Valdiponte’s St. Martin feast?”
“If you trust me to guard you, signore,” she said, her voice level, almost solemn beneath the festival din, “then I will.”


📖 The dream in Feast🕯♛♡🐾 

The square had swollen into chaos. Fiddles sawed, pipes shrilled, children darted between legs, and a troupe of jugglers cleared a circle with shouts. Torches flared in their hands though it was still daylight, the smoke biting sweet as pitch.
Filippa’s step faltered as the press closed in around her. Men jostled, women laughed too loud, cloaks brushed her sleeve, and the smell of sweat and roasting pork tangled together. She had never stood so deep in a square — never in the middle of a crush like this. Her fingers tugged instinctively at her veil, wishing for more distance, more air.
Romé was already half a head ahead, slipping between shoulders, grinning like a boy. He looked back once, saw her caught at the edge, and without hesitation pushed to her side. His hand closed around her wrist — firm, warm, unashamed — and he pulled her forward.
“Come,” he said, the word lost in the din but his smile clear.
She stumbled, caught, then found herself suddenly laughing despite herself.
The jugglers’ torches rose and spun, fire wheeling against the sky. The crowd roared, surged, pressed closer. Filippa’s breath caught — the bodies, the heat, the noise.
And then Romé was gone from her side. For one heartbeat she panicked — until his hands came at her waist. He lifted her as if she weighed no more than a cloak, and set her atop an empty wine barrel at the square’s edge.
“Higher,” he said, laughing, his breath warm at her ear as he steadied her hips. “See with both eyes, not just one.”
She clutched her skirts, startled — and then the crowd parted below her like a sea. She could see over them all: the ring of jugglers, the torches blazing, the painted faces grinning as knives spun from hand to hand.
Romé vaulted up beside her in one fluid motion, his shoulder brushing hers. He braced a hand at her back so she would not lose her balance. His eyes glittered in the firelight — not on the jugglers, not on the torches, but on her.
And Filippa, for the first time, looked down at the feast of her own town not as a girl hemmed in by ledgers, but as if it all belonged to her.
When the torches dropped and the crowd roared, Romé jumped down first. He turned, hands raised, and lifted her lightly. For one dizzy instant her feet found nothing, then his strength steadied her, and she was back on the cobbles, skirts swirling.
“Now you’ve seen,” he said, grin quick, “time to taste.”
He did not wait for her answer. Already he was pulling her toward the food stalls, where smoke rose in clouds and grease spit from skewers.
A vintner ladled wine into clay cups, and Romé seized two before coin had even clinked. He pressed one into her hand, lifted the other, and drank deep. When she hesitated, he tilted her cup toward her lips with a laugh. “Drink — saints forgive us, today the monks cannot count every drop.”
She drank. It was sweet, rough, headier than the careful measures at her father’s table.
Romé was already at the next stall — pork roasted black on the outside, steaming pink within. He tore a hunk with his knife, handed her the better cut, and bit his own like a soldier on campaign. Fat dripped to his wrist; he licked it carelessly, grinning.
“Too much,” she whispered, half in scandal, half in delight, as he pressed a skewer of chestnuts into her free hand.
“Exactly,” he said. “That’s the only way to eat on a feast day. Soldiers know: eat while there’s food, drink while the casks are open. Tomorrow — who knows?”
She laughed then, the sound freer than she meant, cheeks flushed with wine and firelight. Around them the crowd shouted, sang, clapped to fiddles. Romé ate and drank without measure, his laughter raw and full, showing her — not court, not courtesy, but the wild edge of life as men lived it when no one counted.
And Filippa, raised in order and ledgers, let herself be pulled into it — a little shocked, a little dizzy, but unwilling to step away.

When the last chestnut shells had scattered at their feet and the cups hung empty from their hands, Romé tugged her toward the river. The feast thinned there — fewer stalls, more children crouched by the gutter where a narrow stream ran fast with the day’s rain.
Little nutshell boats floated on the water, each carrying a stub of candle, flames trembling against the night breeze. The children cupped their palms to shield them, laughing, shouting as the stream caught them and bore them away.
Romé slowed, curiosity flashing sharper than wine. He crouched beside the smallest boy, his knees splashing in the mud. One hand cupped over a flickering light, his face bright in its glow.
“What is it?” he asked, turning back to Filippa. “What do they mean by it?”
She knelt more carefully, her skirts tucked, the firelight catching in her braid. “It’s for San Martino. Each boat is a prayer — or a wish. Some for the dead, some for the year ahead. Children dare each other with them — who will float further, whose flame will last the longest.”
Romé’s gaze lingered on the stream, then back to her. 
“Prayers or dares — both make sense.” he said.
The boy pushed his little boat forward, water tugging it downstream. Romé nodded to him, solemn as a knight saluting another, then looked at Filippa again. “And you? Did you ever set a flame on the water?”
Her smile flickered, half-proud, half-shy.
“When I was a girl, yes. I wished to see Florence, Venice, even Rome. And I wished never to be shut in the house. But girls grow older, and their wishes sink sooner.” Filippa said. 
Romé tilted his head, eyes gleaming, said to her:
“Then let’s see if tonight proves you wrong.” 
He cupped his hand, shielding another child’s flame, then nodded for her to set it free.
She leaned forward, fingers brushing the stream, guiding the shell outward. For a moment, the candle steadied, catching the current. Her breath caught. “It floats,” she whispered.
Romé’s smile curved, soft. “Of course it does. Wishes don’t always sink.”

By the time we reached the bridge, the feast was already sliding into night.
Smoke clung between the houses, fiddles scraped without rest, but the sky above turned tender and blue — the hour when nothing belongs, not day, not night.
Lanterns swayed on ropes between eaves, golden fruit in the darkening air. The river broke them into trembling shivers.
Children leaned on the parapet, palms opening to set nutshell boats afloat — each with a flame no larger than a moth’s wing. One by one, the current carried them downstream, a scatter of stars fleeing the valley.
Romé rested his elbows on the stone rail, watching them go. Filippa stood beside him, her veil slipped, her braid loose against her shoulder. The light painted her eyes green fire.
“I wished to see Florence,” she said softly. “Venice, even Rome.”
Her hand hovered as if to touch the water, then drew back. “It isn’t impossible. I have wished… further.”
Romé turned, smile caught in shadow. “Further than saffron and cinnamon, than the groves you weigh in jars?”
She nodded, lips pressed, then daring. “Further than names I copy and never taste. Why should the road be closed to me?”
For a moment she said nothing, only watched the little flames drift, each one a wish slipping beyond the reach of hands. Her breath shivered, but her eyes were bright.
Romé leaned closer, his voice low enough that it folded into the river’s murmur.
“Careful, signorina. A wish spoken at dusk travels faster than a ship with sails full. You may find the world already moving to meet it.”

She was quiet at first, then her words came fast, almost defiant:
“You said I might find a husband accompany me on the road. But men all want the same thing — the boys in this town. A wife, a dowry, children. They think I am a jar to be filled, kept, passed down. But I will not be that vessel.”
Romé turned, lashes half-shadowed. “And what will you be, signorina?”
She lifted her chin, green eyes steady.
“My father taught me to read herbs — which heal, which burn, which stir the blood. I know the body is not only for bearing heirs. It can ache, it can hunger, it can rejoice. Why should I not choose to taste that? Why should I not have my own adventure with my own body?”
For a long moment, he did not answer. His gaze slid down, taking in the flush along her throat, the rise and fall of her breath. Then, very slowly, he leaned closer.
“Most men would not know what to do with such a flame.”
She did not step back. “Then I have no use for most men.”
The words hung between them, braver than her years, but Romé did not laugh. He brought his hand up, brushing a stray wisp of her hair, fingers lingering a moment longer than courtesy allowed. Her breath caught, but her eyes never left his. 
He did not reach for her quickly. His hand lifted only partway, brushing a wisp of her hair, pausing near her cheek as though asking permission without words. Her breath caught — but she did not turn away.
And so he bent — slow, as if savoring the step — and kissed her.
A landing unhurried, deliberate, so the moment itself could be named.
Her lips met his with a sudden surety, her hand rising to his shoulder, grip firm as though the bridge beneath them might tilt away.
Below them, the nutshell boats drifted on, each flame shaking, scattering down the river into night. Above, the lanterns swayed, caught by the first breath of evening wind.


They walked back through the square, slower than before. The feast pressed around them — smoke, fiddles, shouts — yet it all blurred into noise. When they reached the Basili house, Filippa slipped the key from her sleeve, and the heavy door closed the world outside.
The stairwell was narrow, cool, the scent of herbs rising from the storerooms below. Lantern light caught her braid, the loosened veil, the flush along her cheeks.
Romé lingered a step behind her.
“You know what you risk,” he said softly.
She turned, still standing half a step above him, so her eyes met his level.
“I risk being seen. I risk being whispered about. But I will not risk my life passing like my mother’s — chamber to kitchen to bed, as if that were all a woman’s body was for.”
For a heartbeat he did not move. He shifted his retreat, but instead leaned closer, shoulder brushing the air beside hers. His hand flexed at his side, restless, then rose to the rail where hers rested. Their fingers brushed. The contact was light — yet her warmth trembled through him as if she had pressed her whole will against him.
He almost let it carry him — her youth, her fire, her claim. Desire tightened in him, sharper for the honesty of it.

Then something stilled him.
She was young and pretty, yes, but that was not what held him. It was her fire — the way she flung open doors not yet closed to her, the way she spoke of freedom as if it could be plucked like a herb from a jar. In her, he saw himself — a boy who had once dreamed the same, before the road wrote its price into his skin.
And when he looked into her green eyes, he knew: 
she had not yet paid the price for dreams.
She was still sheltered by her father’s roof, her brothers’ shadow, her family’s love. 
Her flame was bright, but unbloodied.
It was then that Dania’s gaze rose before him — the way Dania had looked at Filippa, still and fierce, as if she longed to hold the girl’s fire unbroken for one season longer.
In that instant, Romé understood Dania — and in understanding Dania, Romé felt himself understood.
No words, no glance exchanged — yet the ache of his life met the ache of hers.
So Romé drew back.
The green-eyed flame would burn one day, when her road demanded its price.
 “Not tonight,” Romé said—but only to himself. His smile went light, graceful, covering what shook.
Filippa frowned, half-breath taken. 
No words. Only watched Romé go down, unhurried. 
His shadow clung to the angle of the stair; then it was gone.


Romé stepped out into the cooler hall and I dropped from the beam onto his shoulder—soft, my claws careful. Romé did not start. He breathed once, long. 
We knew each other.
Do not think them far apart—Romé on the stair, Dania elsewhere. They met without faces.
 They both saw the same small thing—the girl’s flame—and both chose to guard it. Sometimes love comes like that: 
no kiss, no vow; only two souls setting their bodies to the same value. 
And that is a home as true as any.


📖November 12, 1264 wednesday
 After the Feast, lunch time on the healer’s table🕯♛♡🐾  Last Quarter moon

The cloister shook off its feast like a man shaking wine from his head. Ash and nutshells swept, benches dragged back under the arcades. Ferrazo’s chosen men sharpened steel and whispered wagers. The guesthouse still loud with children and servants, the square still stinking of pork-fat. But by midmorning, the rhythm bent back toward order.
Dania took her hours for the spices that those traveling merchants had left in thanks — saffron thin as threads of sun, nutmeg dark as polished stone, cinnamon rolled like bark from Eden. People had also laid gifts quietly at her door: jars of honey, woolen skeins, dried apples, even a loaf still warm. She touched each only lightly before setting them aside, her mind already on the herbs spread out before her.
That was when Luca appeared, arms full of parchment —- he carried all Deonaldo’s studying materials to the heal’s studio, set the stack down on her table with a bow.
So the day folded there. Deo spread the parchments, eyes bent to ink as if the script itself might anchor him. His hand moved often to his notes, neat lines correcting, cross-reckoning, layering one thought against another.
Dania worked beside him but apart, her table filled with jars and spices. She ground, sorted, measured, the sound of pestle and mortar steady as a second heartbeat in the room. Sometimes she broke from it, writing small notes on slips of parchment in her careful hand, or resting a moment by the window, cloak drawn close, the pale light settling around her.
And Romé? He read in the corner by the high window, shoulders loose, hair bent into shadow. Now and then his eyes lifted from the page — not toward the words, but toward the others. To Deo’s pen scratching quick, to Dania’s hand steady at the mortar. He would watch for a breath, then lower his gaze again, as if the act of seeing were its own kind of reading.
Once, when Dania’s knife dulled on the herbs, Deo laid down his quill. Without a word, he took up the whetstone, drew it slow across the blade, then set it back beside her. Another time, his hands moved from ink to glass, wiping the dust from jars, arranging them one by one in neat order along the shelf — rosemary, sage, cinnamon. Work that was not his, yet he made it his, quietly, as though the silence itself had asked it of him.
The hours passed like that: three bodies, each at their own work, yet held together in a stillness more binding than words.
I lay along the sill, tail curling into the sun’s edge.
Sometimes I leapt, light as smoke, into Romé’s lap, where he brushed my ears without lifting his eyes from the page.
Only Donata Maffei’s maids broke the silence, twice, asking for teas, their words falling into the room and vanishing like drops in a well.
And so the day went — ink scratching, pestle grinding, sun creeping its slow arc across the wall.
Nothing happened.
Everything happened.
Deo bent lower and lower over his parchments, but his mind circled still around the square — the flare of steel at Martino’s hip, the hand that stopped it, the hawk’s gaze that left him uncertain whether he had dreamed it. 
Romé, by contrast, read as if the day before had already washed off him like river water. He did not speak of Filippa, nor of wine, nor of dancing — her flame seemed to have slipped into the night like the nutshell boats on the river. Clean gone. He alone carried no shadow.
Around noon, the door stirred, and Matteo appeared with a basket of food — steam carrots and cabbages, bread still warm, its crust split and fragrant; a pat of butter wrapped in damp cloth; a wheel of pale cheese cut clean. He set it on the low table, bowed slightly, and withdrew as silently as he had come.
The three gathered to the dining table without call. Romé slipped from his corner, stretching long legs before folding himself into chair again. Deo closed the parchment with care. Dania rinsed her hands at the basin, the faint scent of sage rising with the steam, then drew the cloth across her palms.
The bread tore soft beneath their fingers, crumbs scattering onto wood. Butter spread pale, cheese crumbled thick. No one hurried.
And me? Deo had set a small bowl at the table’s corner, as he always did — bread softened in milk, crumbs of cheese broken fine. I lapped delicately, whiskers twitching, pretending it was for my own honor and not his quiet care.
Romé leaned back into his armchair, tearing a piece of bread, chewing with the careless ease of one who had danced and drunk too much only yesterday. He bit into the wedge of cheese, the rind cracking between his teeth. He chewed slow, savoring, then laughed under his breath.
“Where did Matteo find this cheese? Such a fine taste on the tongue — sharp with herbs, yet softened as it lingers.”
Deo’s knife paused at the loaf, his voice even.
“From a peasant at the feast yesterday. Not the monks’ cellars — this one was made in the village past the olive ridge. Their women fold thyme and marjoram into the curd while it sets. You taste the hillside in it.”
Romé rested his elbow on the table, tearing another piece of cheese, still amused. 
He asked Deo:
“And you know all this already? From just one day, one feast?”
Deo cut a careful slice, laying it square on his bread. answered:
“Easily learned, if you ask the shopkeepers. They know each peasant’s wheel, each hand that pressed it. Sometimes they even guide the peasants, improve the recipe together. And Rinaldeschi’s house — Rinaldo himself — carries the better wheels to Bologna. Promotes them there, until people begin to ask for them by name.”
Romé’s brows lifted, half-grin curving.
“If you say so — then it reminds me of yesterday’s feast in Valiponte. Their food had taste beyond what I’ve known from villages. More refined, more careful.”
Deo’s gaze flickered, thoughtful.
“Yes. It is incredible to see how merchants and peasants co-exist here. Their kinship — it becomes the foundation of survival.”
Romé wiped his fingers, half-grinning as he lifted the cup of wine.
“You know, Deonardo — you speak so well of shopkeepers, peasants, merchants… you might make a better mayor for this town than half the lords I’ve seen.”
His tone was light, but his thoughts weren’t — he was remembering Corrado della Rocca, the careless way he let villages drain while counting only his soldiers’ worth.
Deo’s knife paused against the board. He looked up, silver hair catching the dim light, but his eyes were flat.
“The lords in Bologna do not send their men here. The safety of this valley rests only on mercenaries’ blades. A mayor’s words do not guard roads. Besides, they don’t need a mayor. The townsfolk organize themselves.”
Romé tipped back in his chair, balancing the wine glass between his fingers, grin tugging wider.
“Then what’s your solution? Don’t tell me you sit here slicing cheese with no plan behind those eyes. If it were your valley — what would you do?”
His tone was light, almost coaxing, as if he expected no more than a clever jest.
Deo’s knife pressed too hard, shaving the rind uneven. He set it down with a sharper click than he meant, shoulders taut. Romé’s teasing had struck something heavier — a thought that already gnawed at him long before he came to San Michele.
“If the lords and the bishops kept even half the agreements they seal with their rings,” he said, voice clipped, “there would be fewer widows in all valleys. Less hunger. Less death bought by coin. But they do not. They never do. And men like Rinaldo are left to barter survival, one cask at a time.”
His words cut sharper than the blade, falling into the air between them.
Romé gave a low whistle, leaning back with mock-solemnity.
“Ah — so the Cavaliere does have a sermon after all.”
Deo’s gaze flicked up, cold, but he said nothing more. His hands found the knife again, slicing clean, precise, as though to prove the matter closed.
Dania did not speak either. Until then she had eaten quietly, her attention circling the cooling jars. Then she lifted her eyes from the jar and let them rest first on Deo, then on Romé — steady, unblinking, only a silent word: enough.
I bent my head low over my bowl, whiskers brushing the rim. The milk had never seemed so urgent — lap, lap, lap — anything not to twitch my tail in the silence.

Wind on the church square 🕯♛♡🐾 afternoon
That afternoon, Dania went to the infirmary carrying jars of her distilled essences. The monks there worked to produce large quantities of herbal paste — medicine meant for the peasants and townsfolk. With her essences added, the infirmary’s remedies spread further, reaching those who could not seek her directly. After the feast, most of their stock had already been taken, so new batches were needed at once.
It was an arrangement that served both sides. The infirmary named the medicine only as the monastery’s charity; the healer’s name was not spoken. And Dania, in turn, could keep her place quiet, untroubled by too much notice. For the townsfolk, this meant they could accept remedies without openly crossing the Church’s caution — she was, after all, never formally blessed. Yet everyone knew how it worked. And some, out of respect, still brought their gifts to her door themselves.

Romé came back to the square as if the sun had been waiting just for him — an apple in hand, a laugh in his throat, boots marking easy prints in the slush. Merchants’ daughters smiled before they knew it; a novice tripped when Romé tossed him the apple; even Pietro the cellarer, bent over a keg, forgot to scowl when he caught a wink. 
For a breath, the square itself seemed lighter, its quarrels and cold set aside. But Romé did not linger with them. His stride carried him past the laughter, past the stalls and barrels, toward the shadow of the bell tower, where Ferrazo stood among his chosen men, ten in all with himself, blades already sharpened for the road to Bologna. The rest of the mercenaries watched from the stone steps, unsettled. 
That’s when Carlo Velluti drifted in — a traveling merchant, new to San Michele this year. I hadn’t seen him before the feast. He was no regular guest, yet he spoke with everyone as if he had always been there, never arriving, never leaving, only sliding in from some road you weren’t invited to follow. 
He didn’t avoid mercenaries; he treated them like a bottle of strong wine. 
Carlo Velluti sidled toward Ferrazo, the way smoke sidles toward flame. His smile was easy, his voice low enough to sound like a confidence.
“I heard you’re bound east,” he said. “Best keep your eyes sharp. I came from that road myself, and Bologna’s neighbors are restless. Skirmishes here, ambushes there — nothing large, but enough to sour the wine. I’ll not be taking that way again soon, I’ll tell you that.”
Ferrazo grunted, iron-heavy, though his men pricked their ears.
Romé moved closer, boots lazy, as if the whole square were his stage. He didn’t press in, only lingered at the edge, smiling faintly at Carlo’s slippery tone — but his eyes measured the words.
And Martino, left out of Ferrazo’s chosen ten, stood there too. His jaw was tight, hand restless on his belt, as if every word about the road east were salt rubbed in his wound. When Romé’s glance brushed him, just for a heartbeat, Martino flared. But in the end, he let it pass. Instead, he turned toward Carlo, voice pitched quick and low.
“You hear of any captains hiring? Strong steel still wanted on that road?”
Carlo’s eyes glimmered with the satisfaction of being asked.
“Oh, aye. I heard enough. Men looking for blades — there’s always coin for a strong arm. But careful. Too many banners fly in those hills now. Some carry the mark of Lord Corrado della Rocca. Pick the wrong side…” He let the words trail, as if no finish was needed.
Ferrazo’s men shifted uneasily. That name lived in every mercenary’s chest — whether as legacy or nightmare, none could say.
Ferrazo spat once, hard, and cut the air with his voice.
“We only go to the last ridge. After that, Bologna’s lords keep their own road.”

The bell rang None, and the square bent back to labor.
Rinaldo Rinaldeschi and his men hauled casks and sacks, the caravans groaning under their weight. 
In the church nearby, Tessa knelt with her veil drawn low, lips moving in prayer for the road ahead. Not for profit, not for ledgers — but for safety. For her husband’s return.
By dawn tomorrow, they would ride out.

📖November 13, 1264 Thursday
 — fluffy rightness 🕯♛♡🐾 

By dawn, Rinaldo and the mercenaries were already on the road, boots thudding frost, their breath lost into the mountain passes. The square felt thinner without them, like a cloak missing its clasp.
Before noon, in the healer’s house ran slow as honey.
Dania measured herbs in silence, the pestle tapping faintly in her hand. Romé leafed a book, more idly than yesterday, his foot tapping under the table to a rhythm only he heard. And Deo — Deo had stepped into the garden, cloak loose, face lifted to the pale sun.
I padded after him, quiet as breath. That was when the bird came.
Small, bright —  round, red breast like a drop of fire, black cap shining. It tilted its head, unafraid, as if it had carried its colors from another world..
Deo stilled. His gaze fixed, as if the world had paused at the small weight of wings. 
He stood near the rosemary bed, motionless, his silver hair touched by noonlight. The corners of his mouth softened.
From the window, Romé saw him.
I saw Romé too — leaning on the sill, chin against his arm, eyes narrowing. He looked at Deo as if he were trying to decide: boy or girl? Young or grown? The shape was one thing, but the joy in his face belonged to neither and both. For an instant Romé’s teasing dropped, and what stayed was a kind of startled tenderness.
He crossed the flagstones, boots soft on the moss. “A fine one,” Romé followed Deo’s gaze, spoke gently.
Deo did not smile. His voice was even, almost reverent.
“Bullfinch. They are rare here. They winter lower, in softer valleys. This one must have wandered.”
Romé’s eyes landing on the bird, voice light enough not to disturb, asked:“And you know its whole tale? ”
Deo answered while his eyes never left the branch.
“I know enough. They prefer alder, birch, the seeds of orchards. The males carry the brighter breast. It is how they guard their mate — with color, not with claw.”
Romé’s grin faded, caught for a moment by the strange seriousness in Deo’s tone.
“You speak of it as you spoke of merchants yesterday. As if every feather carries weight.”
Only then did Deo glance at him, quick, sharp, and yet unguarded.
“Why should it not? To survive is always weight. To guard is always weight.”
Romé did not speak again.
They stood that way a long while, the air holding its breath with them. The bird tilted, feathers trembling, a jewel against the cold sky.
Romé glanced sideways, just once. The light made Deo’s outline soft, his lips touched with a quiet color, his whole presence thinned of weight, as if the years of memory and burden had slipped for a moment.
The bird flicked its wings, scarlet breast flashing once in the cold light, and darted upward, vanishing over the cloister roof. Romé tilted his head back to follow it, then lowered his gaze to Deo —  not as silver-haired scholar or cavaliere, but as a young one, still undecided to be a boy or a girl, watching the world with undiminished wonder.

However, the quiet did not last.
By midday, noise rolled in — wagons creaking, oxen lowing, men shouting as blocks of marble ground against wood and chain. The cloister shook with it. Stone on stone, heavy as thunder.
I flattened my ears. Too much clatter for a holy house. Even the pigeons scattered from the bell tower, wheeling in the pale sky.
Deo stepped out the cloister garden, down to the church square,  his cloak drawn, his face shadowed.  There waited Luca, words spilling faster than his breath.
“They say— among the monks, there is talk. Donna Donata Maffei, she means to lay marble pavement around the altar. But it was not announced. No seal, no public word. Only rumor.”
And then she appeared — not rumor at all, but fact.
Donata crossed the cloister’s edge. Even in black she glowed — fur-lined cloak trailing, rings flashing, braid polished so fine the sun found it at once. Her wealth did not whisper; it clanged, brighter than the oxen’s chains, and peasants bowed longer than they meant to.
Deo’s eyes narrowed, but he said nothing.
Then — Bianca. She played alone with her white dog at the margin. The other merchants’ children kept their laughter apart from her — perhaps her dresses were too fine, perhaps her eyes already held too much knowing. Yet she seemed content, her world folded into the dog’s quick feet, soft moments without fear.
Deo’s gaze lingered, Dania circling in his mind beside the little girl. A likeness trembled there — not likeness of face, but of being. What bound them? The question flickered, then fell away, leaving only the image of child and healer, each in their own quiet.
He walked back to the healer’s garden. Just standing here, his day settled again, even while marble carts dusted the world outside.
Moss still green, though most herbs slept in their roots. The air still remembered their summer voices. When the sun touched the frost, the dark soil turned wet, smelling faintly of sweet mint and woodsmoke.
At the far side, the short wall leans toward the valley — now a pale, frozen ocean, its slopes glazed with ice, its forests drawn in black against the glare.
Hawks circle high above it, fixed to the cold currents. They do not dip lower — as the truth of living things that never bend to please the world.
East of the garden lay her day time rooms — her working studio, the dinning room with kitchen, the treatment room and storages for refined herbal medicines.  
Her books, her jars of teas and essences, her fine tools that hum when you look too long.
Dania sat at her small desk, her hair turned gold by the slant of afternoon light. She wrote her notes, sketched her cures, and sometimes forgot to eat.
Romé had fallen asleep on the wooden couch by the window, two arm-lengths from her — close enough to belong, far enough to be polite. If she was weary, he never spoke; he let the sound of his breathing say I’m here.
Deo slipped in on soft steps, closing the door soundlessly, almost like my paws. He found his seat among parchments; I found mine on the cushion he set in the warm patch of light.
This is her place.
Since Romé and Deo joined her, it became our place.
She moves here like a current through ice — never rushed, never yielding.
It smells right.
It sounds right.
It keeps every quiet, fluffy rightness the outside world cannot touch.





📖 November 14, 1264 Friday
 Intrusion🕯♛♡🐾 — Friday
Clouds pressed low from dawn, and by noon the first flakes were already tumbling — not the sharp kind that sting, but slow, tiny snowflakes that drifted down as if they were shy. The air wasn’t cruel cold; it was the softer kind of winter, when snow falls and the world hushes, and the hearthfire feels warmer only because you can see the flakes melting at the glass.
I curled by the fire, whiskers twitching each time the logs hissed. Outside, roofs turned pale, cloister stones caught a dusting. Inside, the flames burned orange-bright, licking the wood as tinny lovers. 

After the bells of None, Niccolò came in with the smell of outside on him — cold air, wet stone from the cloister stairs, a trace of snow melting on wool. A bundle of books rested against his chest, bound in twine, edges curled with damp. 

Dania and Deo were just closing the day.
She was setting a cork into a small jar of resin, while his shadow moved along the shelf to place sprigs of winter green. It was the kind of quiet that settles in without asking — the dignified sort of quiet I consider my personal property — and I had every intention of keeping it until the chicken leg appeared.

“New arrivals for your shelf,” Niccolò said, easing the bundle down onto her desk.
Dania’s hand met his mid-air for the last book, and the way they exchanged it — eyes, breath, nothing else — was more greeting than most people manage in ten words.

Then he looked at Deonardo, who was wiping the brass edge of her grinder with a cloth.
Niccolò smiled in a tone light enough to sound casual,
“I thought your studies were in books — seems they’re expanding into practice?”

I licked a paw, pretending indifference, but inside I purred at the balance of it — cautious small talk over something that was clearly not small. Delicious.


The quiet had just begun to deepen when the stairs outside began thudding.
Not one set of boots — a whole column of them.
I flattened one ear. That many boots rarely mean anything good for a room’s temperature.

Pietro appeared first, square as a wine cask, cheeks pink from the climb.
Behind him, six monks in work robes, each with a sack or crate slung over their shoulders. 
“Suppliements. ” Pietro shouted, while they flood into the healer’s storage, sacks hitting floor chaotically. 
No reverence for shelves, none for neat vials — just dropped, the way a man throws down his coat when he’s already thinking about the wine he’ll have next.

What is happening?
Every piece of furniture here was confused.

Deo was already moving before they reached the threshold.
He slipped past Niccolò with a murmured “Excuse me” and crossed the garden.
Matteo was already there, waiting to report.
A few words passed between them — low, swift — and the servant vanished toward the storage room.


The small storage seemed filled in no time. Pietro, oblivious, wiped his brow.
“More in the cart — we’ll put the rest in the treatment space.”

That was Romé appeared to stop them im the middle of the doorway– he was hiding in the treatment room, sometimes stayed there with the nostagia of how how met Dania the first time there and cried rediculously.
 “Best not,” he said, a hand to one monk’s shoulder, a glance of charm, “Plenty of room in the storage once Deo’s lads finish arranging.” 
The whole troop was redirected from the treatment  turned without argument –monks grumbled and filed back to storage.
Pietro shrugged, already thinking about the wine cupboard.
The air began knitting itself back together — and then the stairs carried a different sound.
Pietro appeared first, square as a wine cask, cheeks pink from the climb. Behind him came six monks in work robes, each with a sack or crate slung over their shoulders.
“Supplements,” Pietro shouted, while they flooded into the healer’s storage, sacks hitting the floor chaotically. No reverence for shelves, none for neat vials — just dropped, the way a man throws down his coat when he’s already thinking about the wine he’ll have next.
What is happening?
Every piece of furniture here was confused.
Deo was already moving before they reached the threshold. He slipped past Niccolò with a murmured “Excuse me” and crossed the garden. Matteo was there already, waiting to report.
A few words passed between them — low, swift — and the servant vanished toward the storage room.
The small storage seemed full in no time. Pietro, oblivious, wiped his brow.
“More in the cart — we’ll put the rest in the treatment space.”
That was when Romé appeared, stopping them in the middle of the doorway.
He had been hiding in the treatment room, sometimes lingering there with the nostalgia of how he had first met Dania — and cried, ridiculously.
“Best not,” he said, a hand on one monk’s shoulder, a glance of charm. “Plenty of room in the storage once Deo’s lads finish arranging.”
The whole troop was redirected without argument. Monks grumbled but filed back to storage.
The whole troop was redirected without argument, monks grumbling as they filed back to storage. Pietro glanced in — Matteo and Luca had already displayed every sack in order on the shelves, the rest loaded neatly into place. He only shrugged, already thinking about the wine cupboard. 

The air began knitting itself back together — and then the stairs carried a different sound. This time it was lighter, wrapped in the perfume of warm wool. Lorenzo appeared first, a polite smile on his face; on his arm, Madonna Donata Maffei — velvet like midnight water, fur collar framing a face still beautiful, and perfectly aware of it.

Her eyes touched Dania once — the kind of glance you give a chair before deciding it’s not your style — and then slid on, lingering on Niccolò, on the fire, on the arrangement of the room she had not arranged.
I unsheathed one claw into the rug. It helps me stay polite.

Niccolò, good man, was already guiding her to the safest chair — farthest from Dania’s desk and jars.
By the time she sat, it looked like her idea.

Somewhere between one blink and the next, Dania was gone — no rustle, no latch click — just gone, like a ripple smoothing over water. When Romé and Deo came back in together, the chair where she had been was empty.

Donata rose slightly, smiling to them with the air of a legislated hostess whose timing is always perfect. Her voice opened wide enough to draw them in, her posture loosening into the kind of ease that makes others forget the rhythm of the room before she entered. And the two men, with their announcing beauty, seemed to wonder for a moment what had become of the space they had stepped into only five minutes ago.
Romé bowed — just deep enough to flatter — and whatever he murmured made her laugh without knowing quite why. Deo inclined his head with the precision of a man drawing a border on a map. Niccolò’s glance toward Romé was quick but clear: I kept her in the safest chair. The rest was not mine to stop.
Then Carlo Velluti was suddenly there, already talking, untying a wine-red ribbon from a parcel. A curl of citrus and cured fish rose with it, chasing away the last threads of garden rosemary.
From the corridor, Pietro’s voice boomed about “found the better wine for dinner.”
The talk thickened, the fire’s heat bent toward the chairs — and that’s when Romé saw it. Not Donata herself, but the ripple she cast: her maid, soft-footed but too sure of belonging, hovering over the healer’s desk, clearing a space to display gloves for her mistress.
Romé was beside her before she touched the ledger. Not seizing her wrist — too obvious — but shifting the chair just enough to block her way.
“Best keep this for daylight,” he said, all warmth, though his eyes marked the desk as ground.
The maid withdrew with murmured assent. Donata glanced over, catching only the charm.
From my perch, I also saw Deo ready in his guard duty — Romé moved before he could. His shoulder eased; the line of his neck softened.
While Donata sat at ease in the healer’s studio, her maids and the monks worked quickly in the dining room. A white cloth spread, cups polished bright, lamps trimmed. They laid out what she had brought and what the cloister could offer: river trout glazed with honey and rosemary, almonds roasted in sugar, figs stuffed with soft cheese, pork slices laid on fennel, pears stewed in wine. A flagon of malvasia was uncorked, its sweetness meant to flatter the tongue before the salt of olives and smoked fish.
By the time the platters were set, the healer’s table looked less like Dania’s place of plain bread and herbs and more like a merchant’s hall. Donata’s feast spoke loudly, as humans always seem to confuse noise with plenty.  
I flicked my tail under the bench. 
A maid slipped in, head bowed, voice soft but rehearsed:
“Madonna, the dining table is ready.”
Donata rose at once, velvet trailing like she had always been the mistress of this house. With a glance that gathered them all — Niccolò, Deonaldo, Romé — she moved toward the door, her posture not asking but assuming they would follow.
 Lorenzo, ever the churchman, would offer Donata his arm like it were her right, keeping her framed in honor. And Carlo, sly as smoke, he’d chatter, thanking, praising, drawing the others along his ribbon voice.
Romé lingered a half-step behind, his grin quick as a knife drawn in play.
“So it seems, Cavaliere,” he murmured toward Deo, “that Madonna Donata has turned healer’s walls into her own hall. Even the bread here carries her seal tonight.”
The dining room glowed as though it had been waiting all day — lamplight on polished wood, silver dishes breathing steam. Donata swept to the main seat on Lorenzo’s arm. Carlo drifted just behind, his voice flowing, praising her presence as if the room had been hers forever. Romé’s step was loose, amused; Deo’s measured, his silence tight as a sealed jar.
Then Dania came — not leading, not lagging, simply present. Yet the air shifted around her, like a flame steadies when a door closes.
Only then did the child appear — Bianca, long hidden in her mother’s black silk, now perched on a chair, hands folded, the white dog curled at her feet. The lamplight caught her face, pale with childhood, her eyes retreating as though she bore an orphan’s solitude, though her mother sat only an arm away.
When all were steady in their seats, the maids filled the glasses, wine rising dark and full. Lorenzo lifted his hand, his voice in that perfect middle place between blessing and toast.
“A moment before we eat, to thank Madonna Donata Maffei for her generosity. For tonight’s table, and for her ongoing support to our work. Without friends such as these, much would be harder.”
The words were smooth as polished wood, practiced enough to please both priest and merchant. Donata inclined her head with measured grace; Carlo glowed as if the toast were half his own. Romé tipped his cup in salute, lips already curved. Deo’s glass remained steady, fingers exact on the stem.
Lorenzo’s hand did not lower. His voice carried on, warm and measured, meant to honor yet also to bind.
“Tonight’s table is only part of Madonna Donata’s gift. For the coming cold, she has pledged coin for the heating of our halls, that the brothers may keep their duties even in the hardest frost. And with her family’s marble, the altar of San Michele will be paved anew — in memory of her late husband, whose name shall be carried in stone as it is in prayer.”
“It was never mine to give,” she said, her voice lowered into the tremble of humility. “These were my husband’s wishes — he spoke of them often, though God called him too soon to see them done. I am only his vessel, keeping faith with his word, as any good wife would. A widow has no virtue but obedience — first to her husband living, and then to his memory.”
The words poured soft, Donata laid her hand on Bianca’s small shoulder — as though presenting her, placing the child into the gift as one more offering. Bianca shivered under her mother’s touch, seemed to fold smaller into shadow. Around the table, no one spoke; the silence itself seemed to seal Donata’s virtue in place.
“My husband spoke often of Abate Lorenzo,” her tone softened into widow’s tremor. “He admired him greatly — his wisdom, his tireless work for this valley, his ability to guide both monks and people with steady hand. Many nights, he would speak of it to me, as if the very order of our house took courage from knowing such a shepherd stood here.”
She drew a breath, letting it break slightly, as though grief still pressed the ribs.
“It was my husband’s belief that this monastery prospers not from stone or coin, but from Lorenzo’s vision — the vision of harmony, of each man and woman set rightly in their place, the weak lifted, the strong guided, the wayward corrected. Without such guidance, wealth scatters, charity falters, faith itself withers.”
Her hand, all rings and pale skin, rested again on Bianca’s shoulder. The girl shrank, but Donata’s grip only staged her closer, as though presenting proof of legacy.
“I am only the vessel of his wishes,” she continued, her voice steady now, “but I share his conviction — that without such order, all our gifts are wasted.”
Lorenzo bowed his head, his voice tempered, the kind that sought to ease weight away from one shoulder onto many.
“Madonna Donata, you are generous in repeating his words. But the work here is never one man’s vision. It belongs to the valley — to the brothers, the houses, the people. I am only its servant.”
Donata’s smile widened, soft yet edged. She inclined her head as though in concession, yet her voice held steady.
“Of course, Father. But service only bears fruit when rightly ordered. Gold alone does not sustain a house. It is obedience, deference, humility that keep it whole. That was my husband’s belief — and it is mine also. Without such order, all gifts are wasted.”
Her eyes lingered — not on Lorenzo now, but sliding across the table, brushing the air near Dania before settling back with deliberate calm.
“Order needs tending. It is not given once and left to chance. It must be kept — by knowing one’s role, by serving without question. There is strength in obedience. Dignity in humility. Especially…” her tone fell a shade lower, “…for women.”
The smile was still on her mouth, but the softness had gone.
“A woman’s virtue is not in striving for equal ground, but in standing steady where she is placed. Gratitude is not a thought; it is an act — shown daily in respect, in deference, in loyalty to those who guide and provide. A woman who forgets this… forgets herself. And risks not only her own standing, but the harmony of the house that shelters her.”
Her gaze drifted — not quite to Dania, but grazing the air near her like a shadowed finger.
“And in a place such as this, built on faith and service, there is no greater danger than a servant who thinks herself a master. That is the root of disorder, of pride — the oldest sin.”
The lamplight did not flicker, yet the air seemed to shift. Romé’s grin seemed to be weighing which mask would serve best.
Lorenzo’s smile tightened at the edges, a host’s smile that must hold, no matter how much it strains, his voice smooth as poured oil.
“Madonna Donata speaks with care — words her late husband would no doubt have counted as his own. We are grateful for her example, and for the generosity that made tonight’s table possible.”
He lifted his cup, not waiting for dissent. “Now — let us honour both memory and life with the gifts before us. Eat, drink, and give thanks for the season of St. Martin.”
Niccolò seized the gap, refilling glasses, loosing a quiet jest about monks and goose fat. A few chuckled, grateful for the release.

Deo, however, had not moved his gaze from Donata.
His voice was calm when it came — steady, low enough to invite leaning closer, yet carrying over the table as though it belonged there.

“In my own service,” he said, “I have been guided by a woman whose governance is as precise as it is merciful. Lordess Aurelia di Vercado, she does not keep order by reminding others of their place, but by knowing her own so completely that no one doubts it. Her household is strong because she rules it as a captain rules a ship — not by chaining the crew to the deck, but by steering so true that all follow willingly.”

A pause.
“She has my loyalty because she never asks for it in words. She earns it, every day, in the way she governs.”

That one landed — I could feel it under my paws. Not a scratch, not yet. But the floor shifted beneath Donata’s chair.

The words were shaped like a compliment — and they were — but not for anyone at the table.
The space between them and Donata was the space between two standards that would never align.

Niccolò’s fingers drummed once on the stem of his cup, a faint smile rising as though he had just caught the tail of a memory.
“Ah,” he said, as if to himself at first, “then we must be speaking of Porta d’Argento.”

The name settled over the table like a drop of ink in water.
He leaned back, eyes half-lidded in the comfort of a familiar story.
“I have heard of its gates — silver in name, and in certain lights, silver in truth. A palace set above the river, its library rumoured to hold a gospel bound in lapis, a hunting horn once carried by the Count of Canossa, and maps so old the vellum turns to lace at the edges.”

Romé’s mouth curved; he caught the thread instantly.
“I’ve heard,” he added, “that the kitchens there could hold their own against any in Lombardy. There’s a poem — half praise, half recipe — about a feast of river trout stuffed with rosemary and wild figs. It claims even the stewards wrote verses after that meal.”

Pietro laughed, broad and unashamed.
“Verses I’d happily test in person. Though I’d settle for a cup from those vineyards — what was the vintage called, Niccolò? The one that glows red as garnet?”

“Il Sangue di Santa Giulia,” Niccolò supplied, enjoying himself now. “A name to tempt saints and sinners alike.”

Carlo had been quiet until then, but he leaned in, voice low and shrewd.
“All this talk of gates and rivers reminds me — I hear the road from Monteferrano has been troublesome of late. More patrols. More tolls.”


The room’s warmth didn’t falter, but I could feel its direction change — like a boat turning downstream without anyone announcing it.

Niccolò’s naming of Porta d’Argento had set the table into an easy, rolling current — relics, poetry, wine, roadways — each voice catching the thread and passing it on.
Then, without warning, Donata’s voice cut across it.

“Bianca,” she said, not loudly, but with a firmness that cleared a space around the word. “Your knife.”

The girl froze mid-cut. The candlelight picked the blush at her cheeks, the way her eyes flicked to her mother — not for understanding, but for permission to move again. Before this moment, she had been no more present than the chair she sat on — in her polished silk, she was the mirror of Donata’s ambition: perfectly set, perfectly silent until summoned.

Donata reached to adjust the girl’s wrist, speaking on as if explaining to the whole table.
“There is no elegance in rushing. A lady’s grace is measured in control, even at the table.”

Lorenzo kept his eyes on his wine. Niccolò’s smile cooled a fraction.
Deo, who had been sitting in the steady poise of a man among equals, shifted just enough for me to feel the temperature drop in him. The insult was not his, but he took it as one — the mark of a woman who knew how to occupy a room.

For a moment, the whole conversation that had been swelling so easily was gone, replaced by the silent theatre of correction. Bianca moved her knife as instructed. The attention was on her — a place she wore like a garment handed down for this very moment.

It was Dania who broke it.

She had been the calmest presence at the table all evening, her silences unthreatened, her glances unprovoked. But now, she leaned slightly forward, her voice warm but carrying that still, cutting clarity I’d heard in the square.

“You know,” she said, turning her gaze toward Carlo as though Donata’s interruption had never happened, “the tolls on the Monteferrano road aren’t only pressing merchants. The patrols have begun questioning grain caravans bound for the bishop’s granaries. If they delay them past the feast of Saint Hilary, half the hill parishes will be without winter allotments.”

It was a sharp needle of a comment — political, precise, impossible to dismiss — the kind of observation not even most men could make without being an officer of the city or a sworn envoy.
And in that moment, the table belonged to her.








📖 Evening in snowfall🕯♛♡🐾 Cacifer’s Storybook


Dania’s words didn’t just land — they settled like a new map on the table.

Carlo blinked, then leaned forward. “That would mean they’re controlling grain as well as wine,” he said, voice dipping into the habit of a man who speaks more freely than he should when the right company is at hand.

Pietro’s laugh was short, as if to shake off the weight of the topic, but the sound had a wariness in it.
“Questioning granaries is the same as questioning the bishop’s seal. Even Monteferrano’s new lord should know better than to stir the Chapter’s temper.”
Romé’s brow had already tightened, “They’ll test until someone pushes back.”

It should have ended there — a few nods, a turn back toward softer ground.
But instead, the space between words filled with Deonardo’s voice. Calm as a man describing the weather from a hilltop — except the hilltop was high enough to see more than most. “Three caravans last season from San Lazzaro were stopped at the bridge at Piova. One was turned back for lack of ‘proper charter,’ though the bishop’s seal was on the lead crate. Two paid double passage. The road captain claimed it was on account of a patrol increase after last year’s raids. Convenient, for the coffers collecting it.”

Carlo’s head lifted a little at that. “Not many keep track of Piova tolls.”

Deo only smiled, “Grain’s only useful if it arrives. The river crossing is the hinge — slow it, and you control more than bread.”

Niccolò swirled the wine in his cup. “You’ve been here less than a month, and already you sound like a man who’s walked every road between here and Monteferrano.”

“I watch where the wagons come from,” Deo said simply, cutting into his quail as if that were the end of it.
Then, almost as if remembering a minor curiosity — “And whether they come back empty.”

Pietro leaned in, grinning around his goblet. “Empty, lately, from Monteferrano’s side — unless you count soldiers as cargo. I saw three carts late in October, all straw and pikes on the return.”

Niccolò’s brows rose. “Straw? So they’re quartering somewhere close.”

Carlo set down his knife. “North of the river, I’d wager. The old watchtower near Ponte delle Volte has been busy. Smoke from the chimneys at dawn, and that’s not shepherds.”

Romé, who had been silent through the talk of tolls, glanced across the table. “If they hold that tower, they hold the bend. No merchant would risk the crossing without paying twice what it’s worth.”

The pattern was taking shape in the air, without anyone naming it outright — troop movements, new outposts, the tightening of a road that should have been free.

The talk turned, naturally, to the bend itself — how the road narrowed between the cliff and the water, how caravans could be stalled for hours if a single cart was stopped for inspection.
Someone recalled the year the bishop’s grain ships had been delayed, and the market towns ran short before All Saints’ Day. Deo said nothing more than a thoughtful “Ah” at the right moments, but the table was building the map for him, every voice offered another piece on the compass.

From my place under the bench, I could see the weight shift.
Donata’s smile was still fixed, but her place at the centre had gone. The conversation no longer flowed around her; it was sliding down a different channel entirely — one she could not read, and where her gold bought no entry.
When the last course was cleared — almond pastries and quince paste — the guests began to drift in twos and threes toward the cloister walk.
Donata was first to rise. Her goblet was still half-full, her smile perfectly intact, but the light in her eyes had narrowed — as if the air had thinned around her and she had decided not to waste breath on the rest. Bianca followed without a word, trailing her mother’s shadow like a folded fan, the silk of her dress whispering against the rushes.
The others lingered only long enough for final sips of wine, scraps of talk about patrol routes and frost on the valley road. The warmth of the room was already thinning, voices fading beneath the crackle of cooling logs. Candlelight dimmed without the bodies to catch it.
Deo rose without hurry, offering courtesies as though the evening had been nothing more than quail and wine. Matteo and Luca moved quietly, setting the room back into its old shape. Dania did not look at Deo — not directly — but the stillness of her face told me she was keeping something steady inside. She left for the western garden rooms without a word.
Outside, snow kept falling, whitening the cloister stones in silence.
Romé and Deo fell into step without speaking — the way they always did after an evening at the healer’s, as if the path back to their rooms had been worn by more than boots. Sometimes they traded fragments of the evening’s talk; sometimes it was only the sound of their steps and the brush of cloth when Deo’s sleeve caught Romé’s.
I rode Romé’s shoulder for the walk — a better perch than the bench tonight, where too many feet shuffled past. His hood was down despite the cold, and I kept my paws hooked into the fold of his collar, close enough to feel the heat where his neck met his jaw. Deo’s hood was back too, the silver of his hair catching what little the lamplight gave. In him it was sharp as a blade’s edge; I had seen it softer before, when he was otherwise. Romé’s eyes caught it once in passing, not long enough to be called staring, but long enough for me to notice.
There was something in that narrow space — a faint trace of myrrh and cedar from Deo’s cloak, warmed by the body beneath. Romé didn’t lean closer, but he breathed deeper, as though the cold had sharpened the scent.
“You watch the road as if it were a chessboard. Not to collect facts for their own sake,” Romé said, voice low enough for the frost to keep. “You’re feeding something.”
Deo’s mouth curved — not much, just there. “Every man feeds something. The trick is knowing when to starve it.”
We passed under the last arch over the garden path. The guesthouse was nearer; the air smelled faintly of banked coals. Somewhere in the valley, a dog barked once and stopped.
Romé sighed, chilled at his shoulder. “Next full moon will be the real winter, then Christmas.” He didn’t smile when he said it, but his voice carried a softness that wasn’t meant for the feast so much as for the memory of winters past. “Where I’m from, it’s not the tree or the feast that makes it. It’s whether your house is full.”
Deo’s reply was quiet, and in the pause before it, his breath. “I guess that is how yours is?”
They both stopped walking. No reason. A lock of Deo’s hair moved in the wind, and for a moment it was hard to tell whether the light on it came from the torches or from his own glow. He glanced upward once, to the sky, the cold stars behind the clouds. “The Hunter’s belt has shifted,” he murmured, almost to himself. “By midwinter, it will hang above the cloister gate.”
The words came like a note remembered from an old song — not for Romé, not for me, but the habit of someone who keeps watch because no one else will. A man who measures the stars by heart so the seasons cannot slip past unmarked, even if the years already have.
A thin line of candlelight showed in the monks’ dormitory windows. It trembled with the falling snow, uncertain whether to stay with the living or join the whitened stones.
Romé said nothing more, but his gaze lingered on Deo as their steps started again, as though testing a thought he hadn’t meant to have. If he guessed why Deo always matched his steps on these nights, he kept it folded away.
By the time we reached Romé’s door, my paws were numb from the night air. I hopped into his arms, then to the bed, curling over his face before he could protest. The rest of the night belonged to us — his dreams, my warmth. Whatever Deo brought back to his chamber.


📖 November 15, 1264 Saturday
  — Snowing Ridding🕯♛♡🐾 Romé and Fiamma in Snow

The morning was all white breath and iron sky.
Snow lay thick over the slopes, soft in places, hard-crusted in others, waiting to trip hoof and man alike.
Romé rode Fiamma like a flame riding the wind. Snow held the valley quiet, but not him. Fiamma’s red coat burned against the pale world, steam rising from her flanks with every exhale. She tested the ground with quick hooves, clever and proud, while he bent with her, body and saddle one line.
I crouched on a branch above, my paws cold, my eyes warmer.
It was not only training I saw.
It was a conversation — every shift of his weight, every twitch of her ears.
The perfection of his body, dark copper under the winter light, muscles drawn tight and smooth as the stream before a waterfall.
Strength not for show, but tuned — to the horse, to the land, to the rhythm of breath.
He leaned forward when the snow grew deep, rose when she climbed the ridge, hand stroking her neck not as master, but as companion.
And when she tossed her head, unhappy at the sting of wind, he laughed — laughed as though the cold belonged to them alone, as though every flake that struck his cheek was another reason to keep riding.
He did not cut the snow with a sword today.
But he mastered it with his seat, his patience, his ease.
The way he loves is the same: not conquering, but resonating.
Not demanding, but staying — steady as the horse beneath him, fierce as the fire in her breath.
I tucked my paws closer to my fur.
This, too, was a kind of prayer, with body, discipline, and joy, carved into the white silence of the mountain.
—Cacifer 🐾
(observer of horses, keeper of unspoken loves)

When the bell rang Terce, Romé came back from the riding still burning like the sun had followed him down.
Snow clung to his boots, steam lifted from his skin where sweat met frost. His cloak slipped from his shoulder, and he stood there as if the cold had no claim on him — copper flesh alive, breath full, eyes bright as if the ridges themselves still ran in his blood.
He asked her softly, almost sheepishly, if he might wash in the treatment room. Dania gave the smallest nod, as if the breath of mountains had always passed through her doorway.
Deo brought him hot water. His hands steady, though I know his chest wasn’t.
When Romé stripped down, upper body bare in the lamplight, the truth of him showed — the lean cut of shoulders, the slope of muscle caught between youth’s grace and a fighter’s strength, carrying storms in his skin.
Deo looked — only a glance, quick, but caught. The kind of glance that lingers after the eyes have already turned away. He busied his hands with the basin, dipping the cloth, wringing it too tightly, water dripping across his knuckles.
His voice came low, steady, almost clipped:
“The ride must have been hard. Snow weighs more on a horse than on the rider. You should see to Fiamma before yourself.”
Romé took the cloth, dragging it across his neck and shoulder, droplets streaking over the dark copper of his skin. His breath still held the rhythm of riding, quick and alive.
“In my hills, there’s no snow,” he said simply, as if the thought itself carried heat. “But the stone paths cut sharp — we’d pack moss in the hooves, keep them soft. My father swore by olive oil. Not for cold, for dust.”
He rinsed his hands, splashing water up to his face, then looked back at Deo without his teasing notes, only the ease of a man glad to be speaking while his body still remembered the gallop. “Fiamma’s strong. She doesn’t mind the sting — not yet.”
Deo nodded once, as if approving the mention of moss and oil. His hand tightened faintly on the jug before setting it down. “On snow, the trick is not only the hooves,” he said, voice steady, almost too steady. “The lungs take it worse. A hot chest against cold air — it weakens them. Next time, ride shorter.”
He turned as if to busy himself with the folded cloths, but not before his gaze slipped — just once — along the curve from Romé’s narrow waist to the line of his hips, the sweep that no armor, no cloak could disguise.
The sight struck him not as indulgence but as disbelief — as if a body so precise, so alive, could hardly be real. His breath caught, and he forced his eyes away again, left before Romé could glance up.
He slipped back into the healer’s studio. Dania was bent over her desk, quill steady. She didn’t look up at once, but when she did, her eyes rested on him a fraction longer than usual. She saw the slight disarray — the way his breath had not quite settled, the way his hands seemed too quick to reach for parchment.
Her voice came soft, almost thoughtless, as if the words escaped before she weighed them:
“Did you also go riding this morning?”
The question landed strangely in the air, too innocent, almost foolish, and for a heartbeat Deo could only blink at her. I flicked my tail against the desk leg. Ah, my healer — always so precise with herbs, so sharp with wounds.
I leapt from the sill straight onto his folded cloak, paws kneading the wool as if I’d been waiting all morning for this. “Yes, yes, of course he rode,” I purred into the fabric, “straight through storms and snowflakes, all without leaving the healer’s rooms.”
Deo gave me that look — all wide-eyed innocence. I curled tighter, kneading, covering it in fur. If he couldn’t answer, I would. “He smells of frost and horses,” I teased, rubbing my cheek along his sleeve. “Or perhaps that’s just the perfume of unsaid things.”
Ah, how warm humans get when their silence is tugged into play.


📖 November 16, 1264 Sunday
 The Floor Beneath All Floors🕯♛♡🐾 Sunday  Waning Crescent

Sunday, the altar stood ringed by boards and canvas.
Casa Maffei’s marble paving had begun the day before. Saturday noon, as Donata Maffei’s polished carriage rattled back toward Valdiponte — velvet curtains swaying, wheels striking sparks on the stones — Maestro Giusto, the master carver, and his men arrived at San Michele. Their boots were cracked, their hands lined like dry riverbeds, yet they carried their work with a steadiness richer than coin.
These craftsmen had served Casa Maffei for generations. They wore no rings, but their wrists bore the memory of stone — red streaks from granite dust, white callus hardened from the rasp. To them, marble was no commodity to parade; it was a covenant. Quarried with care, measured against the light, cut so each vein of color spoke the truth of the mountain it came from.
They set to work at once, unwilling to lose even an hour of daylight. Giusto had overseen the sketches months before: the altar’s surround drawn in compass-sharp lines — white for purity, green for eternity, red for sacrifice. He reminded his men that each cut must be accurate enough to seal the ground so that centuries of weight could not crack it. Their pride was not in being seen, but in vanishing into the floor they laid — so that worshippers might walk on their work without ever knowing their names.
When they crossed the cloister, their voices stayed low. Their greetings to monks were brief, deferential. They carried themselves not like men arriving with power, but with an old humility: the kind that says craft is service, and service is survival.
These stones carried their own long story: I smelled the quarries still on them, salt and mountain rain. I could almost hear the men at Carrara tapping with hammers, listening for the right ring. I could see the sketches Maestro Giusto once pinned to his wall, circles and squares tangled like prayers — white for purity, green for eternity, red for sacrifice. A whole sermon pressed into marble before the slabs ever met an ox-cart.
Now it was muscle and care. One mason knelt, trowel flashing, laying the mortar thin as cream. Another lowered the slab inch by inch, tapping with a padded mallet until it sang true. Wedges whispered in, seams swallowed lime, dust scattered soft until the joins disappeared.
Today, the hammers slept. Their handles leaned against the wall, the marble veined and waiting under canvas. The altar stood like a captain on a half-built deck — boards and scaffolds all around, but its cloth laid smooth, candles steady.
I curled myself on a beam in the rafters, where the dust still smelled of lime and wet stone. It clung to my whiskers, sharp as salt. Below, the monks gathered tighter than usual, voices rising like they had to press past the planks to reach heaven.
Incense drifted, but it tangled with marble dust, so the air was half prayer, half quarry. A few peasants crossed themselves twice, as if to be safe. Some muttered it was unlucky, praying on unsettled stone. Others whispered it was blessing, as if new life was being pressed into the church’s bones.
But when the bell rang and the Host lifted, even the stones hushed. For that hour, they seemed to hold their breath with the rest of us — boards, canvas, and waiting marble alike, all folded into one silence.
I have seen stones laid lately, around one and half century ago. In Rome, in the belly of San Clemente, where life and death sleep one on top of the other like cats in winter.
They say Clement drowned with an anchor on his throat — a weight fixed forever, holding him fast beneath the waves.
Men thought it cruel. I thought it honest. Anchors were not made to kill. They were made to hold. To root a ship against tide and storm. To say: here we stay, here we do not drift.
And so men told it: God used mortality to anchor the Mundane, and the basilica became man’s harbor. Marble floor instead of shifting water. Stone veins instead of foam.
But I, who have padded through their chapels, know: anchors rust, ropes fray, ships rot even in safe harbor. No stone has ever kept man from drifting where his hunger pulls.
There, The first slab was laid in 1110, under Paschal II, I watched men spread lime thin as milk. I remember the sound most: the mallet tapping soft, not to break but to listen. Tap — wait. Tap — wait. As though the stone might answer back if the fit was true. And when it sang, they smiled.
And yet, not long after, I saw emperors march. In 1143, the people of Rome rose, crying for the Commune — tearing the city from papal hands, if only for a breath. They raised senators where bishops once ruled. For a season, it seemed the stones of San Clemente rang with a different promise — freedom louder than liturgy.
But the years turned. I watched Frederick Barbarossa come down in 1155, his iron crown bright as flame, his horse trampling streets that had prayed for peace. Later, in 1179, I heard the Third Lateran Council, three hundred bishops gathered — decreeing order while outside the poor still begged on marble steps.
And darker yet: in 1209, Innocent III called the crusade against the Cathars. Whole towns burned — Béziers, Carcassonne — while marble saints in Rome looked away. In 1221, I saw the Dominican friars settle into San Clemente itself, sworn to guard the faith with words sharper than swords.
By 1241, the city shuddered again when Frederick II’s men besieged it, even cardinals taken prisoner. And by 1252, I heard hymns for the canonisation of Clare of Assisi — a woman of poverty raised to the altars, while marble floors gleamed under jeweled feet.

Still, they lay their slabs as if each one were a new anchor. In San Clemente, the floor gleamed like a frozen river, circles knotted, spirals looping back on themselves. Not decoration — no, a promise. A map reminding us that even when faiths die, they become foundations. Temples on tombs, martyrs on markets, basilicas on basilicas. Each age lays its stone over the last — so the living may walk on the memory of the dead.

The Sunday High Mass of St. Michele in Vallefredda was nearly over. I stretched my back long, claws against the stone, tail curling to the tip — ah, yes, enough of anchors and emperors. I padded away from San Clemente in my mind and back to where the fire waits.

I can already smell it: resin crackling, chestnuts hidden somewhere in the coals. Dania’s hand steady with her jars, Romé draped careless in the armchair, his boots by the hearth. And Deo — Deo will pour the milk into my bowl without a word, bread softened in it just right, as if he’s known my hunger longer than I have.
That is the only floor I trust: warm boards under paw, the three of them gathered in their quiet evening. The world outside may anchor itself in stone, but my eternity is there — purring, full, firelit.


📖 November 17, 1264 Monday
Lorenzo’s visit🕯♛♡🐾 Monday, lightly cloudy
Monday became even quieter: Romé had taken Fiamma into the snow after lunch, training her steps on the white slopes. For once he heeded Deo’s advice about the cold — waiting for the warmer hours so his breath would not bite too sharp in his chest. Without him, the healer’s place felt different: less fire in the room, more space for silence to stretch.
The knock came just after midday. Snowmelt dripped from the cloister arches, the sound soft against the stillness of her studio. Dania opened the door, and Lorenzo bowed with quiet courtesy.
Lorenzo stood just inside the threshold, his cloak still carrying the damp of thaw. Dania motioned him to sit, but he only bowed, his hands folded neatly.
“Madonna,” he said, “I wished only to thank you for your forbearance. Friday’s dinner was not arranged as it should have been. Your rooms deserved more respect.”
Dania inclined her head, the motion slow, her gaze steady enough to meet his courtesy without denying it. Her voice carried a quiet gentleness, as though to ease the weight from his words.
“The dinner was rich. And on Saturday at first light, Giuliano came with two brothers and cleared the rooms. Everything stands as it did before. There is nothing to thank for.”
Deo, at the desk, set down his quill. He did not rise, only turned slightly, his voice calm, even polished — the way one speaks when choosing each word as if it might be weighed.
“Abate,” he said, “the generous dinner was thoughtful. Yet since that evening I have carried one question. Madonna Donata Maffei must have many other halls — places where her gift might be announced, more fitting to the weight of her wealth. Why here?”
Lorenzo’s pause was no longer than a breath, yet it felt like the weight of a bell between strikes. His hands folded over one another, the gesture of a man who holds both courtesy and caution in equal measure.
“Madonna Donata’s gift is not only marble and coin. It is also her presence. And presence, when joined to generosity, can weigh heavier than she intends. I wished only to acknowledge that, and to thank you for receiving it with patience. Such offerings belong to God’s house, yet sometimes they pass through human hands less gently than they should.”
Dania’s voice answered before the silence could deepen.
“Then let us count it as a gift twice over — once for the marble, once for the lesson in patience. Both may serve their place.”
Her tone was warm enough to close the matter without inviting more. Lorenzo bowed again, more lightly this time.
“Then I leave you to your work.”
When Lorenzo’s step faded from the cloister stair, the silence he left behind was heavier than his words. Deo’s quill had not returned to the page. His eyes lingered on the cleared table, as if the shadow of Donata’s presence still sat there.
Dania closed the jar she had been sealing, her hands steady on the cork.
“You still wonder why she came here?”
“Donata sent her maids for your work every day — tea, oils, remedies — but never comes herself. Why plan a whole evening just to sit in your company?”
Dania’s smile flickered, then went out.
“She didn’t come for my company. She came to remind me this room, this work, all of it — belongs to the monastery. And the monastery’s purse carries her seal.”
Deo frowned. “But she asks for what you make.”
“She asks, yes. And she shapes the asking so I remember it is an honor to be of use to her. She takes without returning — because return would make us even.
“What she wanted, she didn’t get,” Dania said. “So she wrapped her message in a dinner. And she must have asked Lorenzo to send the winter’s supplement at the same time. Pietro knows nothing of this place. Giuliano brings the supplies each week. The meaning was clear enough: see who feeds you, see who sits with you, remember whose seal these walls stand under.”
Her words landed like a seal pressed into wax — final, unyielding.
Deo’s hand clenched on the desk edge, knuckles white. He had not moved, but something in him had — the kind of inward tremor that never shows in the face, only in the breath. I saw it before he did. The air around him grew thin, as if the body he wore was already loosening. His shoulders tightened, then softened; the line of his throat began to blur.
I leapt from the sill, landing on the desk with a thud that startled the inkpot. My tail brushed against his hand, steadying it, though my eyes were sharp on him.
Easy, I purred. Not every chain laid upon you must be worn.
And still, in that unrest, Deo’s body shifted — silver hair falling softer, cheeks pale and rose-touched, the strong lines bending into a woman’s grace.
“She is also a woman. Why carry their rules for them?” Deo’s voice was not the one that had spoken before. It lightened, waterlike — not clipped, not ironbound, but flowing, unmistakably feminine.
The room seemed to still at the sound, as Dania held its becoming until safe, so the jars, the books, even the fire knew a threshold had been crossed. Then she looked up, a small edge in her smile.
“Because Donata wasn’t speaking of men ruling women. She placed herself in the closest chair to power, so that when all kneel before it, they kneel before her as well — whether power wears a crown, a ring, or a robe. Patriarch, matriarch — it makes no difference. The seat is the same.”
Deonardo didn’t answer. The thought lay unsettled, not for lack of grasp but for the manner of Dania’s words — sharp, steady, without a trace of heat. Dania’s detachment struck something raw in Deonardo. Now sitting in woman’s body, Deonardo had bound loyalty to Aurelia’s rule — a woman who governed not by cutting but by sheltering, who steadied the land with patience as much as with law, whose power was weight and warmth both. To her, power was never so cold as Dania had named it.
Her tone steadied, though her eyes did not. Deo asked Dania:
“But power is held for reasons. I agree it is misused by tyrants — often, cruelly. But I have seen it in right hands, too. Power that keeps order, that shields the weak and the poor. If that is not power… then what name do I give it?”
Dania’s voice was low but without heat, as if she were naming a fact too old to argue.
“Power can take many forms — terror, law, patience. But beneath them all is the same fate men have lived ever: creation is built in chains of higher and lower. Thrones above altars, lords above villages, masters above servants. The world, even God’s world, was set into hierarchies, and power is created to guard it.”
Deo’s fingers tightened against the arm of the chair, as though he needed the grain of the wood to hold him.
“Do you mean power exists only to guard the chain?”
Dania’s eyes flicked up, steady, her mouth curling just enough to blur between jest and judgment.
“I ask the question,” she said, almost lightly, “if the chain is God’s will?”
“I will not name it as chains,” Deo said, lower now. “You mix the thing itself with the frame built to hold it. Power is the current. The chains — if we must call them so — were meant to be fixed along the mountain path, to steady the traveler. If that is God’s will.”
“We don’t need to call them chains,” Dania returned, her tone even. “Let us call them the guardian and what is to be guarded. And I ask — if the taste of power is so sweet — does the guardian not drift to the center of the altar, until what should be guarded stands only behind him? Not by chance, but by the nature of men themselves? The nature was created by God?”
I pressed into the woman Deo’s side, head nudging against her palm the way cats do when we mean more than comfort. She held herself calm, but under her skin I felt it — the tide shifting, the pull of something searching for ground, a current restless for an anchor.
For a moment she breathed as though she might steady it. And then — the man returned. Deonardo’s voice firmed, the weight of his shoulders settled, and silver once again framed him where moments ago it had wavered.
“I will not allow a guardian to take over the altar,” he said, his voice steady again.
“I see.” Dania’s smile flickered, like moonlight breaking through cloud. “Cavaliere, you did not come here only to study herbs. You came searching for what might guard your altar. You carry the duty of a guardian — your hands hold power as if it were meant to protect what is most precious to you. Am I right?”
The air stilled. No longer talk.
A moment — and Deonardo was back in the dusk, when Dania’s smile to Martino had shifted into the hawk’s gaze that cut the square in gold, disarming a tempered swordsman. The precision that froze motion — it lived in Dania’s voice now, and it found him again. Even I, with my demon’s old wit, could not tell whether the conversation drifted here by chance or whether Dania led it here on purpose.
I curled tighter against Deonardo’s side. I could feel it — how the same tide rose in him: half-ache, half-anchor.
Time passed. The fire cracked, but neither of them moved.
When Deonardo spoke again, his voice was formal, spine drawn into integrity, as if the weight of the chamber itself demanded it.
“Deonardo di Vallescura. Podestà of Porta d’Argento. Sworn in service to Lady Aurelia of Vercado.”

Cacifer’s soft note, for the reader’s lap:

A podestà is a hired power. The word comes from potestas — authority. In the cities they choose an outsider for a set term, a trained jurist with no local cousins to bribe or avenge. He sits the bench, holds the keys of the militia and the market scales, keeps the city seal, signs the contracts, pays the soldiers. The lord or lady sets the line; the podestà makes the line hold.
Deonardo was that hand in Porta d’Argento. Aurelia kept the hearth — law, sanction, the right to say yes or no. He carried the daytime weight: courts, ledgers, grain, patrols, pay chests, treaties. People met her justice through his voice, his signature, his arrival at the gate with riders behind him. That is why I call him half-ruler. The power was hers, but it moved through him; he made it visible, punctual, exact — the edge of her will.





Deonardo the podestà 🕯♛♡🐾  afternoon 
Dania rose without hurry, her tread so soft even I did not hear it until her shadow fell across his desk. Deo had not moved since the last word—shoulders steady, fingers resting on the page as if parchment could anchor a storm.
She leaned in. Not so far as to crowd him, but close enough that her hair slipped forward, black strands almost brushing his shoulder. Her hand reached, not for him, but for the parchment—turning a page with the same calm she used for jars and herbs. Her eyes moved, not steady reading, not idle glancing either. Half-random, half-concerned, as though she measured the weight of the silence as much as the words.
And then, her voice—quiet, so close it carried more breath than sound.
“But, my lord… what you are searching for isn’t bound in these vellum leaves.”
The words landed with the warmth of her nearness. The desk between them did not hold the podestà’s collapse.
I kneaded my claws once into the cushion. Ah, healer—you know how to turn not only herbs, not only knives, but even a man’s silence in your hand. She weighed him then, as if parchment and power were no different than roots and resin. And Deo, though he said nothing, let her stand so near. That was already more confession than his mouth had given.
Her gaze dipped to the pages, fingers grazing their margins as though she could feel the ink through the skin.
“These are not herbs you’re after,” she said, low. “You want what’s buried deeper—charters, bulls, exemptions. The kind of parchment that could make your capital untouchable, even for a Monteferrano toll captain.”
She did not pause.
“You serve a lady whose land lies between ambition and appetite. You won’t have her dragged into the new alliance against Monteferrano—because once the banners rise, there’s no lowering them. You’d rather turn the Church’s sleeve inside out, so that to press her is to quarrel with Rome itself.”
Her words stayed steady, almost clinical.
“If you find the right document here—the right signature from the right century—you give her a shield no tariff can pierce. And you’ll carry it home before anyone knows the shield was ever missing from this wall.”
Her breath folded into the soft-edged daylight between them. Deonardo did not move when she leaned closer, her hair brushing the shadow of his dress—dyed in the color of the city he governs. Porta d’Argento is no mere castle; it is the hinge of Lady Aurelia’s land. Whoever holds it keeps the trade flowing from valley to sea. In his ink-stained hands lies not only law, but coin, armies, alliances. He weighs tolls, signs charters, seats councils. A podestà is chosen for his distance, sworn exile from his own city, bound to bear the full weight of another’s house so his loyalty bends toward duty, never kin.
Porta d’Argento Violet.
Not the bishop’s purple, not a cardinal’s red, but that rare, shifting dye only Vercado’s vats know how to hold: woad beneath, kermes above, iron’s breath cooling it to plum. In shadow it deepened toward blackberry; in lamplight it slipped nearer to wine. Deo wore it as if it were his own skin—overdyed, layered, tempered, like the man himself.
The silver trim caught the light now and then, just a thread at cuff and throat—enough to remind anyone watching: this was not mere cloth, it was statement. Valleys might never see such a color, but courts, charters, and seals knew it well. So when Dania’s black hair fell near, it brushed not only the man, but the office he carried.
“It is a clever way to reach a solution,” she said, gaze steady. “You’ve weighed every risk, every urgency, seeking to keep Vercado’s households untouched without bleeding. But you haven’t found anything yet—nothing close to what you can use. And being away from your office only sharpens your ear, every scrap of news mapping the tensions from Monteferrano.”
He watched her for a long breath, the pale light of afternoon sliding across the shelves, then glancing silver at his temples.
“What do you want from me, healer?” he asked.
The words came lighter than they felt—as if only curious, not measuring every syllable for trap or tether. It was no scorn; more like a man asking the river if it truly meant to take him under.
“Because,” Dania drew back a little, her voice slackening to the last note of a song, “you often have the look of someone holding secrets alone for too long.”
From the cloister outside, the bells of None began to sound—slow, full strokes that rolled through the stone, carrying her words deeper into the hush.
Deo didn’t move at first; then his eyes shifted, tracing the light, returning to the lost pieces between memories—giving up the search for proofs of trust, tasting instead the ease of letting them fall.
When he spoke, it was to the air between them—not quite to her, not quite to himself.
“I don’t… belong to one face,” he said slowly, testing the shape of each word. “Or one age. Or one body.”
The bells faded, but the air stayed full.
A pause. The corner of his mouth twitched—not a smile, but the reflex of a man stepping past the first guard of a gate he had kept locked.
“My bones have worn more years than I can remember. Literally. What I am… changes. Without asking me. Without warning.”
His voice stretched thinner, not in volume but in the space between syllables.
“I have been a man who woke as a woman. A woman who closed her eyes and opened them in another frame, another height, another age of her own hands. And every time… I am meant to act as though nothing has shifted—as though this body were simply the next season of the same life. Even when the world around me no longer recognizes who I am.”
Then at last he looked at her—directly, unshielded.

📖 Shapeshifters🕯♛♡🐾 
Shapeshifters, I have seen them before. Not only in this cloister, not only in this age. Across centuries and fallen empires I have padded through their shadows. Men who woke in women’s skin. Women who rose with another voice in their throat. Children who grew to thirty overnight, their cradles still warm. They are not demons, though priests have named them so. Not miracles, though kings have tried to own them. They are tides trapped in flesh — pulled and undone by moons no calendar can reckon.
I have watched them lose loves, because a lover married one shape and could not follow the next. I have watched them write their names each morning to keep hold of the self that once lived there. They are not cursed; their hurt is only that they remember. And when memory slips, the loneliness weighs more than any chain.
Deo spoke then, not because trust had made an opening, but because he had been too long his only listener.
“When it comes, the change is not mine to choose. My body veils itself in another shape, another age, another frame. It comes without asking me. And yet it never feels like a stranger. However strange the reflection, it is still mine.”
Dania sat back under the wintergreen leaves; glass jars held little fossils of light. The window laid a square of day across her knees. The air drew close, not to choke, but to wrap.
“I don’t remember my first winter, or my first love, or my first loss,” he said. “The oldest thing I keep is always the same: a beginning of winter, thirty-seven years ago. Before that — no root, no count, whether one century or twenty. I’ve never met another like me. If they exist, they have not crossed my road.”
I know the older story. Their bones carry what countless before them carried: the wonder of becoming; the sorrow of never belonging. There are no rules. No horn before the tide, no omen written in sky. They do not command the Shift; they endure it. Sometimes decades between. Sometimes days. Yet one thread holds: when death leans close, the body turns itself inside out. The blade that should open the vein; the fever that should drown the lungs — folded and refused. A new flesh rises, untouched. Always in the height of its prime, intelligence honed by survival. And stranger still, the shape answers the world: a knight where men rule; a disarming beauty where courts bend at a face. The Shift selects not comfort but advantage — privilege, survival. Miracle, perhaps. Cruelty, I say.
“I hide journals and later find them,” he said. “But the hand is different after every Shift. How do I trust? As the old years fade, how do I know no one else placed them to trick me? I keep at most thirty-seven years. Beyond that, the rope frays. Some men of eighty can tell you what the sun smelled like when they were five. I will never be one of them.”
“There is a night sky I dread,” he went on. “When the constellations fall into a certain place, the goodbye begins. Once a year. My eyes go there on their own. The last memory on the far edge simply loosens — sometimes a season, sometimes a face, sometimes a room of people who once knew me as theirs. It doesn’t matter how precious. It goes.”
Ah. Thirty-seven. Each shifter carries a measure. Twenty. Eighty. The longest I ever saw held one hundred seventy-one. Always there is an edge, and beyond it, forgetting. Not at the grave — death leaves the face clear for many winters yet — but when the limit comes: a word one year, a gesture the next, the sound of laughter after that, the weight of a hand, until even grief loses its shape. Then the world becomes a place where nobody recognizes them. So I pressed my paw to his sleeve and purred into the tremor.
“There’s no clean pattern,” he said. “Even when the frame stays, another side drifts in — between man and woman, not completed, just in between. It lingers, then folds away. I’ve tried to find the root — what stays, what changes, whether the will can command it — but memory cuts the string before I can follow it back. I don’t know who I was the first time, or why.”
“The hardest part,” he said at last, “is that what takes me also keeps me. I cannot say, ‘This is not me.’ Even when it comes like a storm, it feels as if it had been waiting all along.”
He lifted his eyes to her then — not asking comfort, only testing whether she still saw the same person who had walked in. I can never read Dania’s mind, but I felt her listening with my fur. She had already witnessed his shifts: the woman in him arriving like a dream — breath, gesture, turn of face — then folding away as quietly as sleep.
“Sometimes sadness arrives with no messenger,” his voice blurred between his man and woman. “It finds me in a body I barely know, pours itself a cup into hands that were larger yesterday, smaller the year before. It sits with me anyway, as if it recognizes what I no longer carry whole — memory breaking like old glass, faces slipping through the cracks. No bargain, no lesson. Only the ache of being both older and newer than myself.”
His silver hair glowed faintly against the Porta d’Argento violet, the dye drifting from blackberry to wine as the light thinned. I climbed onto his knees and settled. He did not move me. My purr rose and covered what words could not.


📖 Aurelia🕯♛♡🐾  Vesper
“I was twenty-eight the first time I saw her,” he said. “I was known as twenty-eight among others. By then, I could never tell how many lives I’d already worn through.”
He spoke her name with care, as though saying it wrong might bruise the air.
“Donna Aurelia di Vercado. Signora di Porta d’Argento.”
A pause—almost to himself:
“Aurelia — golden. Not for her hair or her dress, but because she reflected back the best in whoever stood before her. Made you believe you were worth the gold in your own name.”
He leaned back slightly, eyes not on Dania but somewhere past the walls.
“We met while serving her father,” he said. “I’d been in his court a year, maybe two — as a diplomat. My work was to carry his words into rooms where his own voice would not carry far. In Vercado, that meant more than speeches and treaties. Every choice — trade, law, harvest, war — touched someone beyond our borders. And every neighbor was watching for advantage.”
He let his palm rest on the table, as if weighing it.
“She was the heir — seventeen then. A little shy, easy to overlook if you didn’t look twice. Quiet. Quick to yield space. The kind of presence you could mistake for uncertainty.”
His gaze flicked upward, as if seeing again the corridors of Porta d’Argento.
“When her father died — sudden, without warning — she was the only one left to wear the crown. And in a place like Vercado, surrounded by lords with louder voices and sharper appetites, a new ruler doesn’t survive by waiting. Every road out leads to another lord’s table, and if you can’t sit at all of them, you’ll be shut out of your own gates.”
His voice eased, almost warmed.
“She called me closer. Not with orders — Aurelia never needed to shout — but with questions. Where to build trust. Where to bargain. Where to bend. And because Vercado’s lifeblood runs through its relationships, every answer I gave pulled me further into the center of her rule. It was supposed to be service. It became something else. Not love in the ballad sense — but a loyalty that lived in the ribs. A belonging to the land we both guarded.”
He breathed out slowly, and when he spoke again, his tone had dropped.
“I have the passing shifts you’ve seen — those come and go in days or hours. But the deeper ones… the kind where my whole body reforms, stays in that shape for years — changing gender, age, everything in my bones. Those shifts have often forced me to leave whole lives I’d built. Made me a stranger to those who once received me as part of their houses. And yet beside her, I remained in one form for eighteen years. The longest stillness I’ve ever known. I don’t know why.”
The afternoon stretched with his words. Light climbed the shelves one rung at a time, dust softening in its wake.
“My memory still thinned, the same as always. Nothing beyond thirty-seven years survives. Always the same length. Always gone without mercy.”
By the time his breath slowed, the sun had shifted low — the pale edges of day sharpening toward gold.
“The stillness with her made me feel more human. More real. More at home. She tolerated the passing shifts — never turned distant for it. Instead, she gave me cover from other eyes. Only said: ‘My court is yours while you are mine.’ And she kept it so. No one in Porta d’Argento dared call it unnatural, not while she lived.”
I noticed the way his shoulders had eased, as if in the telling he’d stepped back into her presence. He had not said love. But every line of him was speaking it.
It was not yet the fire-colored blaze of evening, not yet night. Only that hour when every surface glows, as if remembering the warmth before it leaves. Aurelia’s name seemed to live in that glow, carried in his mouth, carried in the air — golden, not for her hair, but because she made even memory gleam.
The shelf lit last, the line of books catching flame as the sun bent west. Then, before it turned to full sunset, his voice fell quiet. The silence he left behind was the same as the light — stretched thin, waiting to fade.
Dania didn’t speak. Her eyes stayed on him, not demanding, not softening, but present — the kind of presence that doesn’t try to steer a confession, only to hold it where it can keep breathing. I’ve seen that look before. Romé crumpled under it the first time they met. Most people flinch from their own unguarded selves; she keeps the door open without pushing you through it.
The sunset finally climbed full onto the shelves, spilling its weight across the jars and leaves. It gathered in the violet of his dress, then deepened until it struck his hair — silver turning to gold, as if the day itself had crowned him for speaking. The story of Aurelia had drawn something out of him that even Aurelia herself never heard.
“You are looking for something you could use,” Dania said at last. Her words carried no sharpness, only the quiet weight of recognition. “To guard her. To guard her land. Since both became your home.”
Not a question — a bridge laid without ceremony from his past into the stone-cold present. Her eyes rested on him as the golden glow settled over his hair, as if even the light had heard the name — Aurelia — and chose to shield him still.
The air between them thinned — the kind that comes before a storm. She waited.
“It’s not a book anyone will admit exists,” he said at last. “Not a chronicle, not a register. Something older. A charter, a bull — a scrap of parchment that proves what I need it to prove.”
His eyes lifted to hers.
“If I leave without it, we will have to choose a banner. And Vercado cannot fight against all banners at once. The city has endured for centuries by its crafts, its refinements, its ability to adapt — not by military strength. But no banner wins forever. Choose wrong, and we risk the land in fire and blood.”
The bell rang Vespers. The first strike was not the bright peal that calls monks to prayer, but the deep one — the iron-throated bell from the tower’s middle tier. Its note rolled into the room like a slow tide, shaking the air before it faded. A pause. Then a second bell — higher, sharper — answering from the smaller tower by the refectory. Then the first again. Back and forth they went, pressing into the room like a measured heartbeat.
Something not yet spoken, but coming. Something Deo would have to walk toward.
When the last note stilled, daylight was gone with it. The silence felt carved into the dim, deliberate space. Dania rose, her feet moving almost without sound. She touched flame to the lamps, one by one, until the shelves and jars breathed gold again.
“In your capital,” she asked then, her voice quiet under the new light, “are there legends? The kind of miracle that clings to ruins — an object, a sign, a phenomenon no one can quite explain?”
Deo looked at her as her face and outline came into form in the lamplight — the soft flare catching her cheek, the black fall of her hair, the steadiness that seemed to draw the room taller around her. For a moment he forgot the question she had asked.

“There’s a chapel,” he said at last, “or what’s left of it — half-swallowed by the olive terraces. No roof, only one wall standing. They say that wall stayed upright through the quake that leveled the rest of the valley. Some claim the mortar was mixed with water drawn on Saint Lucia’s feast day. Others… that a light is seen there on winter nights, though no lantern’s ever been found.”
The light did not merely touch her; it made a volume in space, a place where she could prepare the birth of miracle. She turned toward him, voice low, like a myth unfolding.
“If there were an old letter… a record… something grave enough to satisfy even bishops — naming that place as a site of divine favor?”
Deo’s breath caught — not sharp, but steady, the way a man leans closer to a fire he does not yet trust.
“How could there be such a thing?” he murmured.
“There must be one,” she said, and in the half-shadow her eyes almost smiled. “As long as somebody once placed it there.”
She leaned a fraction closer, standing between the lamp and Deonardo. He lay fully in her gaze, softened by lamplight — or already slipping toward his feminine side — while Dania remained in shadow.
A witch.
“No matter who or when. As long as it’s there… as long as it’s found… as long as it’s copied under the monastery’s seal, and the news flies out—”
Her hand traced an invisible figure on the wood between them.
“You know the custom here. When a great donation is given, the library opens its deeper section. Certain parchments are brought out, read aloud in the sponsor’s honor. Their name carved into marble beneath the cross, joined to the title of the divine work revealed. And from then on, it is no longer theirs alone. It becomes part of the monastery’s living memory.”
She let the pause do its work before continuing.
“The most recent sponsor? Donna Donata Maffei. Imagine — when the library opens to her gift, a Christian miracle is drawn into sunlight under her name. Something she can carry back to her salons for all the valley to admire.”
Her smile was faint, almost indulgent.
“She would have a copy made with the monastery’s seal. She would speak of it at every table. And in that noise, the parchment would travel — across Valdiponte, through markets where coins and gossip weigh the same, on to Bologna, Pisa, Lucca… until it reached a bishop who found it convenient for his own cause.”
She let him see the next step without saying it: such a parchment, once loosed, could shield a valley or expose it, depending on whose hand set it moving first.
“You must know someone — not bound to your office, but light enough to move in Donata’s circles. Someone who can draw the strings so each tug looks like her own will.”
Her eyes sharpened.
“A woman like Donata cannot resist such a path. Give her a ruin tied to a miracle, a restoration to sponsor, a name carved in stone — she will take it, not for faith, but for reach. She’ll want to walk the ruin herself, perhaps even bring Maffei marble for its repair, to set her name there forever. And then — reluctantly, perhaps — your lady will be pressed to honor her with a visit in her palace.”
Dania did not soften the edge.
“And when the invitation comes, Donata will see more than piety. She will see her chance to make your lady step through her gates — a victory to parade. The wound of being overruled at Friday’s dinner repaid in gold leaf. She will work for it harder than you ever could. And she will make such noise that half of Italy hears it — her ambition to be known ringing louder than any bell.”
Her tone stayed conversational, but the words landed like stones in a foundation.
“By then, everything would be as solid as carved marble. A history no one could question.”
She leaned back, gaze never leaving him.
“And of course, Deonardo… you would know how to use it. To keep Vercado safe, especially now that Monteferrano’s new lord leans toward Florence — and the Guelfs.”
I saw his breath catch — not assent, not denial. She had set a weapon before him; already he was weighing its balance.
I have learned: the dangerous ones are not those who draw a blade at once. It’s the ones who stand very still and imagine exactly where it might fit between the ribs.
For a heartbeat, he didn’t move. Not even the small, habitual gestures that buy him time when the room changes shape. Then — softer than I expected:
“Why are you helping me?”
Two registers braided in the question: the lower, steady — the man’s; the higher, almost a breath — the woman rising through the syllables before he caught her back. He waited for the price hidden under the words. None came.
Dania leaned back, her gaze blurring into an invisible smile, like a ship slipping under the horizon.
“Because I can.”
No weight of debt. No hook for bargain. Only the silence after a stone drops into deep water.
That was when I felt it — the swing in him. Eyes still on her, and in the space between their breaths, the questions knotted:
Why does she see this much of me? How long has she been reading me as part of her map? If I trust her, where will she take me? If I don’t, can I still win?
It was the same unease I’ve seen in him with bishops and envoys — but braided now with something warmer, stranger: the reluctant awe of watching someone pull the thread of your problem and find the other end without you naming it. Aurelia’s shadow moved behind his eyes — not because Dania resembled her, but because both women had stood before him and said, each in her way: I see you, and I will work with you as you are. Yet they were nothing alike. Aurelia held him still; Dania moved the ground. For Deo, that aliveness carried its own fear. This plan could not be half-played. Once begun, it would run until it struck stone or blood. And movement — real movement — has always risked waking the change in him.
Dania didn’t look away.
“You’re waiting for me to convince you that you can trust me,” she said, calm, almost indifferent. “But I’m not going to convince you. You might convince yourself. Or not.”

📖 November 18, 1264 — Tuesday
Frost, dark night before new moon
Bridles at Prime 🕯♛♡🐾 
The frost still held on the stable lintel when Luca lifted Argento’s bridle from its peg. He worked quietly, methodically — the way a court servant is taught: smooth the leather with his thumb, check the stitching, run the bit across his sleeve to catch any grit before it touched the horse’s mouth.
Beside him, Giuliano was half-wrestling Fiamma. The mare tossed her head, teeth clacking against the bit before he’d even raised it high enough.
“Saints preserve me,” he muttered, trying not to lose the strap. “You’d think she was fire herself, not just named for it.”
Nero, already brushed and blanketed, watched with slow blinks, as if amused by the boy’s struggle.
Luca glanced up once but didn’t rush. He slid the bit into Argento’s mouth, adjusted the cheekpiece so it sat just right, then spoke without looking over.
“She’ll test you if she knows she can win.”
Giuliano puffed out a breath, trying again. Fiamma stamped, shoulder shoving him sideways.
“And if she already knows she can?”
“Then you breathe slower,” Luca said simply. He tightened the noseband one notch, tested it with two fingers. “And make her believe you’ve got all day.”
At last, Giuliano coaxed the bridle on, though his cheeks were flushed with the effort. He smoothed Fiamma’s mane — as much to calm himself as her — and shot Luca a sidelong look.
“Tell me… in Porta d’Argento, do the stables have as many horses as the houses here have hens?”
Luca bent to check Argento’s hooves, lifting each in turn and scraping out the packed frost with a small iron pick. His words were measured, like his work.
“Not hens. But yes — a good number. Merchants need them for their carts and litters. And the guards keep their own.” He set the hoof down gently, straightening. “But none like this one. Argento came from Lady Aurelia’s own stock.”
Giuliano’s eyes widened, but he bent back to his brushing, now working the sweat crust from Fiamma’s flank. She flicked her tail sharply across his shoulder. He swatted it away, grinning despite himself.
“Then he must think himself noble, with a name like Silver.”
“Sometimes he does,” Luca allowed. He pulled the girth strap through, tightening it firm but not harsh. “And sometimes he rolls in mud like any mule.”
Giuliano chuckled, dragging the saddle across Fiamma’s back — too far forward. She crowhopped once, nearly unseating it. Luca’s hand was there at once, steadying the leather before it slid.
“Here. Always an inch behind the withers, or she’ll be sore by nightfall.”
Giuliano exhaled through his teeth, tugging the girth strap with both hands. “She doesn’t make it easy to remember.”
“Then she’s teaching you,” Luca said simply.
For a moment, the only sounds were leather straps buckling, hooves shifting, breath steaming in the chill. Then Giuliano, quieter:
“And you? You’ll ride with him always? Wherever your master goes?”
Luca smoothed the last fold of the stirrup leather, eyes on the buckle.
“That is my work.”
And though the words were plain, Giuliano heard the weight beneath them.
The yard was already stirring with the smell of straw and iron when the three riders came.
Romé’s boots struck the flagstones first, light yet certain. His black wool cloak hung loose at the shoulders, drawn back to show the long fall of his tunic — silk the color of deep pomegranate, its weave too fine for Vallefredda’s looms.
Dania followed in her plain riding dress, black hair caught back simply, her stallion Nero waiting as steady as a stone under frost.
Deo came last, silver in his Aurelia Blue surcoat flashing under his cloak, Argento gleaming as he caught the sight of his rider.
Giuliano was bent over Fiamma, still tightening the girth when she lunged forward a step, nearly knocking him against the stall rail.
“By the saints—!” he hissed, fumbling to steady the strap.
The mare tossed her head, mane catching the sun like flame indeed. Romé’s mouth curved as he crossed the yard, hand already lifting. With one motion, easy as breath, he set the saddle firm, his palm flat against her neck. Fiamma stilled under him — not tamed, but quiet, as if she’d only been waiting for his touch.
Giuliano’s cheeks flushed. He stepped back, brushing straw from his sleeve, muttering an apology. Romé only grinned wider, taking the reins as though the whole wrestle had been nothing but foreplay for the ride.
Meanwhile, Luca was finishing Argento’s bridle. He gave the last buckle an efficient tug, then stood aside as Deo approached — reins laid neatly into his master’s hand without a word. The white stallion’s ears flicked forward, patient, as if bred never to test the man who held him.
Dania glanced once between the two scenes — Romé’s easy dominion over fire, Deo’s silver horse already made ready by calm precision — and in the middle, Giuliano, red-faced but earnest, still catching his breath. Her mouth softened, almost a smile, but she said nothing. She only mounted, gathering her reins, and let the balance between them speak for itself.


The Road to Valiponte 🕯♛♡🐾 after Prime

Tuesday again.
After the healer prayed her hour in the Chapel Aromatai, the three of them were ready to ride. Dania to her weekly path toward Valiponte, the Basili house, where the sick waited in quiet rooms. Romé perhaps to wander, to taste food and wine. Deonardo — still turning over the words Dania had set in him the day before.
I slipped past Giuliano’s boots, tail brushing his robe, and greeted Fiamma with a blink. She rolled her eyes at me, tossing her head: Not you too, little one. But she breathed my scent, and I knew she remembered. Fire knows fire, even in fur.
Nero gave me no sign. He never does. For him, silence is enough.
Then Argento. Ah, yes — silver eyes, silver tack, silver calm. I leapt to his stirrup and climbed the saddle as though it were stitched for me. He flicked an ear but held steady, proud as the man he waits for. I pressed my paws into the fine leather, kneading. A true silver throne, I thought. But warm enough for a cat.
The road wound pale through the pines, snow pressed into ridges by cart wheels long gone. Three horses shared it as three souls do — never the same, yet bound by rhythm.
Nero, black and broad, carried the healer with the same patience she gave her patients. No hurry, no flare, but each step steady enough to trust your life on. Dania’s body fit him like a second spine, calm as though the road itself were her workbench.
Fiamma leapt hot against the bit, flame in chestnut hide. Romé let her test the snowbanks, tug her head, carve her own pace — he laughed under his breath when she tried to bite the frost. His seat was loose, alive, more a dance than a command. They were a pair made to dare the edge.
Argento gleamed between them, high and white, silver tack glinting. Every stride measured, as if he counted the ground before he took it. Deonardo’s posture matched: composed, contained, his cloak falling in the same disciplined line. He didn’t let the horse carry him — he carried the horse, quiet as law written in flesh.
Three horses. Three riders. And the snow, holding their different weights in one line of prints, as if the valley itself kept their company.
The snow thinned as the road bent toward the town. Valiponte’s wall rose plain and gray, not grand, but serviceable — a belt of stone no higher than a man’s toss. Smoke from hearths curled over it, drifting into the valley air.
The gate loomed ahead — thick oak planks banded in iron, standing open now. A watchman leaned out from the tower window, cloak wrapped tight, spear butt thudding once on the boards in recognition. He gave Dania a short nod — the healer was known.
The horses felt it first — the shift from open road to the funnel of stone. Nero’s pace never changed, steady as the woman on his back. Argento lifted his head, ears pricked, as if measuring the echo of his hooves on packed earth. Fiamma snorted, tossing her mane against the close walls, impatient until Romé’s hand calmed her.
The arch of Valiponte’s gate cast a long shadow. The mule ahead of them balked, hooves slipping on the thaw-slick stones, while the guard with the spear shifted his weight and waved it through with no more than a grunt.
On the bench under the arch, a scribe bent over his wax tablet. His stylus scratched quickly when Argento passed — silver tack, violet cloth, the bearing of a rider who was not valley-born. Not all names were asked, but some presences were worth noting.
Romé’s cloak drew a glance too, foreign-cut, brighter than the wool of the valleys. The stylus paused, then pressed again.
But just behind, a knot of men in patched cloaks shuffled under the arch. Pilgrims, or so they looked — scallop shells stitched crooked to their hats, staffs worn at the tips. None were stopped. None were written. They moved past with their heads low, muttering prayers in no language anyone bothered to test.
Romé’s eyes narrowed, catching the contrast. Silver tack was noted. A pilgrim’s face was not.
The bowman under the arch yawned, string still unstrung across his knees, and let them all file through.
They passed beneath the arch and the smell struck first — not incense or hearth-smoke, but iron and hide. The road inside ran narrow, the houses shouldering close, workshops open to the street.
To their right, a blacksmith’s hammer beat steady, each blow ringing sharp against the frost. Sparks leapt from the mouth of the forge, orange against the pale day. A boy pumped the bellows, hair plastered to his forehead with sweat despite the cold.
Across the way, tanners had set hides to stretch on wooden frames, scraping with blunt knives while children sloshed buckets of water into the gutter. The stench of lime and flesh clung so thick that Fiamma tossed her head, ears flicking. Romé steadied her with a hand, his jaw set against the reek.
Nero went steady as a townman walking around home, his hooves slow and deliberate through the mucked cobbles. The stench of lime made my whiskers twitch, smoke clung sharp in my fur, and every hammer blow thudded straight through my paws on Argento’s saddle.
He held himself calm, silver-steady under me, but I felt the flick of his skin at each hiss of the forge, as if even marble-mannered horses disliked the street’s bite.
Further along, the air shifted: wood-shavings now, resin, the soft rasp of planes. Carpenters worked with benches pulled almost into the street, shutters stacked, planks lined like ribs against the walls. One greeted Dania with a nod, wiping sawdust from his beard.
The whole street moved like a living thing — hammer-beats, saws, the cry of a hawker with baskets of nails and rope. Men and women glanced up briefly at the mounted trio but went on with their work; horses on the street were no spectacle here, only one more part of the traffic that carried coin through Valiponte.

📖 Dania in The Basili Hall🕯♛♡🐾 late morning

Romé excused himself at the corner, easy as smoke slipping out a door. He didn’t say why — whether to avoid the little merchant daughter who had flushed at him last week, or whether the taverns and shopfronts tugged at his hunger for new taste, new faces. He vanished into the turn of the market street, Fiamma’s mane catching the sun like flame.
Deonardo went on with Dania. Argento’s tack gleamed, his pace controlled, but the man atop him was quieter than usual. Their conversation — especially her final sentence — still tangled heavily in his mind.
The wooden gates opened on the quiet Basili courtyard. A thin thread of smoke rose from the kitchen wing; a mule shifted in the corner stall. The main hall smelled faintly of apples and beeswax.
I slipped from Argento’s saddle and padded across the stones, paws grateful for the hush after the street’s hammering and lime-stink. Apples, beeswax, mule-straw — ah, this was air I could breathe again. My tail flicked once.
Filippa Basili came out alone, in a plain wool dress, dark blue, her sleeves rolled past the wrist. She bowed quickly, nervous in her composure.
“Madonna Dania, Messer Deonardo — welcome. My father sends his regrets. He has been called to council this morning.”
She gestured them in, steadying herself with the role.
Deo couldn’t help but frown when they entered the Basili Hall. Benches ran along the walls, already crowded — shawls wrapped close, caps in hand, children perched on knees. The air was thick with coughs, whispers, and the faint sourness of damp wool. A woman fanned her infant against the heat of the hearth.
Some had been waiting since dawn — old men leaning forward on their sticks, women with swollen joints massaging their fingers, apprentices with bandaged hands cradled in their laps. A lame boy tapped the flagstone nervously with his good foot; a girl beside him traced circles in the frost on the window.
When Dania crossed the threshold, the room shifted — not hushed, but drawn. The sick sat straighter, the mothers nudged their children forward. It was not awe, but recognition: she came every Tuesday, and Tuesday meant a chance.
Filippa moved briskly, guiding Dania toward the cleared bench, already set with cloth, jars, and water. The young lady’s voice carried over the low murmur:
“Madonna will see you each in turn. Please keep your order.”
And Dania?
She moved through it like a current through stone — not hurried, not halting. She gave each one what they needed and no more, as if her mind kept a ledger sharper than any merchant’s. Her voice was calming, her eyes clear.
“Your cough is worse in the mornings?” — and before the man answered, she had already taken note of the smoke that clung to his cloak, the soot around his nails.
“You eat late, with bread only?” — to the woman with pain in her side. Her question cut straight to the stomach as though she had already walked the rhythm of that house’s table.
To another: “You carry water for the tannery. How many buckets a day? And do you drink before or after?”
Each patient seemed held, named, seen — but never overlong. A touch of the hand here, a short pause for a child’s eyes, then she turned, and the Basili stewards bent their quills, quick as if they too might lose the thread if they didn’t write at once. She spoke in a rhythm that let them catch every detail: what illness showed itself, what habit worsened it, what remedy had failed last week, and which might be tried anew.
And still, she remembered. Not just today’s cases, but years.
“You said last winter the fevers took your youngest — has your wife been well since?”
“You told me your pain eases in the spring — has it returned earlier this year?”
It was almost incredible. Forty or more pressed into the hall, yet she carried each one as if they were a single clear note in her mind.
I slipped low under the benches, tail brushing skirts, ears full of the murmur and coughs. To me it was chaos — smoke, noise, the scrape of quills — but to her, I could see, it was order. A pattern she could read the way a monk reads a psalm.
Deonardo stood a little apart. His eyes followed the healer, weighing how such clarity could live in the same mind that, only a day ago, had mapped a bold plan to rescue Aurelia’s land — shaping politics and legacy to give breath again to his dying mission.
Can he trust her? What matters to her? Why would she help him?
That was the thought folding tighter inside him as he watched.
Yet the answer was happening in front of his eyes. She gave no softness. No smile meant to soothe, no warmth performed to make the poor love her more. She did not bend herself into kindness. She asked, she listened, she judged — cleanly, almost coldly — and yet every soul in the room trusted her as if trust were a natural law.
They did not look at her as they might look at a saint’s image, or a lady dispensing alms with a tender word. They looked at her the way a farmer looks at the sky: indifferent to affection, but certain it governs life and death.
No one in that hall seemed to ask who she was. They did not seek her heart, her story, her nature. They sought her because she worked. Because she answered. Because she did not need to be believed in — only to be obeyed, as if God Himself had lent her hands to hold their sickness.
And Deonardo, who had been moving through titles and identities — diplomat, judge, podestà — felt the strangeness cut deeper, almost frightening: they did not need to know her as a person, yet they trusted her more than they would ever trust themselves.
I crept higher onto the bench near his knees, tail flicking once against his sleeve. Ah, Deo — you are used to men following the weight of titles, colors, seals. But here, they follow her without any of those. You wonder if it is faith. I tell you it is something sharper: need. And she does not flinch from being needed.
I twitched my whiskers. Humans call it power, cats know it differently. Such a talent, when kept and used, becomes either a gift… or a wound.
Filippa crossed the hall quietly once the next patient had seated before Dania. Her hands, still faintly stained with herb dust, folded politely before her as she stopped near Deonardo.
“Messer Deonardo,” she said softly, careful not to disturb the sick, “would you be more at ease elsewhere? This room is not meant for waiting.”
Her tone carried no slight — only the calm practical kindness of someone used to balancing the sick, the healer, and the household at once.
“If you wish, I can bring tea to the side chamber. My father keeps it ready for visitors who come with Madonna Dania. It is quieter there.”
Deonardo inclined his head. “That would be welcome.”
Filippa led him down a narrow passage, the echoes of coughs and murmurs fading behind them. She pushed open a door and stood aside.
The guest room was not large, but it carried weight. Jars of resin and spice lined the shelves, Latin labels neat and deliberate. A map of the Levant spread across the wall near the window, corners anchored with bronze pins. On the side table: an ivory box, its lid carved with vines, and beside it a Turkish beaker, its glass faintly stained with saffron. No gold to boast, no velvet drapery — but a room that spoke of distance crossed, of trade turned into memory.
Deo’s eyes moved slowly, marking the choices. Not wealth for its own sake. Proof. A ledger of where they have been, written in objects.
She set one cup before Deo and held her own with both hands. Her voice grew steadier as she spoke:
“Every Tuesday is the same. We open our hall, Madonna Dania sees the sick, and we provide herbal remedies. The patients — some bring silver, some bring herbs, some bring nothing. No one is turned away. My father says a merchant house has two ledgers: one of profit, one of mercy. A town that forgets the second will not keep the first for long.”
Deonardo let his eyes travel the room once more — jars, maps, relics — before meeting hers.
“The stewards who sit beside her,” he asked, “always writing. What is it they note so carefully?”
Filippa’s answer came at once, as if she had repeated it often:
“They collect what we cannot afford to ignore — symptoms, patterns, the way sickness spreads with the seasons. Fever in the spring, cough in the damp, aching bones when the frosts begin. Later, we compare them and refine our receipts. The aim is not costly imports, but remedies from what grows near: fennel, sage, nettle, mint. Medicines the poor can gather with their own hands.”
A flicker crossed her face, half pride, half weariness.
“It is endless work. But if we fail to listen, illness will cost us more than coin ever could.”
Deo’s fingers stilled on the cup, his gaze holding hers a little longer.
“And does every house in Valiponte keep the two ledgers so faithfully?”
The phrasing was light, almost idle, but the tilt of his head gave the question a sharper cut.
Filippa’s mouth pressed faintly, then eased.
“No,” she said simply. “Most keep only the first. The Maffei, for example… they do not trade in herbs, nor in mercy. Their measure is marble, silk, the gleam of what can be seen. Their gifts to the poor are feasts on holy days — and even those are meant to be seen. But remedies for winter coughs? Notes on fevers among shepherds? That is not their work. They leave such things to us.”
Deo set his cup down lightly, the rim making no sound on the wood.
“And when differences rise between houses — Basili and Maffei, or others — how are they weighed? I mean… the town council. Who speaks? Who decides?”
Filippa straightened a little, as if repeating something she had heard her father explain many times.
“The council meets every month in the loggia. Each of the greater houses sends a master, or sometimes his eldest son. The guilds have their benches as well — the weavers, the smiths, the tanners. The priest sits among them, though his voice is more for conscience than for coin. Together they weigh disputes: tolls, bridges, rents, even quarrels between villages. Decisions are written, sealed, and read aloud in the square, so none can claim ignorance.”
Her eyes flicked down, then back to him.
“But it is not a senate. The strong voices carry further — those with more coin, more clients, more fear behind them. My father says the council is like a scale: fair only so long as every weight is seen. Hide a weight, or lean too hard, and it tilts.”
Deo listened, his expression unchanged, but the words folded into the maps he was already drawing in his head.
Her words paused, and for a moment she looked down into her cup as if reading the dregs. When she spoke again, the steadiness thinned, something sharper breaking through.
“Today was no ordinary gathering. It was called in haste. At dawn, two of our own merchants were found dead on the north road — cut down, their wagons taken. Not strangers, not pilgrims. Townsmen. Neighbors.”
Deo tilted his head slightly, testing her composure.
“And your town — how does it keep itself safe? These walls alone?”
Filippa answered without hesitation, though her voice kept the plain cadence of fact, not pride:
“The gates are manned by townsmen. Each house gives names. We also pay into the purse that keeps the captain and his company — Ferrazzo, you’ve met. The watch turns with the weeks, and on feast days we double it. That is the measure.”
She glanced down at her cup, then added, quieter:
“It works well enough… until it does not. The council argues whenever the purse grows thin.”

📖 Filippa🕯♛♡🐾 afternoon
This day, Deonardo was struck by two women — Dania and Filippa. Dania impressed him every day, but always in a new register, each time a depth he could not have imagined until it surfaced.
But Filippa, the green-eyed Basili daughter, moved him differently. A pretty young lady, born into a house as rich as Basili, she stepped into the rooms of men’s knowledge to navigate it; her plain dark-blue wool dress could not restrain the fluid freshness in her chest; her humble tone did nothing to dim the accuracy and clarity of her answers.
Deonardo, co-governor to Lady Aurelia di Vercado, seasoned in courts thick with speculation, cowardice, and betrayal — weighed Filippa’s stance more seriously than that of any man around her.
Leaning back slightly, Deonardo let his eyes trace the spice jars and faded maps. His voice was gentle, without probing:
“How long has Madonna Dania been coming here?”
Filippa’s fingers tightened briefly on her cup. Her tone, so clear before, softened — not in volume, but in tempo, as if brushing against something older inside her.
“Seven years.”
He waited quietly. And perhaps it was that stillness — the way he didn’t rush her — that let the next words come.
“I was twelve when she first came. I remember because… I’d just returned from a gathering with other girls. I was still dusty from the walk. And when I came through the arch, I saw her — here, in this house, speaking with my father and brothers.”
She gave a small breath of laughter, almost invisible.
“She may not have noticed me. But I remember. The way she stood — not asking to be adored. Just…”
“Just?” Deo’s gaze stayed gentle.
“Just… I had never seen a woman so beautiful as Madonna Dania.”
Filippa’s tone faded into a flush. For a breath, she was the girl she’d once been.
And I — Cacifer — purred beside the Turkish beaker. I saw it flicker in him then: his woman, surfacing just for an instant, also flushed.
“At that time,” Filippa continued, “I used to go every week to a reading gathering at House Maffei, with other girls. They were all older than me… and prettier. We admired Madonna Donata Maffei — her husband was still alive then, and she was the most beautifully dressed woman in Valiponte. No one wore color like her.”
She hesitated, then added more softly:
“I felt a little ashamed around the others. My mother… she has a chronic illness. She’s kind, very kind, but she doesn’t go out much, and she never cared about dresses or jewelry. And I’m the only girl in the family. My father would have given me anything if I asked — silk, ribbons, whatever I wanted. But… I just didn’t feel right dressing myself in those things. I was confused. And I didn’t have anyone to ask.”
Deonardo said nothing, but his stillness was warm. Something in his gaze gave her the courage to go on.
“After I saw Dania… I started to understand what might feel right to me. I watched her — every time she came to our house, or when we saw her in San Michele for High Mass. I kept wondering: how can a woman be so beautiful in a plain black dress? And not try to please anyone?”
Her voice had grown quiet again — not hushed, but inward, like someone thinking aloud.
Filippa glanced down, thumb smoothing the rim of her cup.
“It didn’t happen all at once.”
Her words were slow now, not shy — more like she was circling something she still didn’t quite know how to name.
“But little by little, I stopped going to the Tuesday gatherings with the girls. They were kind — they were never cruel — but their talk stayed with silks and dowries, and I found myself drifting away from it. I didn’t tell anyone. I just… stayed near the stewards, near the ledgers, near the spice crates when they came in.”
She gave a small breath. “At first, I only wanted to understand what Dania was doing. Later, I think… I wanted to stand where she stood. Not beside a husband. Not under anyone’s arm. But… where she stood.”
She looked up then, and her green eyes were clear.
“And I found I loved the work. The counting, the weighing, the tracing of routes. I began to ask to sit with my father when merchants came, and he let me, as he always does.”
Her voice dipped slightly, as if recognizing how unusual that was.
“Now I keep one of the ledgers myself. The second one. The one that doesn’t bring profit.”
A flicker of a smile. “But it keeps other things alive.”
Deonardo’s gaze didn’t waver. He sat between her words and the glow of winter daylight, as if between the weight and soft-edged memory. After a moment, he asked — softly, but not casually:
“Have you ever told her that?”
Filippa looked at him, startled for the first time.
Not offended — simply caught at a gate she hadn’t thought of opening.
“Told… Madonna Dania?”
She shook her head faintly, as if the idea had never quite formed in words before.
“No. I’ve never…”
Filippa looked down briefly, then back at Deonardo.
“The first time I spoke to her… it was by accident. I was fifteen. That Tuesday, my father was at council, my brothers had gone traveling, and my mother wasn’t well enough to come out of bed. So it was just me.
After the servant let her in, I was sure she would ask where everyone was — or wait for someone older. But she didn’t. She just said: ‘Good morning,’ to me, as if I were naturally the one to host the house.
I was shaking. I hadn’t even tied my sleeves right. But somehow… I just knew what to do. I found the right herbs. I set everything up. She asked questions and I answered. Not like a girl — just like myself. For the first time.”
She paused, then added, almost smiling:
“I don’t remember what I said. I only remember how it felt. Like stepping into a boat — and the river already knew the way.
And then… it was the day I became an adult.”
Deonardo didn’t speak at once.
But his mouth curved, gently. It was not his man’s smile.
I saw his feminine beauty lit in that moment — the softness behind the cheekbone, the eyelids lowering with care, the chin unguarded.
He held her gaze, and asked — not for curiosity’s sake, but as one who had once longed to be near someone and not known how:
“And since then… did you become close to her?”
Filippa looked down at her cup, then back at him — not shy, but careful. She had never said this aloud.
“I think I did try… or I hoped to. To be close.”
A small breath.
“But… Madonna Dania is not close to people the way most mean it.”
Her tone wasn’t bitter — not even sad. Just… clear, like someone who had watched the seasons long enough to understand them.
“She listens. She sees. She lets people stand in front of her as who they are — but she doesn’t draw them near, not like a mother or a friend would. It’s not coldness. It’s… a different shape.”
She glanced up, eyes brighter now, steady.
“But I never felt she kept me away. She always received me — in her way. Never indulgent. Never dismissive. Even that first Tuesday… she looked at me and spoke as if I already belonged. As if I could meet her where she was.”
Her voice softened further, almost a whisper now:
“And maybe that was what made me try so hard. Not to win her closeness… but to become someone who could stand there — beside her. On the same ground.”
It struck him then — not like lightning, but like the turning of a key in a long-locked door.
What Filippa had just named — the shape of Dania’s closeness, the unflinching distance that was not rejection — was the very thing he had felt but not understood the day before, watching Dania work through the lines of the sick in the Basili hall.
She never bowed.
Not upward, not downward.
Donata had tried, with the elegance of pride, to press her back into a lower place. The poor raised their eyes as if to a holy image.
But Dania stood. Simply stood. Not above, not below.
And this girl — no, this young woman beside him —
She had spent seven years trying to do the same.
Not to win favor.
Not to please.
But to meet Dania on the same ground.
Deonardo had seen how frightening that place was — especially after their conversation the day before, when even language seemed to shake beneath them.
And Filippa?
She had stepped into it at twelve years old, barefoot in her uncertainty, with no name for what she was doing… and no armor to protect her if she failed.
He looked at her then — not as the merchant’s daughter offering spiced tea in her father’s guest room, but as someone rare:
a young woman who had not asked to be seen, but who had shaped herself quietly, day by day, into someone who could stand where others flinched.

🕯♛🐾 after the first conversation
He thanked her with a quiet smile, touched the rim of his empty cup in farewell, and stepped out. The door closed with a soft click.
Filippa didn’t move at once.
The silence returned — not the hush of sickness in the hall, but something different. This room had always been a place of jars and maps, of trade and memory. But now it held the echo of her own voice — telling him things she had never told anyone.
She stood for a moment, fingers still resting on the table’s edge.
Had she said too much?
Had she appeared too eager?
No… that wasn’t it. What lingered wasn’t regret — it was heat. The warmth of being seen. Not as the loved daughter in her family, not as the young Basili Lady in Valiponte, but as herself. The self she was still learning to hold upright.
She looked down at the cup he had touched — still faintly marked by the shape of his fingers — and then glanced at the map of the Levant on the wall. Her eyes followed the copper pins. Not the ports, but the roads between them.
“He asked questions like he was listening for something already half-known,” she thought. “Like he saw the path behind my words.”
And still, he had asked gently.
She exhaled, slowly, then took both cups and brought them to the small side basin. As she rinsed them, her movements were calm — but something had shifted. A thread inside her was pulled taut, quietly humming with the possibility of something more.
She did not name it. Not love, not even hope.
But she would like to speak with him again.
Not because he was noble, or elegant, or beautiful — though he was all these.
But because he had made her feel, for a moment, like she belonged in the world she had dreamed of entering.
Like the ledger of her life might yet hold more than one column.
I watched her from the doorway, curled between shadows. And though she never glanced my way, I saw what she didn’t name: a curiosity lit, a boundary crossed, and a quiet fire kindling in the space where future begins.


📖November 19, 1264 Wednesday
Returned soldiers 🕯♛♡🐾 , cloudy morning 


Middle of the day, sky the color of tin. Not cold, no wind.
Two horses came back through the gate.
Ten men had ridden out with Rinaldo and the year’s rent. Now Ferrazo and Agnolo “the Oak” returned alone — saddles stained, leather creaking in the flat air. No clatter, no shouting. Just the slow, heavy thud of hooves and the wet rasp of reins. The square noticed before anyone spoke; talk thinned, then stopped. Dogs did not bark.
And under the bell tower, Martino and the idle mercenaries were waiting — those who had not been chosen. Martino leaned against the wall, arms folded, boots crossed, half a sneer ready on his lips. The sight of the wounded cut it short. His eyes flicked from the blood to Ferrazo’s face, and something unsettled him.
A younger mercenary muttered: “Saints… is that all of them?”
No one answered.
Ferrazo swung down, boots thudding on the flat, cold stone. He did not look at Martino or the loungers under the tower. One hand stayed on his sword-belt — not to draw, only to steady himself.
Martino spat sideways to hide his nerves.
He said, voice cracking: “What did they meet out there — wolves? Or curses?”
No one laughed.
Fear began to move through the place in small currents: merchants whispering of ambush, peasants muttering of hexes, hired blades shifting their weight beneath the bell tower.
Ferrazo said nothing to the men in the square. He only jerked his chin once toward the infirmary, and the monks closed around the riders at once, guiding them off the stones and out of sight.
Inside, the monks moved without fuss — hands sure, voices low, each man taking a task and not letting it go.
Lamps burned low against the cold light seeping through the shutters; the air smelled of boiled linen and sharp wine. Ferrazo sat on the edge of a pallet, cloak half-torn, blood dried dark along the hem. Agnolo “the Oak” held himself rigid as they bound his wounds.
Father Lorenzo entered first, his face drawn tight, followed by Tessa, her hands clenched in her cloak. She did not wait for courtesies. Her voice trembled against its own restraint: “My husband?”
Ferrazo lifted his head, heavy with exhaustion, and answered:
“Alive, Madonna. He rides on. You may set your heart at ease on that.”
Relief broke her shoulders for a moment, but fear lingered in her eyes.
She asked: “Then what—?”
Ferrazo spat into the straw before his boots, his teeth grinding, his tone flat, too tired for embellishment: “Not bandits. We met soldiers. Fifty at least. Trained. Armored.”
Lorenzo asked, brow creased: “Soldiers? Whose?”
Ferrazo’s mouth twisted. He said:
“That’s the foul taste. They flew no banner. But their blades were no mountain rabble.” He glanced at Lorenzo, then at Tessa. “Four of mine cut down before we knew it.”
Ferrazo said, jaw set: “It was slaughter. Would have been worse. Then came the knights.”
Lorenzo’s tone sharpened: “Knights?”
Ferrazo nodded once, slowly, as if still not believing what he had seen. Then he said:
“Two only. Rode in from the east, clean as steel. They cut down twenty men like straw. I’ve never seen blades move so fast.”
His hand clenched on the pallet. “Not even wolves eat so clean.”
Lorenzo asked softly: “You knew them?”
Ferrazo’s gaze darkened. “Aye. I’d heard the crest. First time I saw it in the flesh. They were Lord Corrado della Rocca’s elites.”
The words hit hard. Bandages paused mid-knot; cloth stopped scraping. Even Tessa flinched, breath catching in her throat.
Ferrazo went on, quieter now, though rage simmered beneath. His eyes moved to Tessa:
“Your Rinaldo lived because of them. They cleared the road. Not for us — for Bologna. We were only in the way. I lost eight good men, and I do not even know whose war spilled their blood.”
He spat again, softly, then lowered his head.
A pause. No one could say how long it lasted. They just stayed within the weight of it.
Finally, Ferrazo looked at Lorenzo and said:
“The road is cursed, Father. Stronger wolves walk it now. If this valley thinks itself safe, it is blind.”


📖November 20, 1264 Thursday
History of bandits and Valley’s guarding customs🕯♛♡🐾 Thursday new moon

They found Romé where anyone with sense would look first: in the healer’s studio. Dania stood at the table, knife steady over roots; Deonardo beside the brazier, tending embers as if they were lines of a brief. I watched from the beam.
Lorenzo said from the threshold, giving the full dignity of the name: “Messere Romualdo di Orellana. Forgive the interruption.”
Romé set down the folded linen. “Father.”
Ser Basili stepped forward, courteous but plain. “We come for counsel — and, if you’ll grant it, for your hand. You will have heard what happened yesterday. Captain Ferrazo returned with only one of his men. Last Saturday, ten rode out with Rinaldo and the year’s rent to Bologna. Only two came back.”
Romé asked, without drama: “Bandits?”
Ser Basili’s mouth tightened. “An ambush — but not rabble. Trained soldiers. Fifty or more.”
Romé’s eyes narrowed. “More than fifty trained soldiers, and still two of yours returned? How?”
Ser Basili answered: “In the midst of the slaughter, two knights arrived — Corrado’s elites. They had come to receive Bologna’s rent. They cut through the killers like wheat.”
His eyes measured Romé. “I know the name Orellana. Your house has long served Lord Corrado della Rocca.”
Romé’s jaw set, but he said nothing.
Lorenzo drew a breath, laying the weight where it belonged. “Twenty years ago this valley nearly broke. Not thieves, but an ordered company — lost soldiers, near two hundred, with banners and captains — fortified a hill above Valiponte. They sealed the passes, bled the markets, starved eight villages through three winters and into a fourth. Wars swelled them; Pisa bought their swords and, by fortune, buried them elsewhere. We were spared by purchase, not miracle.”
He looked older than his usual days, fear and sorrow in his tone. “When I hear hoofbeats return fewer than they left, I hear that old sound again.”
Dania did not look up, but the knife paused once. Deonardo’s hand stilled over the coals.
—
At last Ser Basili spoke again, hands folded, voice low but clear. “In this valley, Signore,” he said to Romé, “we have no lord’s garrison. The Guidarini sit in Bologna. They do not send men for our roads. So we learned to keep order ourselves.”
He glanced to Lorenzo, then back to Romé. “The peasants pay their rents in kind — grain, oil, cheese, wine. All of it comes here, to San Michele. Rinaldo tallies it under the monastery’s seal. Part goes on to Bologna, as is owed. But not all.”
His fingers tapped once, controlled. “By old agreement, a portion stays in Vallefredda. Guidarini consented, provided every sack and jar is written and receipted. Lorenzo calls it Christian duty — protection for pilgrims, mercy for the poor. Rinaldo calls it necessity.”
Romé listened, elbows on his knees, eyes steady.
“Food does not pay mercenaries,” Basili went on. “Coin does. So the goods we keep do not sit in cellars. Maffei’s marble quarries must feed fifty, sixty men above Carrara. Instead of buying elsewhere, they take the rents-in-kind straight from Rinaldo’s lists — grain for bread, oil for lamps, cheese for the tables.”
He met Romé’s gaze fully now.
 “In return, Maffei pays coin. Not always gladly, but they pay. Those coins never touch Bologna. With Guidarini’s leave, they are counted here and go straight to men like Ferrazo. Winter after winter, that is how the passes stayed clean enough for caravans to move.”
Ser Basili drew a slow breath. 
“It is not a grand system. Just three houses doing what they must. The peasants’ rents are only part of the security fund; the houses of Valiponte contribute the rest. Each house adds its own share. Maffei feeds its quarries. Rinaldo makes the numbers stand straight. And Father Lorenzo”—his mouth twitched, almost a smile—“blesses the contracts and tells us God approves of well-paid guards.”
The humor died at once. His voice hardened. 
“Now eight men are dead, and wolves in mail walk the road. Such a company is not one hired blades can expel. The same coins will not be enough. We can gather more coins for more swords, yes. We can squeeze another measure from our own houses. But we do not know what we face, nor from which spur the blow will come next.”
He inclined his head, formally this time. “That is why we ask you, Messer Romualdo di Orellana. We can raise men. We can count sacks. We can sign contracts. What we cannot do is read the shape of a war as you can. Tell us how to spend what we have, so the valley lives to pay rent next year at all.”


📖November 21, 1264 Friday
I remember San Michele from before it had a name.🕯♛♡🐾 - Cacifer
Before bells. Before herbs. Before any fool ever thought to hang a painted angel over a ledge of rock that drops straight into the dark.
Back then it was only a fortress — Lombard stone and Carolingian fear. A square tower like a broken tooth, rough walls clinging to the spur above Vallefredda. Men in rusted mail, horses that smelled of iron and old sweat. They watched the pass for enemies and tolls, nothing more. No one prayed there; they cursed, drank, and bled.
Then the world shifted, as it always does.
Magyar raids swept through the valleys, lords gnawed at one another, and the garrison died or drifted away. By the time your year-count reached the nine-hundreds, the place was already hollowed.
The tower stood, but empty.
The walls stood, but cracked.
Bandits came, of course. Bandits always come when men grow tired of guarding.
For a while the ruin was theirs: fires under the broken vault, stolen cloth flapping from battlements, horses stabled where an altar would one day stand. Villagers feared the height above them; children crossed themselves when they saw the torchlight at night. The pass was no longer a road, but a wound.
And then, in 984, came the Benedictines.
Ah, I remember them well: too thin, too stubborn, eyes set like flint. Cluniac zeal in worn boots. They climbed the broken stair with nothing but a cross, a rule, and the certainty that God wanted this rock for Himself.
They did something very simple and very dangerous:
 they walked straight into the bandits’ den and refused to leave.
Some of the thieves fled.
 Some were driven out.
 One or two — the ones with eyes already tired of killing — stayed.
Those men cut their hair, took new names, and traded knife for psalm. In the stories told later, the peasants made it sound grand — angels descending, demons cast out. But I was there. It was quieter than that. Hunger, fatigue, and a strange courage met on that rock, and violence bent its knee for once.
That paradox is San Michele’s true beginning:
 a monastery founded in a place where swords had ruled,
 raised partly by the hands that once robbed its road.
They dedicated it to Michael the Archangel — warrior and judge. Sword and scales. A fitting patron for this precarious ledge where lives and cargo both must be weighed.
From the first winter, the monks understood the pass:
 snow, avalanches, narrow ledges, merchants with more fear than coin.
They built not just a church, but a refuge.
They roofed the lower terrace, turned it into a shelter where merchants could stall their mules and sleep with their boots near their heads. They kept a stable fire, hot broth, and a little chapel where a man could mumble to God before he risked the road again.
And in the old hall of the fortress they set up their true strength:
 the scriptorium.
Tables. Lamps. Ink thick in cold air.
 At first they copied psalms, then homilies, then the lives of saints.
But soon enough the merchants began to see what I had always known:
stone walls are good, but ink is better.
A sword can fail. A parchment, properly kept, outlives the man who signs it.
So the contracts came.
Charters of land.
Lists of tithes and toll exemptions.
Receipts for debts repaid.
When a merchant left a copy of his ledger under San Michele’s seal, he slept easier. If some lord later denied a bargain, the angel’s script could be brought forth, the scales of ink and law set right again.
Thus the double symbol took root:
The sword in the cliff:
 San Michele’s walls and men — shelter against bandits, a place where mercenaries were fed and guided, where caravans could cluster under watchful eyes.
The scales in the library:
 their ledgers and charters — weighing not only souls at Judgement, but grain, oil, tolls, and promises between cities.
Over the years a proverb grew in merchants’ mouths, worn smooth by use:
“At San Michele, the scales are just and the sword is sharp.”
It meant:
 If it is written there, no man cheats.
 If you rest there, no thief stays.
As the centuries turned, the valley changed its lords — Lombards, cities, distant Bolognese nobili who cared more for council seats than mountain snow. But the monastery stayed where it was, bone in the spine of the pass.
It never grew rich in lands. San Michele owned no vast estates, no endless wheat fields. Instead it lived like the creatures of that rock:
A strip of vineyard on the warm slope — enough wine for feast days and honored guests.
Goats for milk and cheese, their hides stretched into parchment.
Mules and horses, bought cheap, bred and rented dear to the merchants who needed them.
The real wealth came through the gate:
Cloth from Genoa, salt from the Ligurian coast, dyes from Florence, spices and sugar carried inland from the Adriatic.
 Merchants paused in the lower terrace huts; nobles and finer traders climbed higher, to the cloister guest range facing the rosemary garden. They paid in coin for beds, fodder, and guarded storage. They paid again, in larger sums and subtler favors, to have their contracts copied, sealed, and remembered.
On feast days the place changed its skin entirely.
 St. Michael. Christmas. Easter. Marian feasts.
Peasants from Valdiponte and the satellite villages climbed the path, leading goats, carrying wheels of cheese, bundles of woven cloth, sacks of chestnuts. The bell tower square turned into a market: bread, soup, scraps of wool, little jars of herbs sold under the shadow of the church. Children splashed in the fountain’s rim; monks pretended not to notice the wine skins passed hand to hand.
You might think such a place would be a pure sanctuary.
You would be wrong.
San Michele stood on a fault line.
Lucca called the Garfagnana its own.
 The Este of Modena and Ferrara sniffed at the passes.
 Florence and Pisa clashed like rams farther west.
Messages flew back and forth — Guelf and Ghibelline, Papal and Imperial, all thinking themselves cleverer than the last. And in between, on this ledge, monks kept their ears open and their mouths mostly shut.
Hence the whispers:
That San Michele was neutral — a place where messengers from both sides could sleep under the same roof, each pretending not to hear the other’s name.
That San Michele was never truly neutral — because any house that knows everyone’s secrets serves power whether it wishes to or not.
The library became a collector’s trove:
 Cluniac copies of old laws.
 Charters from forgotten donors whose grandsons could barely recall their titles.
 A herbal treatise brought from Salerno, patched and re-inked.
 A commentary on the Peace of God movements, copied in a winter when wolves howled closer than usual.
None of it was grand like Monte Cassino or Cluny itself.
But for this valley, this pass, it was enough to make San Michele a kind of memory organ — holding what the lords and peasants alike would otherwise lose.
The monks themselves? Fifty in winter, seventy at most, including lay brothers.
Half old, half young.
Some sent here as penance, others as refuge, others as quiet exile from louder courts.
They prayed, they shoveled snow, they mended roofs, they copied books.
They blessed caravans and, sometimes, hired the men who would guard them.
Always that duality:
Refuge and gatekeeper.
 Prayer and toll.
 Ink and steel.
Later, when the healer came — Dania with her southern herbs and her refusal to bow at every bell — the stories grew stranger. Some monks muttered “witch” behind their sleeves, even as they sidled to her door for remedies. Merchants praised her salves, peasants brought their fevers, nobles weighed her eyes with mistrust and need.
From outside, people said San Michele was blessed.
From inside, I heard it creak under the weight of all it tried to be.
Still, through wars, plagues, snow-blocked passes and years of empty roads, one thing remained steady:
Men and women with goods to move, letters to carry, treaties to hide, always found their way to that ledge.
They knew the proverb.
They trusted the seal.
Where the angel weighs, no man cheats.
Where the angel stands, no thief stays.
And I — Cacifer, once Lucifer, now cat on the terrace wall — have watched it all:
bandits turned brothers, scrolls turned weapons, prayers turned contracts.
San Michele’s legacy is not stone and bell alone.
It is this: a knife-edge place where heaven and commerce made a pact.
Where ink learned to act like steel.
And where, one winter in 1264, three particular souls — Romé, Deonardo, Dania — arrived for different reasons, and by finding one another, shook what had long stood firm.
But that, little reader, is another chapter.



📖November 22, 1264 Saturday 
📖 Cacifer’s Time-Travel — Return to September 16, 1264 — Porta d’Argento 🕯♛♡🐾

Today is November 22, the Feast of Saint Cecilia. Vespers runs long; the precentor threads the psalm tones with a special chant; if Valiponte keeps a musicians’ confraternity, they lend their tenors to the nave and let the altos carry the descant. Candles move like small rivers through the aisles. And the Brotherhood — ah, they love a crowd — may drift in at the edges with their singing and their lights, wearing borrowed piety like a cloak.
Deonardo didn’t speak when Lorenzo and Ser Basili visited, but the valley’s crisis reached him all the same. The weakness was the same that brought him to San Michele in the first place: a land of finely made strength — trade, craft, and letters — with too little iron to guard it. 
While the monastery breathed in measured chords, I stepped sideways. I entered his memory the way mist enters a room — quiet, unavoidable. Demon work. You would not have seen me; I was only the pressure on polished wood, the cool at a window latch, the space beside a flame where no smoke moves.
This was not the monastery. This was Porta d’Argento, capital of the Signoria di Vercado — Aurelia’s hill city above the bay — terraces rising in olive light, river-silver threading under its arch as a promise that also collects tax. Their emblem was everywhere if you knew where to look: a silver river beneath an arch framed by olive boughs — trade and peace on a small shield.
September 16, 1264 — Porta d’Argento.
Deonardo woke before the city admitted morning. Limewash held a pale gold in his chamber; walnut oil sweetened the coffered beams; terracotta kept the chill. He bound his silver hair with a black silk cord, washed from a hammered-brass ewer ringed in laurel, and opened his traveling desk. The day’s tools were a liturgy: reed pens, pumice, penknife, jettons, red and black wax, the hawk seal, a palm abacus polished by years of sums. A servant entered with the bow this city saves for those it trusts — palms joined, eyes down until his micro-nod loosed them. Bread still steaming, cheese with a modest rind, rosemary water. He ate as men do who believe food is an ally, not theater.
The palace moved like a machine that respects silence. Alabaster panes poured milk-light along the east loggia; capitals wore vines and small beasts with discretion; rush mats made footsteps moral. Officers and minor nobles greeted him with trimmed warmth — two fingers to the breast, a fractional bow, a phrase polished by use. He paid them back in exact coin and entered the long room where decisions dry.
Petitioners stood tidy as a ledger. A dyer held too long at the gate. A mason shaved by a steward. A carter whose axle died on the north grade. Deonardo listened for arithmetic beneath complaint — grain, time, pride. Jettons migrated: bolognini to wages; florins to salt; a purse to the ferry repair. Parchment took ink that refuses ambiguity; sand drank the excess; beeswax and iron gall laid their sober scent over everything.
By midmorning the court drew him like a magnet. Donna Aurelia di Vercado — lordess of this Signoria — sat in winter-fine wool the color of smoke, a narrow circlet that never had to argue for itself. The great hall was frescoed in mineral patience: saints in the apse to remind the city it answers elsewhere; ships and markets on the walls to remind the saints what cities cost. He took his half-step to her left — their compact visible in the angle, shoulder to shoulder — and the envoys were shown in.
They came from the Brewing Alliance, neighbors squeezed by the Marchese of Monteferrano’s son: doubled criver tariffs, hilltop outposts, mercenaries patrolling what did not need patrol. Their question, poured sweet and repeated: When will Vercado sign? Their danger hid in the sweetness: sign too soon and fix the valley Guelf against Ghibelline before harvest is safe; refuse and invite the Marchese to prove ambition with imperial backing.
Aurelia let them spend their sound. Deonardo read slope and water on the map while they performed loyalty. When her ring tapped once on the river mark, he folded two sentences and bought the city an hour of future calm.
“Porta d’Argento is a crossing, not a drum,” he said. “We will not lend our people to someone else’s banner before we have priced the winter. The Marchese’s son prefers gestures to grain. That is his confession. Let us count what we can move without sharpening his appetite.”
He laid the plan as a merchant lays cloth:
 — Hold strict at the gate; be merciful inside.
 — Drop the river toll for two market days to starve his tariff of spectacle and make allies by relief, not oath.
 — Advance credit to the hill towns he has frightened, but do it through their women and guilds, not their councils.
 — Shift olive export to the southern road for a month; let him fortify to watch emptiness.
 — Write to the bishop two valleys over — no faction named — requesting a visitation on mercenary abuse; bishops travel slowly, but letters travel with them.
 — Send wine, salt, and a very clean scribe to the Marchese’s steward with a question about bridge maintenance that forces him to answer in writing; a written answer is a leash if you know your knots.
Aurelia listened with the attention of someone who does not waste heat. Their intimacy was calibration, not display: her palm flattening zeal where it would burn coin; his forefinger resting at the dyers’ quarter to remind the room who keeps the market breathing. When an envoy pressed for oath, she smiled as people do who have already chosen and prefer others to think they helped. “Vercado remembers who we are,” she said. “We turn without spilling.” Vercere — the verb inside her name — bowed to the room.
After council, couriers traded boots for bread at his door. He wrote three letters that sounded like four: one opening the granary at reduced price for a week; one refusing a merchant’s exemption and keeping him as an ally; one notifying bridge wardens to measure stress before rain; one thanking a guild-mother for not letting panic become policy. He corrected a tithe roll in a margin where only just readers read. Loops of harm were caught upstream and unknotted without applause.
If the day allowed an hour, he spent it in the artisans’ streets. Glass cooled sky-blue in a long breath; looms thudded yellow out of weld; a gilder beat leaf until it believed it was light. He asked to see the undersides — backs of carvings, hidden seams — because lies live where eyes don’t. Nobles in litters practiced their choreography; he returned exactly the recognition that keeps cities from tripping on their own feet.
Dusk put honey in the corridors. Lantern horns made the air gentle. In Aurelia’s solar — a pietra serena table worn honest at one corner, two chairs that respect tired spines, a brazier that heats without boasting — they took the day’s last measures. She set her circlet on bare wood. He poured rosemary water and placed it within reach without performing humility. “The Brewing Alliance will await our turn,” she said, which in her mouth meant: we will act, but not to someone else’s drum. He laid his hand flat on the ledger; she watched the weight move from her to him and did not call it love. Intimacy here wore a plain dress and earned its keep.
Night drew him up to the house the clerks call a villa — loggia over two cypresses, trestle table with a truthful scar, brass lamp that burns steady, dye ribbons on a small shelf labeled in his tight hand: Weld, Alder, Walnut. The city hummed like a hive that knows its queen. He opened the shutters and let the river draw a cold line along the floor. Two witnesses counted coin; keys turned; a purse was left for dawn — the courier’s wage, the gate midwife’s passage, the candle for the clerk who sees what others pretend not to.
This was a rich land — trade, making, art, beauty — and he refused to be its ornament. 
He remained its grammar. 
And then, the sleepless midnight. He lay awake, struggling again for a way to keep the land neutrally safe in the middle. What if there were a miracle — an ancient, undeniable writing to keep the Marchese’s son away? A charter older than appetite, a letter that made neutrality into obedience to God rather than defiance of men. A plan began to gather itself — not yet shape, but pressure.
One month and a half later, under a scholar’s plain cover and a courteous request for remedies against winter fever, Deonardo di Alferini arrived at San Michele.

📖November 23, 1264 Sunday
Dania Reads the Blade Reed_San Michele
After Lorenzo and Ser Basili came to ask for help, Romé answered briefly. He neither accepted nor refused. Calm, almost unreadable, he gave them precise tasks:
“From Valiponte and the valley villages, collect names of witnesses and who sent them; what was seen, where, and at what hour; routes taken, tracks, horseshoes, cart ruts; weapons noticed; accents or badges; any rumor of paymasters or banners. Write dates. Separate hearsay from sight.”
Then he said, “Next Tuesday I will come down to Casa Basili with Dania. I’ll hear what you’ve gathered.”
That was all. After they left, he was quiet. Over the next three days, without explanation, he took out his sword and began to prepare himself—oiling, honing, and practicing in silence.
In that same hush, Dania found the scabbard propped against the end of the bench, leather darkened by use, chape dulled by travel dust. She did not reach for it at once. She watched how the strap lay, how the silk at the grip had worn smooth in two places: one where the thumb rested before a draw, one where the fingers pinched for a quick recovery. When she finally lifted it, she did so with a healer’s hands—testing weight before purpose.
She eased the blade free a hand’s width. The watered pattern caught the window light and went still again. “Forward balance,” she said, mostly to herself. “But not greedy. You don’t batter; you redirect.”
Romé was at the table, preparing his gear for the ride. “Go on,” he said, without looking up.
Dania drew the sword fully and let it float in the air. She did not cut; she traced the arcs a body would want to make. The edge told her its habits. She turned her wrist and felt the slight bite of the false back bevel near the point. “You added this later,” she said. “After the first season. For inside work—close quarters, quick reversals. It isn’t meant to frighten. It’s meant to end decisions.”
Romé looked over, a shadow of a smile at the corner of his mouth.
She stepped once, twice, feet silent on the stone, and moved the blade in a short, decisive draw as if clearing cloth from an opponent’s forearm. “Your marks are straight,” she said. “You write on people where pride lives. Collars. Cuffs. Cheekbones.” She turned the blade flat and offered it back across her palms. “And when you must, you write the last line.”
Dania’s gaze softened—half appraisal, half recognition. “It’s a reed,” she said finally. “You write.”
He nodded once, surprised and not surprised. “A good reed listens to the hand holding it.”
“And to the wind moving through,” she added.
He took it lightly, tested the balance out of habit, and slid it home. The scabbard whispered; the leather at the throat swallowed the rest. Dania noticed how he did not look down to re-sheath. “The throat is cut wide,” she said. “You return the blade by memory.”
“By promise,” he said.
She tilted her head. “And what does it promise?”
“That if it comes out,” Romé answered, fastening the strap, “it has already decided what it will say.”
For a moment he felt strangely seen—more naked than when anyone had stripped him. Reed was the part of him he almost trusted more than his body: the habit of refusing force, the choice to mark before he cut, the promise to finish only when there was no other page left to write on. Watching her trace the air with it, he felt relief, even gratitude. She would not make it perform. She would not force it to prove itself. She moved like someone listening to a door she refused to walk through.
And yet a thread of unease ran through the relief. It was in the way her breath shortened by half a measure when the point came inside her span; in the way her off-hand hovered to catch, then did not, as if her body kept memory that her mind no longer allowed. He didn’t know her past, but he recognized the pressure of an old discipline held at arm’s length. It felt like standing beside a river that had decided to be a well.
What he felt most was respect—the rare kind that steadies desire instead of sharpening it. Dania had touched the closest thing he owned and returned it unchanged, but not unspoken to. Reed had answered her, and she had heard it, and nothing in the room had to bleed for that truth to exist. Romé thought of Aureliano then, not with pain, but with the memory of being witnessed. He realized Dania had done the same for his blade—and, by that, for him.
I remembered how lightly he said it—light enough to pass for a joke if anyone needed it to. Romé buckled the strap and tapped the chape against the cot leg, then glanced up, half-smile, half-dare.
“If I die, could you keep it?”
Dania looked at him, then at the Reed, and let a small silence do the weighing. When she answered, it was without softness or sting—just truth.
“Better to somebody who uses it.”
The words landed clean.
“I can keep your letters,” she added after a moment. “Your ash. Your name for it. But a sword belongs in a hand that will speak its grammar.”
Romé’s smile thinned, then steadied. “Then keep the ribbon,” he said, touching the indigo at the grip. “Choose the hand when I cannot.”
“I’ll guard it until it’s claimed by someone who listens as it does,” she said. “But I’d rather you choose it yourself.”
“Then I’ll have to live,” he answered, almost cheerful.
“Do,” Dania said.
Outside, the kettle clicked off the boil. Inside, the Reed rested.


📖 Adriatic Coast, Summer 1249 — The Day the Reed Chose Him 🕯♛♡🐾
I have worn fur and shadow and winter’s breath, but steel is another night entirely. When I slip into a blade, I do not purr — I ring. I pressed myself through the grain of the grip, slid down the narrow spine, and the sword’s memory opened to me like water taking moonlight. Salt first — pitch and warm fish, gulls threading the sky — and two boys riding out of the Wild Corridor as if the mountain itself had signed their leave.
Aureliano at fourteen: new height, clean strength, a mind already measuring the world. Romé at thirteen: tall and fine-boned, sun-browned, tuned like a drawn bow — elders frowning and looking again. The port was a thicket of masts and lateen sails, the air full of indigo and pepper, Levant glass holding daylight like breath you dare not let go.
On a coil of rope sat a man with a scar through one eyebrow, oilcloth across his knee. “Sarkis of Acre,” the steel remembers him saying, Italian careful, eyes not. He unwrapped without show: short arming swords of honest temper, two long knives with horn grips, and — almost afterthought — a slender curve in oiled linen. When he drew it, the metal was not bright. It was darkly lucid, watered like river-silk. Farther east than boys are supposed to imagine truthfully.
Aureliano looked first, as he always did, weighing work before wonder. Romé did not touch. He let his breath find the blade’s weather.
“Try,” Sarkis said, offering the steel across both palms the way a question is offered when a man already knows the answer.
I felt the balance again, the way the memory keeps it: just forward of the guard, eager but not greedy. Romé’s wrist turned — not to show, to listen — and the edge told him where it lived. Aureliano, dutiful, took a heavier straight blade, tested a parry against a pier post, nodded at the good sense of it, laid it down. Romé stepped back with the curved one, lifted, and the world narrowed to the span between his thumb and the steel. That was the first yes — not from the boy to the blade, but from the blade to the boy.
Sarkis whistled, and a smith came from the shade: Niko of Ragusa, forearms charred with old fire. Fig-wood core. Rawhide. A roll of plain linen for the grip. Bronze for the guard. No jewels. No story. Work first.
Under a mast’s shadow he built the hilt to Romé’s hand — trim, bind, test, trim again — setting the quillons low so cloth would not catch. “You don’t crash,” Niko said, widening the scabbard throat with a file the size of a prayer. “You pass through.” He cut a wet rope from a yardarm, left it swinging its cool weight. “Write one line.”
Romé drew. The smallest motion possible. The rope parted with a whisper, all its water clinging to the cut face for the briefest proud instant before letting go. Aureliano’s grin slipped toward mischief and pride together. Sarkis rubbed his eyebrow and spoke a quiet sentence in a language neither boy pretended to know.
They haggled like sons carrying a mother’s folded letter of credit and a father’s instruction to spend like men who mean to earn it back. Aureliano did the numbers. Romé did the listening. The price settled like a well-tied knot.
Before they turned away, Niko took the file to the last hand’s breadth of the back and drew a whisper of bevel there — no more than breath — “for the space inside an elbow,” he said, “for the moment after pride.”
I held the rest of the walk the way steel holds starlight. The new scabbard slung high; gulls stitching air; ship-bells counting a larger hour than any their valley had yet allowed. They did not name the blade aloud — not men who make a pact with tools. Still, as they passed back through the city’s throat toward the road home, Romé leaned once toward his brother and let the rightness win him.
“Il Calamo,” he said, almost embarrassed. “The Reed.”
Aureliano bumped his shoulder. Agreement enough.
From there the sword’s memory becomes the grammar of a life. Isabella’s indigo silk would come later, spiraled soft over the grip so long fingers could rest and rise without slipping. The leather at the scabbard mouth would be cut wider, so the blade could return without looking. The faint false edge Niko whispered into being would one day write the last line when talk had run out. But all of that belongs to the choosing, and the choosing belongs to this day on the Adriatic, where strange ships perched and two brothers learned that a true tool speaks back.
I felt the blade read them both. It noted Aureliano’s steadiness — the boy who would count costs before the fight began, who loved the straight line that made a house stand. It recognized Romé’s refusal to crash, his hunger for the narrow door a body can pass through without tearing the fabric that does not need to be torn. The steel liked that hunger. It understood it. The balance shifted a breath toward the guard, a courtesy no eye could see, and the edge laid its future along the boy’s wrist and said, in the language metal uses, yes.
You ask how I know? I was inside it, little reader. I felt how the salt air skinned the temper and how the boys’ pulses set a rhythm in the hilt. Blades remember. They keep the heat that birthed them and the hands that first believed them. When Romé lifted the Reed again on the pier, his breath fell in with the sword’s, and the sword, being older than both boys and their houses, did what wise elders do: it chose.
“Write one line,” the smith had said.
He would spend years honoring the command. Not to frighten — to end decisions. To mark collars, cuffs, cheekbones where pride lives; to finish only when the page refused any more ink. A reed listens to the hand holding it, and to the wind moving through the hand. That is how I knew him later, when I watched Dania lift the blade by a healer’s habit and hear its habits without making it perform. That is how I knew him earlier, here, at thirteen, on a pier where glass held the light and pepper gave the wind a throat. He did not need the blade to make him a man. He needed it to let him write what he already knew: pass through; do not crash; return by memory.
And I, who have walked centuries on four feet and sometimes on none, tell you: the best choices are the quiet ones, the kind that cut a wet rope with a whisper, hold their water for an instant, and then let it go.

📖November 24, 1264 Monday

 What the Vallefredda Crisis Means for Romé 🕯♛♡🐾waxing crescent
I slipped into his past the way ash slips into broth — so fine no hand can strain it out. In the healer’s studio he was quiet, but quiet is never empty with Romé. When Lorenzo and Ser Basili left, he gave them tasks and no promise; then the room filled with the shape of all the wars he has already lived.
He grew under Lord Corrado della Rocca, where the sword writes law and the ledger is an afterthought. I watched him as a boy, long-boned and bright, following Aureliano across hawk-run holdings that should have thrummed with grain and trade. Instead, roofs sagged, ovens cold, seed stores thin — the price of a lord who sells glory abroad and rents out fear at home. Again and again the brothers argued the same circle: how to make villages breathe — roads, mills, seed, fair tolls — and again and again the plan broke on Corrado’s habit of ruling with the blade first. Corrado became a legend other lords and richest merchants would pay for; his own land, he did not keep.
Tommas della Rocca tried, poor steward of a hungry house, but a steward cannot make rain. Aureliano burned himself down to kindness and bone to hold the line. Then in the winter of 1260, the snow took him. I remember the silence that moved in after — a silence with teeth.
This past October, Corrado sent for three of his elites to answer Bologna’s call — a neat little cull to prove the old lion still roars. Romé was one of the three. They won the field the way trained steel does. But when the line broke, the scattered soldiers went to the villages — not to fight, to feed. Unpaid, unled, unashamed: ovens kicked in, seed grain stolen, the old man by the stove cut down while the soup still hung on the hook. That is the shape carved deepest in him, the shape he knows how to stop.
So when Ferrazo named “soldiers,” not rabble, I felt Romé’s mind take the straightest road. Not demons in the pass. Not witches in the trees. Scattered companies, hired and then forgotten by rivals of Bologna, now eating the valley itself because hunger walks faster than pay.
The ache rose — Aureliano’s arm across his shoulders, the weight of a brother who believed the land could be kept. For a breath, panic leapt as it always does when memory finds him unarmed. Then his hand touched the Reed.
Steel set his breath in order. I heard it — the small inward click when a body chooses direction. He did not swear an oath, did not make a speech. He oiled and honed. He laid out the scabbard and buckled the strap. He asked for names, routes, tracks, accents, hours. He told them to divide rumor from sight and to write the dates. He promised only this: “Tuesday, at Casa Basili, I will listen.”
Under it all, the old argument with Corrado kept breathing — the one between sword and livelihood. Corrado sells the sword. Romé, who learned under that shadow, now chooses the other task: keep people alive. Do not let ovens go cold. Do not let seed be eaten. Do not let old men die by their own hearths for the price of another lord’s gesture.
He did not say any of this aloud. He did not need to. I am a cat. I hear the grammar of a man more clearly than his words. And the grammar in him was simple as winter bread: count what we can save, spend only to keep the road open, strike only where it stops the next theft. If the foe is scattered mercenaries, then the cure is not banners but discipline — guides on the ridge, watchers at the fords, coin to the right hands, a blade that writes the last line only when it must.
I lay along the bench and listened to the Reed rest in its throat. The leather there is cut wide so the blade can find home without looking. Romé has lived long under a lord who loves the draw. He has learned, instead, the craft of the return.
The valley thinks it is asking him for war. It is really asking him for bread next spring. 
He knows the difference.
📖November 25, 1264 Tuesday
The Council in Casa Basili 🕯♛♡🐾
Tuesday, as always, Dania rode to Casa Basili to see the patients, and Romé and Deonardo rode with her. I woke the house before the bells, tail held like a quill, whiskers tuned to butter and politics. I inspected Dania’s boots, approved Romé’s cloak with a delicate paw-print, and performed my sacred duty on Deonardo’s hem—one soft head-bonk, because even the clever require a blessing of fur. Fiamma tried to roll an eye at me; I answered with a contented purr. We struck a pact: I would sit very regally, and she would pretend not to notice—so she allowed me to ride with Romé. When we reached Casa Basili and Dania slipped into the hall of herbs, I followed Romé at a dignified trot, ready to supervise both medicine and strategy.
By Sext the great table in Casa Basili was already ringed: Ser Basili at the head, Lorenzo beside him, Ferrazo broad-shouldered and bandaged, Deonardo present but silent, two elders from the council, and three villagers called as witnesses. Romé stood, not seated, one palm on the map-board, eyes steady as if the wood could listen.
I took the beam above the hearth. Heat in my belly, ears open.
The first witness spoke, a miller from the east slope. “I saw them at dusk—tents low, fires banked. Not peasants. Helm and jack. They moved like men used to orders.”
The second, a carter: “Horses shod with new iron. Two sentries. No banner.”
The third, a shepherd: “A tall one they called ‘Capitano.’ He did not steal the goats. He paid with a silver clipped close. Told us to keep quiet and keep the path clear.”
Romé nodded once, then said, crisp and spare: “We confirm the shape. Scattered soldiers, mercenaries from Bologna’s losing side. A captain has gathered them. They are raw yet, but forming a camp on the east ridge. If left, they will net themselves in the nearby valleys and grow into a company. We know that nightmare.”
Lorenzo’s mouth tightened. Ferrazo’s jaw set.
Romé’s questions began, each like a pin put through a wing.
Romé said: “Numbers. After the two knights cut down near thirty—how many remain?”
The miller: “I counted fires—ten, maybe twelve. Four men to a fire, not all mounted.”
Ferrazo answered, measuring: “Call it forty to fifty still standing, if the count holds.”
Romé said: “Camp. Where?”
The carter pointed with a blackened nail. “Here—below the east beech stand, above the shale run. Water close. Trees to mask smoke.”
Romé marked it with a neat cross, then asked: “Captain. Manner?”
The shepherd said: “Cool. Not cruel. Hungry-eyed. He keeps men from drink till dark.”
Romé’s gaze passed over us like a blade that chooses not to cut. “Good. Now—what result do you expect?”
Ser Basili blinked. “Result?”
Romé said: “Say you win. What is the best shape of winning for Valiponte? Drive them off? They return. Kill them? Their friends replace them. Buy them? Perhaps they keep the road. You must choose a shape before you choose a move.”
Lorenzo said, careful: “If they could be made to leave the valley—”
Romé cut in, not unkind: “They will be back by hunger’s law. Or another band will take their place. What can this valley actually offer?”
Ser Basili answered, carefully: “Coin enough for a season if we squeeze. Bread enough if we open stores. Men—about two hundred townsmen who can loose a crossbow with sense. Ten of Ferrazo’s still sound.”
Romé said: “Arms?”
Ser Basili: “Crossbows, a dozen spears, leather jacks, a few old mail shirts. No heavy horse.”
Romé nodded once. “Then hear the leverage on your side. You have walls, winter food, a name for honest contracts, and two hundred men who can make a sky of bolts if the gate is threatened. They have hunger, scraps of pay, a captain who prefers order to loose theft, and no banner to shield them from a bishop’s anger. Your advantage is not glory. It is grammar.”
Deonardo glanced up at that, the faintest ghost of a smile.
Romé laid the palm of his hand flat on the board. “My suggestion: make an offer. Hire them as mercenaries under Captain Ferrazo’s command to guard your own roads. Wage by the month, bread by the week, quarter outside the town. Terms written under monastery seal. Obedience required; abuses punished. If they accept, you turn wolves into dogs. If they refuse, you have named the terms of peace in public and may name them enemies with clean hands.”
Ser Basili exhaled, then leaned forward. “Who will carry the offer? It must be a man who can stand inside a ring of fifty and not shake.”
Silence followed, close as a glove.
Deonardo spoke at last, voice even. “Matteo.”
Every head turned.
Deonardo said, simple as stating weather: “My older servant. He has been a soldier longer than most men are fathers. He knows how to speak to captains and how to read the hunger beneath their words. Entrust the errand to him.”
Ferrazo measured the idea, then grunted assent. “I can ride shadow on him with two, no closer than an arrow-flight.”
Lorenzo said, quiet but firm: “Let Matteo go—and take Gianno with him. Gianno knows the east paths and the men who use them. He came to me last night of his own will, asking to help against this company.”
Romé considered, then nodded once. “Two is better. Matteo speaks. Gianno reads ground and watches flanks.”
Ferrazo answered: “I’ll keep them in sight without spooking the camp.”
Ser Basili asked the next plain question. “Where will Matteo find them?”
Romé took up a bit of charcoal and drew quickly—ridge lines sloping to the shale run, two fords below, the mill path threading up, the beech stand masking smoke, water marked within an easy hour’s guard. “Here they are. Fires counted place them near the beech; fresh shoes mean they stay close to stone, not bog. He goes by the mill path at dusk, flags peace with an unstrung bow, stops short of spear-cast, calls for the captain by title, and keeps his hands where eyes can count ten fingers.”
Ser Basili’s mouth thinned. “And if they pretend to accept—come to ‘parley’—and use it to enter the town for slaughter and robbery?”
Romé met it without decor. “Then you meet them outside. Captain Ferrazo goes down the slope with as many of his own as he can still command—perhaps ten. I go with him. We set the terms on open ground where the town’s stone does not tempt them. If they mean treachery, Ferrazo will smell it in their feet before the first word is done.”
Ferrazo’s eyes flicked, cold-pleased. “I will.”
Romé added, already moving to the next hinge: “Meanwhile, put your two hundred crossbowmen on the wall-walk as reserve. Not bunched—spaced. Quivers full. A bailiff on the bell with orders he can understand. If they rush the gate, you make the air sting.”
Lorenzo said, low: “And if they accept and then fail in the keeping of the peace?”
Romé answered: “Then the contract names fines first, exile second, death last. Bread before blood, always. A captain who prefers order will prefer coin that arrives on time.”
Ser Basili looked around the table, then back to Romé. “It is a hard bargain,” he said. “But hard like winter—useful. We can offer wage for forty men, bread for fifty, for six weeks. After that, the council must vote a larger purse.”
Deonardo said, almost to himself: “Six weeks is time enough to teach obedience—or to learn where it refuses to be taught.”
”
Ferrazo rose half from his chair. “When?”
Romé: “Tonight we write the terms. At Lauds we seal them. At None, Matteo walks.”
All eyes turned to Deonardo. He inclined his head once. “He will be ready.”
I let my claws flex into the beam. The room tasted of iron, wax, and a plan that could either feed spring or salt it. But when men choose grammar over glory, cats purr. I purred.

📖November 26, 1264 Wednesday
📖 Gianno — Ash Tree man  🕯♛♡🐾waxing crescent
I will tell you a story from the side of the road, where men often decide what they are. You can trust me; I have made a profession of watching choices.
Four winters ago, under the ash trees on the east path, a man lay bleeding where his fellows had left him. Not slain for cowardice, not spared for kindness—simply discarded because his wound would slow the plunder. He had a cut across the ribs, deep enough to empty his warmth, not so deep as to deny the world one more story. The ground smelled of old leaves and iron. He pressed his palm there as if to keep the hill from tipping.
I saw him first from a branch that held the last brown clatter of seed. Crows kept their counsel. The wind offered no opinion. Then hoofbeats came—one horse only, black, patient, with the restraint of a creature who knows its rider has a use for quiet. Dania dismounted without flourish. She asked nothing of his name. She looked, and looking was enough. Binding is a language; she spoke it in a few, exact syllables. She cleaned what needed cleaning, set what needed setting, left what wanted leaving. She did not promise him a future. She gave him a chance to reach it.
A month later he walked into San Michele on his own feet. Not a confession in sight. Just a man with a scar that had decided to heal and a face that had decided to work. The monks read him like a ledger at the gate: rope scars at the wrists, a knife worn thin at the heel, mismatched hobnails, rein-callus on the right palm and a thumb thickened from tightening wet straps—eyes that counted exits before faces. He took the wall where no one could circle him, and the monks wrote him down without a confession: camp-trained, road-bred, a bandit’s sense still screwed on straight.
Some muttered. Some moved away. Father Lorenzo moved toward him.
There is a law older than the ink that keeps it. Lorenzo carries it without displaying it: this monastery was raised to keep merchants from wolves and thieves. If wolves can be turned by bread and work, even the wolves may stay. He offered Gianno no ceremony. He offered a job list.
Brother Pietro took the risk that looks like practicality from the outside and like mercy from within. The vineyards needed hands, the storerooms needed backs, the outer gate needed a hinge of a man who would not bend more than the hinge. Gianno bowed his head the smallest amount that keeps pride from shattering and said he would do what he was given.
He learned the slope of the terrace wall and the stubbornness of lime. He learned where the frost bites first on the yard and how to lay straw so a cart meets morning without complaint. He carried the weight that belongs to the unremarkable: skins, sacks, barrels, bodies on the bad days when the rope must be held steady while the prayer is spoken. He did not speak much. When he did, the words came out like gravel rolling where it should. He prayed with his fists closed, then opened them by degrees as if teaching the hands a gentler grammar.
No one was fond of him. That is different from not being respected. Fondness depends on warmth; respect depends on truth. The truth was this: he rose early, he did the hard jobs, he did not quarrel with orders or steal the hours. He did not pass gossip through the cloister like a bowl of soup. When Dania passed on the high terrace he looked down, not with shame, but with the control of a man who refuses to tug a thread he cannot afford to follow.
The first winter under our roof weather-tested him. The road closed twice, the stores thinned, the infirmary filled, and the bell for the dead struck once too often for comfort. Men under pressure either break along their worst vein or set like good clay. Gianno set. He stood gate-duty through a night that had teeth, traded shifts with a novice whose courage had not yet learned how to dress itself, and walked the north wall at dawn counting the chimney-smokes to make sure the poorest two houses still believed in morning. No one thanked him. He did not appear to require it.

When the ten rode out with Rinaldo and the year’s rent and only two returned, San Michele wore the subtraction in its shoulders. The air thinned. Oil burned a little dirtier in the lamps.
That night Gianno went to Lorenzo of his own will. No preface—only the fact of a man laying down his particular history where it might be useful. He said he knew the east paths and the kind of men who walked them. He said, simply, that he could still fight. 
Morning came the way it does when a plan is heavy: without brightness, without delay. The council in Casa Basili weighed its words. Romé made his map in charcoal, the lines firm where ground forces the choice, light where the future still might turn. When the question fell—who will carry the offer—Deonardo said “Matteo,” and Lorenzo said, “Let Matteo go—and take Gianno.” Ferrazo measured both and nodded because sometimes the simple adds up to the necessary.
Lorenzo placed a loaf and a flask near their feet in the yard, as if the day itself were a traveler who needed seeing off with decency. He did not bless them; he assigned them to the hour. Matteo adjusted his quiver by the width of two fingers, then set the oilskin packet flat against his breast: the parchment sealed under the monastery’s stamp, the valley’s offer to a captain who might yet prefer coin to hunger. His kit was quiet and complete—good mail under a fitted jack, winter shoes on a steady horse, a hooded lantern, spare bowstring, a knife that knew its own weight.
Gianno checked his string for fray and then stood with his hands open where eyes could count ten fingers. His gear was the valley’s plain truth: patched leather, a crossbow from the storeroom, three bolts with unmatched fletching, a rope, a knife with one horn scale chipped. The contrast showed without shame.
Deonardo stepped in and set a hand on Matteo’s shoulder—no priest’s rite, only a clear charge: go, return. Dania passed close beside him and stopped. Gianno turned to her and did not speak; he bent in a long, deep bow that carried reverence more than plea. She received it with stillness.
 They mounted. Matteo took the mill path; Gianno fell in at his flank. The gate lifted its bar, the hinges breathed, and the two of them rode out into the grey.
The day was already shortened to its rind. Dusk came fast. The sky held a light, rinsed blue, scrolled with pink like ink that had just begun to bloom. When the sun leaned against the mountain shoulders, the ridges caught a last band of gold, and above it the waxing crescent hung—pale, fine, and white—in that layered light.

You might want a swelling finish here, with vows and the clatter of resolve. That is not Gianno’s style. He walks like a man who remembers what running costs. He carries the bandit still in his bones—not the boasting, but the measuring. He has learned to spend motion like money and to save speech for when it buys something. If you ask me whether he is redeemed, I will answer like this: the ash trees are still on the east path, and he has passed them many times without once lying down.
He is a lay brother now, always half in Pietro’s shadow, always in Lorenzo’s ledger as a column that balances when the day closes. He prays in iron. He works in stone and straw. He belongs, not because affection has made a warm nest around him, but because the place has room for a man who shows his worth by leaving fewer problems behind him than he found.
When he walks out at None with Matteo to carry the contract to wolves who might agree to be dogs, I will walk the garden wall and keep watch in my fashion. If he returns, the key will still open his door. If he does not, someone will say a psalm with gravel in it, because that is the right sound.
I am Cacifer. I collect stories, but I watch men more carefully than tales. Gianno is not a legend; he is a lever. In times like these, levers matter. They move weight without wanting to be loved for it. And that is why I keep him written here, under ash and iron, where the road remembers who stood up and kept walking.

–

📖November 27, 1264 Thursday
The day of Soft Paws and Hard Plans–Cacifer watches Deonardo move 🕯♛♡🐾
Unlike the others, this was when Deonardo moved—Thursday, the day after Matteo and Gianno had ridden out under Lorenzo’s plain blessing and the gate’s iron had swallowed their hoofbeats. The monastery felt hollow in their wake: wool-wrapped voices, spoons set down more carefully than needed, novices walking as if the stone remembered every step. Caution moved like cold through the cloister. Under the bell tower, Martino talked too loudly to hide listening; in the bakehouse, the lad over-salted the noon loaf. Even the lamp-smoke kept close to the ceiling, as if unwilling to be noticed. I tasted that pressure on the air and watched it settle behind Deonardo’s ribs the way a swimmer learns to use a weight. Then he swung to the saddle and rode for Valiponte with Luca steady at his stirrup— and I, of course, leapt onto the cantle behind him, tail high, to supervise the day from a proper height.
They rode down while the valley rehearsed its calm. Valiponte’s arch kept its practiced yawn of welcome, but the street heard sharper: smiths working with hammers muffled in rag, tanners arguing softly, shutters hesitating before they opened. Rumor had already chosen its own route—three pilgrims telling five versions, a fishwife naming curses, a guild boy insisting he’d seen a mailed boot-print in the frost where no patrol should be. The Brotherhood drifted like smoke along the poorer lanes, fingers warm on a beggar’s wrist, words colder than the alms: talk of judgment, of hidden sins that call wolves to roads, of healers who bend God’s order. Shopkeepers kept their faces smooth and their inventories nearer than usual. Deonardo let all of it wash around him and looked where the current set: toward Casa Maffei.
Deonardo spoke to council men as if unease had driven him from the cloister—hands folded, voice grave, the proper guest troubled for his hosts. In truth, he was counting Maffei. At the loggia he listened to the tally of quarry rations and heard who signed the chit. At the scales he watched which carts were waved on with the quick nod reserved for patrons. At the stationer he bought wax and learned whose seal pressed Donata’s invitations, and how often. At the goldsmith he weighed a chain for a reliquary lid and learned which feast she meant to adorn. In the quarry yard he watched how her name traveled: as wages on Friday, as bread on Monday, as rumor every day between.
He mapped her circle the way a surgeon maps veins. Bevilacqua, the steward whose hand greased every hinge. Maestro Giusto, whose chisel sang louder when patrons were present. Sister Benedetta, the confessor’s cousin, ferrying scraps of piety between chapel and salon. A Lucca factor who arrived each new moon with a ledger wrapped like a reliquary. Two poets who left owing more than they were paid. Three matrons who repeated her phrasing in market two days later as if it had always been theirs.
By late light he knew the wager inside Dania’s thread: Donata does not pray; she presents. She collects occasions the way a broker collects fees. She is moved by reach, by visible usefulness, by the sight of her name curing the valley’s worry in public. Her devotions are carved, not felt; her mercy is measured in who attends.
So he set his little knots. A word at the council about donors and the opening of deeper shelves when gifts are great. A soft suggestion to the librarian’s boy that parchment with old seals reads sweeter when a lady is present. A casual remark to Maestro Giusto that certain ruins take marble handsomely, and a lady of means could fix a wall to suit a legend. A quiet purchase of rosemary and saffron at the apothecary, because her kitchen notes what enters her gate and tells her what the town now values.
He left no edge that could cut him. Every phrase wore another purpose: roads, safety, rents, prayers for the dead men on the north road. Under it ran the plan’s true grammar: give Donata a miracle to sponsor, a ruin to restore, a seal to parade, a visit to demand. Let her do the pulling. Let the parchment travel in her noise.
Dusk found him slipping back under the arch while the Brotherhood’s candles gathered at corners and the shutters closed with the careful softness fear learns. The town had no attention to spare for his errand; the bandit camp drank it all. He carried home what he needed: the outline of Donata’s will, the channels her coin swims in, the points where a letter, a legend, and a lady might be braided into one rope strong enough to draw Aurelia through her doors—on Valiponte’s terms, not the Marchese’s. Not certainty, not yet, but enough to tighten his mind around a sketch: each step checked, each knot tested for slip.
At the square he paused. The Brotherhood had thickened since his first ride—helping the poor with bread and small alms, lighted faces in gathering dark, and still… something in their cadence did not set true. He marked the shadow of it, then let it pass; the plan in his head demanded both hands.
Night took the streets. The town gate was down. Guardians doubled at their posts. Curfew laid its paw over Valiponte and the town put on the heaviest armor a mountain place can manage for itself. I approved the fastening of every buckle with a blink.
Luca kept to Deonardo’s stirrup, steady as a hinge. They turned to Casa Basili and asked, with the proper courtesy, for a chamber. Ser Basili met them at the threshold with his practiced humbleness, warm as bread, brief as sense. A steward led on with a lamp; Filippa walked beside, the tilt of her keys telling the truth of who keeps a house alive. The guest rooms lay above her office—over the ledgers, over the storerooms she husbands like orchards in winter. Three maids had already flown ahead: beds turned, sheets aired, braziers breathing a steady glow into both chambers. I leapt to the door lintel and made a small inspection. Acceptable. Warm enough for fur and plans.
At the threshold, Filippa paused with Deonardo, voice lowered so the lamp would not hear too much.
“Curfew makes the town feel smaller,” Filippa said. “But the fear is large. The council pretends calm. The markets pretend habit. I am… less skilled at pretending.”
Deonardo’s mouth softened. “Pretending is a poor guard. Locks and ledgers do more.”
She glanced at his cloak, then away. “Romé stood very still today. When men like him stand still, it feels like the air is choosing a weather. I don’t know whether to be comforted or afraid.”
“Both,” Deonardo said. “Comfort is honest when it knows what it fears.”
A breath; the barest curve of her lips.
Filippa’s eyes lifted to his. Worry, gratitude, and a small, uninvited warmth shared the same chair. “You see paths under words. It… helps.”
“It helps me that you keep the doors that words must pass,” he said. “Sleep a little, Lady Filippa. The ledgers will run straighter if you do.”
She bowed, not courtly—household true. “If you need anything, call for me. Even if the bell says otherwise.”
“The bell and I are old colleagues,” Deonardo said. “We negotiate.”
That won her a clearer breath. She stepped back, keys settling, and the steward drew the door to with the gentleness that fear teaches. I curled on the warmed sill. Outside, Valiponte held its armor; the waxing crescent hung silver-bright, sinking behind the dark line of the roofs, and the town was locked—no matter where the danger lay.




📖November 28, 1264 Friday
Messengers return 🕯♛♡🐾 cloudy, first quarter moon, behind clouds

Friday at dusk, Matteo and Gianno rode back under the arch, frost thin on their cloaks. They came back in the hour when the light goes thin. Matteo rode first, shoulders easy, pace exact; Gianno kept his flank, eyes measuring the wall-walk before the gate-keepers had finished lifting the bar. The hinges breathed; the arch took them in. Ferrazo was already there with two men. Lorenzo crossed the square without cloak or ceremony. No trumpet, no shout—only work returning to its bench.
At Casa Basili the table filled as before: Ser Basili at the head, Lorenzo beside him, Ferrazo broad and bandaged, Deonardo present but silent, two elders of the council, and Romé standing with one palm on the map-board. I took the beam, heat in my belly, ears forward.
Matteo set the sealed parchment on the board. “From their captain,” he said. “They will come to parley on Monday, the first of December.”
Then he gave the room what it needed, plain and in order.
“Numbers,” Matteo said.
 “Fires counted, ten to twelve. Four men to a fire, not all mounted. Call it forty to fifty under arms. Mostly on foot. Leather jacks on many backs, a handful of old mail shirts, a few round shields but sound enough. Spears and short arming swords in plenty, three crossbows of decent pull with tired strings, knives on every belt. No lances. No heavy horse. Half their mounts newly shod, the rest loose and ribby, hobbled close to water.”
“Their camp,” he went on. He tapped the map.
 “Below the east beech stand, above the shale run. Low tents, fires banked, latrines cut with order enough, but not the old kind of drill. You can see they were gathered in haste. Men settle late, quarrel over bread, stand easy on watch. Winter presses them—cloaks thin, no spare canvas, fodder short. The ground is good, but their supply is not.”
“The captain himself is tall, mail under a dark cloak. Quiet, not cruel. Drinks only after full dark. Spoke little, looked long. He took our measure with his own eyes.”

Gianno added, his voice rough but sure.
“Ground holds them. Water close, smoke hidden under beech. One goat path east that flanks the camp. If they move, it will be toward the ford at the mill. Their men watch the sky more than the brush—soldiers’ habit, not brigands’.”
Matteo nodded once, then gave the words that had brought them home.
“Their captain accepts the terms,” he said. “His men will come on Monday. He agrees to stand outside the walls.”
Silence held a moment, then—
“Outside the gate, on the slope,” Ferrazo said. “I will read their feet.”
Ser Basili looked to the room. “Then we prepare our side.”
He did not waste words. “Two hundred crossbowmen on the wall-walk, by watches, not bunched. Strings waxed tonight. Bolts counted and issued, twenty to a man. Old mail comes down from the chest and is fitted where it fits. The gate—hinges oiled, bar manned, straw lifted from the threshold. The steward will post buckets at intervals. We will not burn our own square if the worst comes.”
“Signals,” Ferrazo said. “Bell codes: one stroke for sighting, three slow for parley begun, five fast for treachery. No man touches the bell but the bailiff.”
“I will draft the terms,” Lorenzo said. “Two copies ready by Lauds. One for the captain’s hand, one for our library. Wages named by the month, bread by the week, quarter kept outside the walls.”
“Obedience required. Abuses fined. Exile written before blood,” Deonardo added, voice even, as if laying a final line of ink across the page.
“Witnesses,” an elder said. “Two guild-masters at the parley’s edge, to see and later to say.”
Then all eyes, as they do, slid to Romé. He had listened without hurry, thumb resting against the map’s edge. When he spoke, it was simply:
“On Monday I will stand as moderator. Captain Ferrazo will speak for the town in matters of keeping the road. Ser Basili and the council will hold the purse and the pledge. I will keep the ground even and the hour clean.”
It was a small sentence that meant more than it said. Everyone in the room knew he was the center of the shape—steel and reading both—but he set himself where the town could learn to bear its own weight.
He let the valley do the valley’s work. If he did it all, it would end when he rode away. If they did it themselves, it would endure.
He did not explain any of this, of course. Romé cared in the same way Dania did: by setting the work right with his own hands, and leaving the reasons between himself and whatever watched from the rafters.


📖November 29, 1264 Saturday
Cacifer’s stitched diary — Interpage: Matteo🕯♛♡🐾 first quarter moon
Saturday, the day after he returned from carrying messages to the bandit company, Matteo brought breakfast to Deonardo as if nothing in the world had been dangerous yesterday. The tray was simple—bread still warm, soft cheese, a pear, rosemary water—and he set it down with the unshowy care of a man who knows which surfaces hold worry and which should not. He greeted his master in the court’s grammar, eyes lifting only when Deonardo’s small nod released him. Then he stood the way he always stands at the start of a day: present, readable, and ready to make the room breathe.
Matteo is around forty, older in the face than Deonardo looks, but strong where it matters: shoulders that can carry bad news without making a parade of it; hands that remember every latch and secret in a house; a mind quick enough to untangle a steward’s half-truth while letting the steward keep his dignity. He has served Deonardo more than fifteen years. Before that he belonged to another noble household that went down hard—wages late, loyalty spent thin, too many servants learning how hunger is spelled in accounts. In that year Deonardo was just becoming Donna Aurelia’s close counsel. He met Matteo by chance, saw the clean line of potential under the fatigue, and hired him on the spot. Deonardo gave Matteo a place. Matteo gave Deonardo what a city cannot function without: loyalty that doesn’t need an audience.
He is not married. He has lovers on the road; now and then, he buys an evening without shame. Deonardo knows and leaves him room to be a person, not a tool—space to travel, to return without confession. That is part of why Matteo’s service is light in the hand and heavy in the scale. His happiness is not confused with permission; it is braided with usefulness. He wakes ahead of the day, mends what should not be seen to break, and notices the small things that keep dignity intact: a cloak brushed before it is asked for; a seal warmed so wax takes clean; a door closed at the right moment so a man may be two breaths softer than the city allows.
Over years, Matteo has seen what others would name an evil and flee. He knows his master has a woman’s form as well as a man’s. The Church would call it darkness. Matteo calls it none of their business. He has watched Deonardo govern with care, spend coin like medicine, and bend punishment away from cruelty. He knows what evil is—waste, spectacle, the kind of fear that fattens on other people’s hunger—and he has never found it in his master. When the shift comes, Matteo does what protection looks like in a good house: he moves the mirror to a kinder light; he adjusts the angle of the latch; he positions himself outside the door as if he were guarding a ledger, not a body. If anyone asks, nothing unusual happened, and the room was always exactly as you see it now.
Between them runs a clean current of regard that ignores theater and remembers breakfast. Deonardo trusts Matteo with the things that cannot go wrong: the first page of the day, the last key at night, the message that must arrive with its dignity unbroken. Matteo trusts Deonardo to keep his work meaningful and his person intact. This is not romance; it is the kind of attachment that builds a city quietly. On this Saturday, after danger and before council, it looked like bread set down without clatter, a pear chosen for ripeness, and a servant standing at ease until the smallest nod said: go on, I am ready.
Snow coda — returning to Saturday sunset
Outside this page, the world has been snow all day. San Michele and Valdiponte lie muffled; paths vanish; bells ring shorter in the thick air. At the bell of Vespers, I stood on the low wall of the healer’s garden. The forest beyond broke at the blue edge of day: the sun sat low and every shadow grew long. Fresh snow silenced the ground; trunks and branches wore thick white, bending slightly under the weight. The drifts glowed, the shadows breathed, and far between the trees a last ember of orange held at the horizon. Windows answered with small rectangles of fire. I closed Matteo’s page, shook the snow from my whiskers, and stepped back into the evening, where voices gathered and the storm made a room of its own.



📖November 30, 1264 Sunday
Cesare➻ the lover when he was seventeen 🕯♛♡🐾
Sunday woke into a space between danger and decision — tomorrow the bandit company would come. Outside the walls the valley wore its fear like a tight cloak; inside the healer’s garden the air stayed disciplined and calm, as if thyme and rosemary could teach breath how to behave. The mortar had cooled on the bench, the braziers were banked, and even the bees, those little clerks of daylight, had shut their ledgers. I walked the sill like an inspector and pronounced us briefly safe.
Romé had drifted into a day-sleep on the pallet in the treatment room. It was the kind of sleep with the door left ajar, where past and present visit without knocking. Reed lay in shadow near the wall, not sulking, only resting the way good steel rests: awake to the idea of work and saving its edge. Deonardo was elsewhere about his arrangements; otherwise he would have scolded Romé for sleeping in a space meant for the healer’s work.
Late afternoon thinned the light to honey through the shutters — warm, dim, unruled by Deonardo’s disciplines. I curled beside Romé, half listening to him breathe. He dreamed of Cesare — the lover gone ten years, a name he no longer said even to himself. Waking at the hinge of dream and day, he reached without looking and caught the nearest certainty — Dania’s dress, perhaps her wrist, as she drifted past.
He didn’t know. He only said, low and immediate, “Sit with me.”
 She did. No one rejected Romé when he asked like that. Or perhaps she would never have rejected him anyway.
His eyes stayed closed. “How did you look when you were seventeen?” he asked, voice wandering. “I wanted to meet you there. I wanted to meet you when we were both seventeen.” 
A pause, a breath; a smile shaped itself across his mouth as if he could see her younger face gathering out of the dusk.
“Proud,” he murmured. 
“Hard to touch. Men could only watch you secretly, because if their sight reached you boldly they’d freeze. Afraid of you, wanting you, pulled toward you all at once.” He laughed, eyes still shut, laughing at the girl he saw behind her present calm. “I would definitely have talked to you.”
“You would have been cursed,” Dania said lightly. 
Even I did not expect her to answer. 
Her tone was clear; the thread beneath it was iron. “Cursed, and given nothing in return.”
Romé opened his eyes halfway, dream still on him like a veil.
“I knew I would be cursed,” he said. “It didn’t frighten me. I would have fallen in love with you. I would have fallen for you so much that I would have died if I could not talk to you.” The words came from the deep place where his voice was most honest. “And you would have loved me. Back then I was a creature made to be loved.”
Dania smiled—faint, tender, with that slight glow of bleeding one sees when an old scar warms. “I wasn’t a human yet,” she said. An answer and not an answer.
He didn’t ask what she meant. He only kept talking, still half inside the past. 
“I had my first man at seventeen,” he said. “I’d already tasted love with many women, but he was the first love where I felt myself.” His hand loosened from her sleeve; she stayed.
“I don’t think about him anymore,” Romé went on, which meant he did, just not where speech could reach. “Sometimes the winter air smelled like him. Bergamot and ink. It passed.” He blinked, then looked fully at Dania at last. “We walked by at the wrong moment.”
“At the right one,” she said.
Silence laid itself down between them, just enough to hear the linen breathe. Dania sat beside Romé, her side against his ribs, and his arm could have gone around her waist easily, but nobody reached to touch more. I curled deeper and felt their breathing find the same length. 
“He played oud the way I had never heard before, like someone who had once been silent too long,” he said, voice low, fading into the middle distance. “And now didn’t care if the words came out wrong—as long as they came.”
Dania listened to him the way Romé had listened to Cesare’s oud.
“That night changed me,” Romé went on. “Not because I fell in love—but because I realized music could ache and not explain itself.” He turned then, his chest leaning into Dania’s dress and the blanket between them. 
The room held the rest. Outside, a dove started and settled again. The light climbed the plaster and faded back. Dania remained exactly where he had asked her to be—near, unworried by the ghosts he had invited in. Romé breathed once, more deeply, as if the melody had finally found its end inside him. Then he closed his eyes, and for a moment the palace garden returned: the lemon shade, the oud’s low question, a man who did not move when the music stopped.
“Stay until I cross,” he whispered.

🌿 Before the Ashes: The Love of Romé and Cesare 1254-1255
 --a full account, interpages stitched by Cacifer

It was early summer in Vallescura, the year Romé turned seventeen. 
He had grown into a beauty that did not ask permission to be read—long lines without heaviness, cheekbones drawn like a compass stroke, a mouth made for laughter or vows and unwilling to choose. His dress turned talk into weather: an indigo doublet cut narrow at the shoulder and clean at the waist, white linen at the throat like a blade, silk laces the color of bruised figs, soft boots that made no sound on stone, a thin ribbon at the wrist instead of a signet. 
Laundresses whispered that he looked like a saint painted by a heretic; magistrates’ wives called him dangerous with a smile they did not explain. Tailors copied his sleeves and missed the point. Boys in the yard hated him for an afternoon and then tried his stance in the mirror; old captains went quiet, unsure whether to warn him or ask for lessons. 
He did not tilt toward masculine or feminine; he held the middle as if it were a kingdom and wore it like law. The courtyard stones still held spring’s chill, but the air had begun to smell of herbs and lime-wash and vines in bloom.

Cesare arrived with a traveling court from Lombardy — a musician hired to entertain Lord Corrado’s northern guests. He was older. Twenty-seven.
Not noble-born, but dressed in noble quiet whose grace had watched kings lose their crowns.
His scent: bergamot, faint ink, and pine smoke.
Romé saw him first in the chapel gallery — not during a concert, but in silence.
Cesare was sitting alone, reading. The light through the stained glass cast red across his throat.
Romé did not speak.
He just passed.
But Cesare looked up — and followed him with his eyes.
The second time Romé saw Cesare, it was in the inner garden of Lord della Rocca’s palace — a quiet cloistered space framed by vine-covered columns, warm fig trees, and flowering herbs. The air held the late heat of the day, thick with the scent of crushed thyme, sun-warmed stone, and old lemons splitting in the branches. Somewhere behind the hedges, a peacock muttered softly, but otherwise, the garden was silent. Even the fountain in the center dripped as if half-asleep, casting slow ripples that caught the gold light and fractured it.
Romé hadn’t meant to find him. He had slipped away from a feast of Corrado’s swordsmen and followed the sound of music he didn’t yet realize he was following. There was no audience. No host. Just Cesare, sitting alone on a stone bench under the pergola, his back straight but at ease, head slightly tilted in concentration. He didn’t see Romé — or at least, he made no sign of it.
The oud rested across his lap, dark and weathered, its shape curved like a teardrop held sideways. Romé had seen the instrument before — in banquet halls, in the hands of respectable musicians hired for their precision. He’d even handled one once in the Orellana storerooms, among the dusty relics of past crusades and spice routes, where strange instruments, parchments, and silks were stored without order. The oud, to him, had always belonged to the realm of decorum — courtly, elevated, a polite accompaniment to almonds and wine.
But what Cesare did with it was different.
He didn’t strum. He summoned. The sound curved — not as melody, but as question. His fingers moved as if deciphering a memory, or tracing words written on the skin of someone long lost.
The oud’s voice under Cesare’s hand was raw — not in volume, but in intimacy.
It made Romé feel as if he had stepped into someone else’s heart by accident — and forgotten how to step back out.
Romé stood frozen in the garden, watching the shapes of sound move through Cesare’s body — his hands, his shoulders, the low breath he held between each pluck.
He didn’t know how long he stood there. The golden light was fading by the time Cesare’s fingers stilled. He never looked up. Or if he did, it was only after Romé had already turned, walked away, heart pounding like he’d overheard something too private to name.
That night, Romé wrote a single line in ink on a fold of parchment:
 “I dreamed of water that knew my name.”
 He left it on Cesare’s music stand the next morning — unsigned.
 And the morning after, it returned.
 Same parchment. Same fold. One new line, beneath his own:
 “Then I am the river. Say it again.”

First Kiss with Cesare — The Exam of Swordship at Lord Corrado della Rocca’s Palace
 When Romé and Aureliano were elected to the first range of the Lord’s elites
The ring was chalked on packed sand beneath the east loggia, where the palace turned its face toward the terraces and the far, pale hills. Morning kept its breath cool. Banners hung still. Lord Corrado’s captains stood in a half-circle, iron quiet, while clerks scratched names and odds.Young nobles came dressed like parade; the smiths’ sons came dressed like work. Romé and Aureliano stood side by side—Aureliano already the taller by a head, Romé all clean line and listening poise.
“Three passes each, clean marks only,” called Maestro Ruggieri, the master of the ring. “Speed finds the door. Measure walks through it.”
They began with the dross—boys who swung like bells, men who mistook noise for edge. Aureliano moved first, efficient as a mill at harvest, turning heavy wrists aside and planting light, correct touches at cuff and collar. Romé watched, learned, and saved his heat. When his name was called, he stepped into the chalk with a stillness that made the loud ones uneasy.
The city swordsman came late to the exam—velvet on his shoulders, elegant, too handsome for his own temper. He smiled easily at the onlookers and bowed deeply to the captains. 
“If I win,” he called across the ring, between serious and smiling, “I claim a kiss from the Orellana peacock.”
A ripple ran the circle. 
Lord Corrado said nothing. Maestro Ruggieri only lifted his two fingers—the sign that meant, Proceed.
Steel met without shriek. The first pass, Romé let him speak: a flourish, a feint, the pretty cut that looks like victory from a distance. The second pass, Romé answered in the language of wrists. The third, he stepped off the man’s line and found the seam between pride and breath. On the fourth, he placed the cleanest touch of the morning at the inside of the bracer—so light the dust rose after the point left.
Silence, then breath. Maestro Ruggieri’s hand cut the air. “Mark to Orellana.”
The crowd’s noise swelled—bets changed hands, names repeated themselves. Before the mockery could grow teeth, Romé crossed the sand, took the city swordsman’s jaw gently in his hand, and kissed him—brief, exact, like a signature on good parchment. Tribute to good steel.
Aureliano’s mouth broke into a grin he tried to hide. One captain coughed a laugh into his glove. Lord Corrado did not move, but a line eased at his eyes.
That should have been the end of it. But at the edge of the loggia, in the lintel’s shade, a man had been standing since dawn with an instrument case at his feet. He had the kind of quiet that makes a room tell the truth. 
Cesare. The court musician hired for the northern guests, bergamot and ink somewhere in the turn of his sleeve. He did not applaud. He only watched Romé step back from the kiss and return to stillness, and in that watching something passed between them that did not belong to the ring.
The exam continued. Aureliano met the captain’s best and sent him away marked at the cuff and cheek—two lines, neat as a ledger. Romé took the final bout against a heavier man and wrote three quick answers where pride keeps its buttons. At noon, names were called under the arch: “Aureliano di Orellana.” “Romualdo di Orellana.” Chosen for the first range of Corrado’s elites. The captains saluted; the clerks closed their books with a thump that felt like winter promised and paid.
Nightfall. When the ring emptied and the sand lay raked, Romé cut through the back colonnade to breathe the lemon shade. The torch-brackets still smelled of last night’s oil. Cesare was there, leaning one shoulder to the cool stone, oud in hand but silent.
“I heard you kissed your opponent,” he said, no judgment in it—only the faint curl of a smile.
“I did,” Romé answered. “He asked for it in victory. He fought well in defeat.”
“You give alms in kisses?” Cesare murmured.
“I pay debts,” Romé said, and, for the first time that day, let his body go still.
They stood with the torchlight flickering against the columns, the garden a dark square beyond. Cesare set the oud down carefully on the bench. He did not reach for Romé. He only waited, eyes steady, as if listening for a note that had not yet been played.
Romé stepped forward. He had claimed many things in his life—bodies, rooms, silence. This felt different. He lifted a hand to Cesare’s collarbone, rested it there—not a demand, a measure. Cesare’s breath answered, a small change Romé felt in his palm.
He leaned in and kissed him.
It was not like the kiss in the courtyard. It was slow and exact, given and received at once. Cesare did not move first, but when he did, it was with a care that unraveled Romé’s practiced certainty—the kind of answer that turns a statement into speech, then conversation. The world narrowed to the taste of wine left on Romé’s mouth, the heat of Cesare’s cheek, the lamp’s flutter. When they parted, neither laughed, neither spoke. They only looked—two men who had seen the same river and recognized it.
“Now you owe me nothing,” Cesare said softly.
Romé shook his head. “Now I owe you everything I haven’t said yet.”



🌒 Their First Night
Vallescura, Summer 1254
It was late — past the final bell, past supper, past even the hush that settled over the palace long after servants crept away. A soft wind moved through the carved shutters, and the air smelled faintly of lemon blossom and burned oil. Cesare’s chamber was high in the north wing, meant for traveling artists and minor guests — but that night, it might as well have been the center of the world.
Romé had come late, by no arrangement. Just silence. Just the understanding of all the days before — of glances, of notes folded into each other’s hands, of the kiss in the ivy window that neither of them forgot.
When he entered, Cesare had not lit the room fully. Only a small lamp glowed in the corner, and beside it, the oud lay resting on a cloth. Only the faint scent of pine smoke clung to Cesare’s tunic, and Romé — heart loud, mouth silent — touched the edge of the desk, as if unsure whether the world would shift if he let go.
Cesare stood with his back to the lamp.
He looked at Romé the way one looks at a candle just before blowing it out —
He said only this:  “If you stay, I will not lie to you.”
Romé had known many bodies.
He had learned early how to enchant —
how to take pleasure in the tension of being seen, how to control the pace, how to turn desire into rhythm, gesture, breath.
He had claimed many women before — often with charm, sometimes with hunger, always with confidence.
But Cesare did not undress him.
He watched.
And that was what undid Romé.
Because in that stillness, in the absence of demand, Romé realized — he wanted to be known, not just enjoyed. He wanted to bring this man pleasure, not through artifice, but with full attention.
So he slowed.
The summer evening wind slipped into the small chamber and thinned the little lamp. Romé touched him with instinct, yes, but also with something new—a sensing, an echo of what Cesare had taught him with the oud.
That rhythm was not just in motion, but in listening.
Cesare lay back on the linen, never commanding, but never letting his eyes leave Romé.
Romé kissed his chest —
and waited. Let his hands learn the map.
He traced Cesare’s collarbone, felt the slight tremble under his palm. His own breath deepened.
He lowered himself slowly, listening for every shift of muscle, every ripple of response.
When he touched Cesare’s thighs, when he circled with his tongue, when he finally entered the rhythm they had both been waiting for — it was the first time Romé made someone else tremble and felt that trembling echo back in his own chest, as belonging.
And that night, for the first time in his life, he fell asleep in someone’s arms not as a conquest, but as himself.

🕊 A Later Afternoon
Mid-Summer 1254, Vallescura
It was the fourth week of their secret exchange.
Afternoons had become their time — when nobles slept, when corridors cooled, when excuses to disappear grew easier.
They met in the abandoned top floor of the old dovecote tower, where only pigeons and heat ghosts gathered.
No one came. No one watched.
That afternoon, Cesare brought a folded cloth and set it on the floor. He unpacked figs, bread, and a flask of sour cherry wine. Romé came barefoot, wearing a tunic too fine for dust.
He sat cross-legged beside Cesare, leaned back on his elbows, and asked, for the first time:
“How did you know? That I wouldn’t run?”
Cesare replied without looking at him:
“I didn’t.”
Romé laughed — a true laugh, the kind that didn’t need to shield itself.
And Cesare smiled, slowly, before reaching across the cloth to touch the side of Romé’s neck — not a caress, but a gesture of knowing.
That day, they didn’t make love. They talked — about music, about the ache of not belonging, about Romé’s brothers, about how Cesare once wept while tuning his oud, and no one in the court noticed.
Romé rested his head in Cesare’s lap and closed his eyes.
“You don’t always have to be desired,” Cesare said, stroking his temple.
“You are already beautiful. Just like this.”
Romé didn’t answer.
He only turned slightly, cheek to thigh, and let the silence cover him like skin.


✉️ The Last Letter from Cesare (Summer, 1255)
Delivered by a traveling scribe. The seal unmarked, the paper faintly scented with lavender and ink.
My beloved R.,
I did not know the cost of beauty until I met you —
 nor how much silence could grow inside a man who dares to love without promise.
I left not because I stopped loving you,
 but because I knew I would begin to measure myself against your gaze.
 And your gaze deserves no mirror.
You were the poem I could not finish.
 Not because I lacked words —
 but because you kept moving.
If I am ever brave again,
 I will find you.
 Not as a thief in night,
 but as a man who has finally become enough to deserve you.
Until then,
 take this — a stanza unfinished,
 a kiss unreturned,
 a life rewritten without your name at the start.
Yours — still,
 C.
Romé read it once.
 Then burned it.
 But kept the ashes in a little shell,
 tucked into a velvet pouch
 he wore only on winter rides.


📖December 1, 1264 Monday
Rome meets the bandits — the slope below Valiponte 🕯♛♡🐾 cloudy
They came at dusk, forty or so—what was left after the earlier cutting and their own culling. Pieces of better armor sat wrong on thinner men; two crossbows with tired strings; most on foot, a few on poor horses. Above them, hidden on the walkpath along the town wall, about two hundred townsmen waited with cranked crossbows, bolts hooded, orders set: only on signal.
On the slope stood Ferrazo with eight of his best, Brother Gianno, Matteo, and Romé—and me, of course, borrowing a horse’s eyesight for the occasion, whiskers full of leather-smell and snow, tail doing the serious-business flick.
Romé introduced himself as moderator, not lord. He did not posture. He unfastened his cloak to let the crest of Lord Corrado’s Twelve show on the cuirass. The band’s noise thinned. Respect rustled; doubt did too—too fine, too clean, is it real? He felt the skepticism and let it pass. Words would not cure that; conduct would.
Ferrazo spoke terms, voice flat as a contract: winter billets for 20–25 men, paid weekly at the abbey gate; work is road-days, snow-cut, bridge watch; no arms in villages, no recruits, no night fires near mills; break a term and you forfeit pay and are hunted out. The rest to leave the valley at once—advised route southwest toward Florence, where captains were hiring for spring. The split was intentional; Romé meant to halve the animal, keep the workable muscle, starve the appetite.
The captain listened, asked tidy questions about pay, rations, rope, billets. His face never showed heat. But his eyes counted the town gate, watched the angle of the portcullis groove, measured the heartbeat between wall-walk shadows. Gianno’s shoulders went still. Ferrazo’s jaw tightened. Romé turned a palm—I see it.
“Test us,” the captain said at last, smiling. “If we pass, we take your coin. If not, we take your bread.”
“Fair,” Ferrazo answered, already moving the words toward a shape that would keep blood off snow.
But steel winked in the front rank—six men easing blades out of habit more than need, a fractional shift of feet toward the gate that gave the thought away. The captain let his left hand fall in the smallest call Romé had seen all year.
Romé moved.
No flourish. No warning. The Reed came clear with the whisper of silk and wrote a single, straight decision across the captain’s throat—clean, shallow-deep, final. In the same breath, the back-edge kissed the sword-hands of the five nearest—tendons parted, hilts fell with the hands still wrapped around them. The sixth flinched too slow and kept his hand by an act of luck he would later call providence.
Silence found the slope like snow.
On the wall, two hundred crossbow strings did not release. Ferrazo had not raised his hand.
Romé stepped back one pace so the body could fall without touching him. He let the blade rest low, point toward earth, voice even.
“No one moves.”
He looked at the second-in-command—the one whose eyes had gone wide rather than bright. “You were about to break parley on a gate. That is banditry. It ends here.”
No one answered.
He went on, since no one else dared to speak: “The contract stands—maximum for twenty. Captain Ferrazo will select. Lay down your arms in a heap. Bind your wounded. We will bind them better.”
He nodded to Matteo and Gianno; both stepped forward with cloth and cord.
“You will take billets in the abbey outer yard, under watch. You will be fed, paid, and worked thin. You will not enter villages armed. You will not touch a boy, a girl, a granary, a rope that is not handed to you.”
He turned his head slightly toward the rest. “The others walk south tonight. Each gets bread and salt, a turn by the brazier, then out with a monk to the ridge—no torches near our mills. If you come back with steel for our gates, the wall will speak first.” Above, the walkpath scraped softly as men shifted, reminded.
He let them see the dead man once more, not to humiliate, but to show where decisions end. “Contract or road. There is no third thing—unless you insist on making one.”
Something in the knot gave. A blade clinked into the snow, then another. The second-in-command raised both empty hands to chest height. “Contract,” he said, hoarse. “We’ll take terms.”
Romé nodded once and turned to Ferrazo. 
The old captain paced the line like a man choosing rope, not trophies, tapping a shoulder here, a breastplate there. 
“You. You. Not you.” The knot split cleanly: those he could use stepped left; the rest drifted right, eyes on the snow. Names came clumsy and quick, half in fear. Ferrazo set each down on a scrap and sorted them as he wrote—ropes, shovels, watch—work before sleep. 
Brother Gianno moved among them binding hands and ribs without judgment, as if this were simply what evening required. Matteo knelt where blood made the ground slick, pressed a cloth to a stump, and tied it smooth—eyes steady, the practiced stillness of a servant who becomes furniture while saving a life.
When it was done, Romé wiped the Reed, sheathed it, and finally looked up toward the walkpath. A single, small nod. The shadowed line of crossbows relaxed like a held breath released.
He said one more thing to the soilders who were chosen by Ferrazo: “If any of your former brothers trouble our villages, you will be the first to stop them. That is part of your pay.”
They understood. A few stiffened as if at insult; most as at an order that made the world simple again.
The column split under the first stars. The contracted ones turned toward the abbey’s side gate under escort; the rest filed down the road with bread, salt, and a monk’s lamp to see them to the ridge. The captain’s body stayed where it fell, given back to the ground that had wanted him.
No cheering. No trophies. Just the sound of rope being cut from a coil and the river talking to itself below the bridge, as if the valley had decided to keep its own hour after all.

📖December 2, 1264 Tuesday
The Angle Where a Hand Could Find Me  🕯♛♡🐾 waxing gibbous 
Ferrazo began early, boots already salted, voice flat as a contract. He chose sixteen out of what was left, signed them to clean terms, and sent the rest out of the gate with coin enough to make leaving cheaper than trouble. The town breathed—lighter by a finger’s width, still cautious. At the well the gossip braided itself: the evil beauty, they said, arrived like a rumor and solved like a blade. No one saw how he tended the edges; they retold only the instant—how fast the captain’s throat opened, how five hands hit the snow before minds caught up. The rest—the mercy, the rooms made, the bread set aside—moved quieter than talk does.
Deonardo woke at Basili’s house to the smell of ash and clean linen. Luca had gone at first pass over the road, letters sealed and warm from Deonardo’s hand tucked into his satchel—the ones that would carry Dania’s plan further down the valley, where coin becomes grain and grain becomes time. The house held its particular order: doors that close without pronouncing it, floors that remember to be quiet.
On the stair he heard Filippa’s ledger-hand already at work. Not the thin scratch of vanity accounts—Filippa never plays at numbers—but the even pace of someone balancing a town’s breath against winter. Without thinking, he knocked.
“Come.”
Deonardo paused in Filippa’s doorway. The ledger lay open; her quill moved with that even, fearless tempo Basili women inherit.
“You’re early,” she said, not looking up yet.
“I sleep like a clerk in winter,” he answered, stepping in. “Long enough to keep the sums straight.”
She glanced up. “The town is breathing. A little.”
“Ferrazo made clean choices,” he said. “Sixteen on paper, the rest sent past the gate with coin enough to prefer leaving.”
Filippa nodded, then hesitated—just the smallest catch at the mouth. “They’re calling him the evil beauty,” she said, too lightly. “At the well. How fast it happened. Five hands before anyone understood. A throat. I wasn’t there.”
“You were carrying the town,” he said.
She set the quill down. “I don’t understand him,” she admitted, eyes on the ledger margins. “I did, once. Or I thought I did. Now he feels—” Her fingers searched for a word, found none. “I hear danger in every sentence. And I remember kindness. It doesn’t add.”
“It can,” Deonardo said. He stayed in the doorway, scholar’s mantle uncreased, voice low. “Sometimes the cleanest mercy has a blade tucked under it. He used the edge so the room would not need to bleed all day.”
Filippa’s gaze lifted, steady and a little tired. “He left me,” she said simply. “Before the line. I keep thinking there was something wrong with me, or with him. Now everyone says he’s—” She exhaled. “I don’t want the town to decide what I remember.”
Deonardo crossed only as far as the corner of her table, stopping where a guest stops. “Your memory is sound,” he said. “He is not a riddle you failed to solve. He is a man practicing a grammar the town hasn’t learned yet. That can feel like danger.”
Her hand went to the ledger’s edge, gripping it as if it were a rail. “And to you?”
“To me,” he said, “he reads as someone who refuses waste. Of coin. Of shame.” He let a breath pass. “You were not refused, Filippa. You were spared a wound the day did not require.”
She leaned a fraction toward his steadiness without thinking, then caught herself and smiled—Basili’s wry, practical smile. “You make it sound almost reasonable.”
“It is only weather,” he said, softer. “Let it pass through. Keep your hands on what feeds people. The town will learn his measure by work, not gossip.”
Filippa nodded, some tightness leaving her shoulders. “Thank you.”
He inclined his head. “If you need a signature, send for me. Otherwise, I’m due at the scriptorium—someone must look like a scholar today.”
“Deonardo,” she said as he turned. “You see more than you say.”
He allowed the smallest warmth. “And you carry more than you name.” He tapped the ledger once with two fingers. “Bread first.”
“Bread first,” she echoed.
He stepped back into the corridor, leaving her with the ink’s small weather and a room that felt steadier than before.



Before the bell for None, Deonardo returned to the healer’s garden and found the world refusing drama. Nothing, it said—and meant mercy.
Dania stood at the hearth, stirring the poor soup she calls lunch, steam lifting rosemary and onion into a light that had not learned fear. Romé eased into the court with his usual clumsy grace, the book already open, reading as if pages could keep weather in its proper place. A cup on the table. A knife laid true. The small sounds of a house that has decided to be gentle.
That was when Deonardo felt the hinge inside him turn.
His chest went first—something unclenched beneath the sternum, and breath dropped lower, fuller.
Sight turned tactile. Watching Romé became a kind of touch: the plane of a cheekbone; the way a palm settled on wood; the small economy at the mouth when he decided where to stop reading. Each detail arrived with weight.
The pulse changed its work. It no longer hammered against restraint; it called to contact. Heat gathered low and lengthwise—abdomen to throat—asking for nearness without disguise. Not hunger that scrapes, but warmth that reaches. 

Touch learned a language he could not unsay. The brief custody of Romé’s wrist while passing a cup—small, exact gestures carrying the whole truth inside them: stay. When Romé’s voice went quiet, Deonardo’s palms warmed; when Romé stepped closer, the back of Deonardo’s neck answered as if named. 
He did not understand why his body moved this way, and at the same time he understood perfectly.
In the last days Deonardo was busy—packets, sums, riders in and out—he hadn’t been close to Romé or Dania. But he kept catching the edges of Romé’s work—more than the town saw: he kept every man’s dignity in play and put the knife back quietly. And just so, in the deepest place where Deonardo’s heart beats, Romé had already touched him.
Dania lifted the spoon, tasted, added salt. It will do, she said in her steady manner, and the room agreed. Romé turned a page and looked up just long enough to mark Deonardo’s place in the air. The house kept being itself.
📖December 3, 1264 
📖 an old letter— the perfect maricle🕯♛♡🐾 full moon

It could have been an oil painting — the kind Rembrandt would paint three centuries from now, more shadow than light, the flame of a single candle folded into its own haze.
 Romé sat at the desk, his black hair loose enough for a few strands to catch the glint. The shadow of his lashes drew fine lines down his cheek, moving when his eyes shifted. His jacket was dark wool, cut close at the waist, the collar open to show a fine chain lying in the hollow of his throat. One gold drop earring trembled when he turned.
 The healer stood beside him, her black dress taking the light without giving it back, her hair swept back to reveal the clean geometry of her face. She was still, but the air leaned toward her. Between them, lamp smoke mingled with the sharper scent of rosemary and burnt orange peel from the brazier.
I felt Deo’s chest go tight beside me. Two rivers, I thought — running so near their waters touched, yet each pulling toward its own course. Watching them was like standing where currents meet, unsure which one will take you.
The letter lay open on the desk.
 Romé’s long fingers rested on the edge, rings sending faint glints into the dim. He bent toward it, candlelight rimming his profile in gold.
“Not city work,” he said, almost to himself. “Not trained in chancery… the ink’s oak gall, but the salt was local — west of Perugia.”
Deo’s eyes went to Dania. She poured wine into Romé’s glass with steady hands, not spilling a drop. She didn’t look at Deo, and I wondered if she knew how that steadiness cut into him.
Romé tilted the parchment toward the light, not reading the words but the rhythm of their making.
 “Two centuries past, a bishop would’ve signed like this only in Lent.”
Dania’s voice was quiet.
 “And the seal?”
Romé touched the wax with one fingertip — careful, almost reverent.
 “It’s right. Not perfect… but the flaw is the kind only time would make.”
Something in Deo’s posture changed. For a breath, I think he believed it. I think he believed her. The letter might have been real. Or perhaps it always had been.
A knock — light enough to be mistaken for the fire’s settling wood.
 Niccolò stepped in, hair wind-ruffled, arms cradling a bundle wrapped in monk’s cloth.
“I was passing the library,” he said, “and thought you might want these back before the rain starts.”
He set the bundle on the table: ledgers, missals, an herbarium with a cracked leather spine. As he shifted one aside, his gaze caught the letter. He didn’t touch it, but his eyes sharpened.
“That seal…” His voice was low, as if he’d stumbled upon an old face in the crowd. “San Bartolomeo’s hand. He used it after the plague year, when half his clergy were gone.”
Romé’s head lifted slightly. “You’ve seen it before?”
Niccolò’s mouth curved — not into a smile, but something smaller, private.
 “In the archives. On a grant to rebuild the olive terraces.” He tapped the wax’s chipped edge. “Even this break — it’s the same. He cracked it when he fell on the steps at Pentecost.”
Deo’s gaze darted between them — Romé’s surety, Niccolò’s certainty, Dania’s calm — as if trying to find the seam where invention ended and truth began. I could feel the question pressing in his mind: did she make this, or has it always been here, waiting?
Dania poured Niccolò mulled wine without asking, and he took it without looking at her. That was their way — an exchange always smooth, never named.
⸻
The letter was the keystone of Dania’s plan: an old bishop’s hand from two centuries past, speaking of a Christian marvel that had truly occurred in the chapel at Porta d’Argento — Aurelia’s capital, Deo’s city. It was the perfect document he needed; therefore he could not believe it. Therefore he had to.
⸻

 The candle had burned low enough that its halo was more smoke than flame. Shadows gathered in the folds of the healer’s dress, in the angles of Romé’s jaw, in the corners where the brazier’s heat could not reach.
Romé leaned back in his chair, the letter still under his hand, rings catching the light in brief, stubborn flashes. The cut of his jacket held him like armor, but his posture did not — there was an ease in it, almost careless, as if the candle were painting him from memory.
Dania moved to refill his cup, the lavender and mint of her passing settling in the air between them. She set the jug down without sound, her fingers brushing the stem of a glass the way a blade might test its own edge. Her gaze didn’t linger on Romé, nor on the letter — but on the table as a whole, as if the three of them were pieces placed with care.
Deo’s eyes moved from her to him, then back again. One was stillness in motion, the other motion stilled. The distance between them felt smaller than the desk should allow.
Romé’s fingertip traced the edge of the seal, his lashes low, his mouth curved in thought rather than in smile. Dania’s hand rested on the table’s edge, just shy of touching his sleeve.
Some part of Deo’s mind tried to keep count — of the glances, of the spaces, of the places where a breath shifted the air — but the numbers refused to add to anything simple. When Romé looked up at him, it wasn’t to invite or to challenge. It was simply to see if he was still there.
And Dania, without looking away from the desk, said quietly,
 “It will hold.”
The words were for the letter, perhaps. Or for something else entirely.
I could’ve sworn the candle leaned toward them.
 Deo on one side, Romé and Dania close enough that their shadows tangled on the desk. No words. Just the way a room changes when three currents meet and realize they’re not pulling apart.
Romé looked up first, his head bent over the letter, black hair falling to shadow his temple. The candle’s edge traced the line of his cheek, caught on the ring at his finger, then slid down to vanish in the folds of his sleeve.
 Dania followed, her lavender–mint warmth spilling into the lamplight — a freshness edged with a faint bitterness, like chrysanthemum leaves scrubbed between fingers in late autumn. It rose from the edge of her black dress, from where her skin breathed beneath the cloth. And higher still, the fall of fabric shaped the upper line of her hips — not shown, only implied, like a shoreline seen through mist, the place where balance waits before it tips.
And then it happened — no one agreed to it, but they all knew.
 Romé eased back an inch, leaving a space wide enough for Deo’s shoulder. Between them, the letter lay open — and Deo, still as carved stone, felt its margins closing in like the edges of a stage. Dania’s hip angled just so, the line of her dress brushing the edge of his jacket. A silent, joint invitation.
Deo stepped forward the way you test ice — slowly, as if listening for a crack.
 If he shifted left, his arm would brush Romé’s.
 If he leaned right, his hand would circle Dania’s wrist.
 Either would’ve been easy.
 Both were possible.
That’s when I noticed Niccolò — sitting sideways in the corner chair, legs folded, a ledger open on his knee as if he’d been reading for hours. But his eyes were above the page, following the space between the three of them the way a chess player studies a board. Not unkindly, not even curiously — just storing something away, like he did with every odd relic or overheard phrase.
They were watching Deo — not the way people watch a performance, but the way hunters watch the gap in a hedge. And Niccolò was watching all of them, as if the real game was being played in the air, not on the desk.
From my corner by the brazier, I saw it:
 the lines of Deo’s jaw cut clean in shadow, the cheek gone soft in light, his breath slowing like he’d walked into deeper water. His lips parted — not for words. And then he stayed right there, in that knife’s-edge distance, as if the space itself was the kiss.
Niccolò turned one page in his ledger without looking down. Then, his voice uncoiled, warm with a thread of mischief.
 “You know,” he said, as if they’d been speaking to him all along, “if it were mine, I’d keep it with the Terracina accounts. No one touches them. Dust thick enough to stop a mouse.”
He was in the chair nearest the brazier, though none of them had noticed him settle there. Turning the cup in his hands as if warming it — though the wine had long since cooled — he watched not the letter, not the people, but the small hinge-points of the room: the flicker of a shadow when someone shifted, the way Romé’s ring caught the light when his hand stilled.
He sipped. “Only time they open those ledgers is when the Bishop wants to see which donor’s great-grandfather planted which olives.”
Romé glanced up, the candle making a half-light in his eyes.
 “And you’d leave it there?”
Niccolò shrugged, a curve of the shoulder as lazy as smoke.
 “Better than leaving it where people are actually looking.”
Dania poured the last of the wine into his cup. “Good to know.”
Her voice was even, but something in the way she set the jug down told me she’d already tucked his remark into the plan.
 The candle hissed as a drop of wax fell, the sound small but sharp. No one moved to trim the wick.
The seal gleamed dully, the rings on Romé’s hand caught another brief flare of light, and the shadows leaned just a little longer toward the desk — as if the room itself were listening.



📖December 4, 1264 Thursday
📜 cacifer’s interPage about  Niccolò 🕯♛♡🐾The Bishop They Built
🐾 Cacifer’s private entry (paws off, unless you have whiskers)
From the outside, Niccolò is the sort of monk you could set your teacup on and trust it wouldn’t spill.
Scapular smooth, hair tidy, smile fixed exactly at the polite notch between “helpful” and “harmless.” He’s the one donors remember for his perfect bows and the way he never smudges ink on the wrong page.

But I have eyes. And so does Dania.
Inside that tidy head is a fox. A bored fox. The kind that will pace the same length of fence until it knows every splinter — and then look at you as if to say, I could be out of here in three heartbeats if I felt like it.
He was born with a mind that refuses to nap in the same corner twice. Give him a puzzle, he’ll solve it; give him the answer, he’ll just sniff at it and ask what else you’ve got.
He could have been anything — scholar, engineer, poet, physician — but that would require keeping something once he’s mastered it. And Niccolò sheds interests the way I shed fur in summer: all at once, and everywhere.
The library suits him because it’s all puzzles, all the time, and no one notices if you disappear into the dust for hours.
That’s where Dania found him.

Or maybe they found each other — two creatures built for different forests, meeting in the same patch of shade.
It started small:
“This book doesn’t belong on this shelf.”
“That margin note isn’t the same hand as the text.”
“These herbs aren’t from here.”
Little things most people let pass — but to them, they were pawprints in snow.

After that, he started bringing her “returns” from the library. Always wrapped in the monk’s cloth, as if they were formal deliveries — and inside, some oddment from the monastery’s corners: a glove too small for any of the brethren, a pressed violet with writing invisible unless you breathed on it, a wooden bead carved with a saint no one could name.
They would tilt their heads over each one like magpies in council: Whose? Why? Lost, or left behind on purpose?
And then came the bronze ring.
Old, greened with time, its crest just familiar enough for Niccolò to squint and say, “I’ve seen this before.”
Dania reached for sealing wax before he’d even finished speaking.
By nightfall, the room smelled of smoke and honey, and they were leaning over two pools of candlelight, pressing and re-pressing the ring into wax until the seal matched one from a bishop’s letter two centuries past.
From there, the bishop became their game.

They gave him flaws, favourites, secret lovers, a bad knee in winter. They tucked scraps of his “life” into the library like seeds: a folded note in a psalter, a date in a margin that no almanac could explain, a recipe for wine syrup “in the hand of His Grace.”
Never to fool anyone — only to make each other grin.
Their bishop was their shorthand.
A glance could mean remember when he married the widow from Siena in secret? or what if he tried astronomy and fell off the roof?
I’ve never seen them touch the way the world counts as touching.
But I’ve seen how they move around each other — the way you circle the one other creature who knows the language you were born speaking.
And if you’ve never been that child, you can’t know what it’s worth.

And I’ve seen the little girl in Dania who was never allowed to play.
Niccolò gives her that back. And she gives him something better than escape: a place where he actually wants to stay.
So when the seal turned up again — this time pressed into a letter aged so perfectly it could make a Bishop doubt his own memory — Niccolò didn’t question it.
Why would he?
He’d been helping make it real for years.
📖December 5, 1264  Friday
The Boy in the Sanction 🕯♛♡🐾 waning gibbious, late moon behind pale clouds


The knock came before the candle guttered.
A young monk in a rough winter cloak stood at the door, cheeks raw from cold. His voice carried the flat carefulness of one repeating another man’s words:
“Sister Dania — Brother Anselmo requests you at the infirmary. A boy’s been brought in from the south road. Urgent.”
Brother Anselmo — keeper of hospital accounts, guardian of the storeroom key, whose hands stayed cleaner than the floors he ordered scrubbed — had not come himself. He rarely did when risk was near.
Dania’s eyes narrowed almost imperceptibly. She rose, setting aside the jug without a clink.

Romé moved with her. His jacket still carried the faint musk of candle smoke as he fell into step beside her, silent as a shadow.


The infirmary smelled of damp linen and boiled sage.
On the central cot lay a boy no older than nine, skin the colour of wax, lips edged blue. His breath rasped, wet and shallow.
The parents hovered — the mother clutching her rosary until the beads cut her palm, the father’s jaw locked against the urge to demand a miracle.

Dania bent over him without change of breath.
Her hands knew what the others could not: the faint bitter-metal trace in his lungs, the weight of several substances pressing the body toward collapse. Whether by chance or careless hand, it was enough to break him.

Her face did not betray the thought.
Instead, she spoke clearly, her tone steady as stone:
“This will not cure him. It will give him strength enough for some days. If his body can endure, he may survive. If not, he will suffer less in passing.”

She showed the parents each measure — which powders to mix with broth, which leaves to steep, when to touch the drops to his lips. She placed bundles in their hands with the patience of one teaching letters, repeating until they could not mistake her meaning.

The mother’s mouth trembled; her eyes clung to Dania’s face as if words alone could change the verdict. She nodded, once, twice — the kind of nod meant to keep herself from breaking.

The father’s throat worked, jaw straining against words he could not allow to leave him. He swallowed, again and again, as though each stone lodged lower, heavier. His hand closed around the boy’s ankle beneath the blanket — not shaking, not stroking, only holding, as if the pulse beneath the bone might remember its duty.

Silence thickened in the room. Not refusal, not yet grief — the silence of those already rehearsing a death they cannot stop.
Dania did not move to soften it. She placed the final pouch in the mother’s palm, repeating the measure one last time, her voice clear, her hands steady. Nothing in her face bent toward pity, or away from truth.

Romé could not look away. The weight of their stillness dragged him inward, backward — to the nights his own arms had been the cot, his chest the prayer that never reached the heavens. Aureliano’s breath had rattled like this boy’s, slow collapse hidden in each inhale, and no bundle of powders or leaves had been set in his hands. Only the long night and the slow breaking.

His shoulders stiffened, then gave way, his lashes lowering as if they might shield him from memory.

The candle failed twice before it learned to burn true. Outside, fog crouched low on the terrace and made the bell tower look farther than it was. The moon rose late and pale, climbing through a seam in the clouds like a coin lifted from murky water. The night had the taste of something arranged — footsteps that didn’t echo, courtyards that blinked and pretended not to. I sat on the sill and watched Dania’s shadow move along the infirmary wall while Romé stood very still, listening to a past that wouldn’t stop breathing. Somewhere, a latch settled without a hand on it. “Fog,” I thought, and felt my fur lift along the spine. Not weather. Intention.

Cacifer’s Interpage: 🕯♛♡🐾Brother Anselmo, Polished Halo, Sticky Hands
Here is your cherub, little reader. Brother Anselmo. Keeper of keys, counter of linens, apostle of clean accounts. He smiles with all his teeth and none of his hunger. The novices call him “kind” because he remembers their names; the stewards call him “holy” because he remembers their errors.
He dislikes Dania the way wax dislikes flame: politely, publicly, and always in the name of order. He performs empathy the way a clerk performs sums — neat columns, no remainder. I have watched him “help” a mother by moving her to a bench and “comfort” a father by folding his hands for him. He adores suffering so long as it stays on schedule and answers to his ledger.
He did not bring the boy himself. Of course not. Anselmo sends others where breath is ragged and outcomes are rude. He arrives for the part where witnesses gather and gratitude can be tallied. Tonight’s message was word-perfect, cold from rehearsal. Brother Anselmo is a bell that rings to prove it is a bell.
Does he poison? No. That would stain his fingers. He introduces “procedures.” He tidies doors that should stand open. He “centralizes” mercy and then sells access to the key by the ounce of reputation. What he cannot own, he undermines. Dania will never belong to him; therefore he weighs her with rumor until she measures lighter.
Remember him. He is the kind that builds pyres out of tidy paperwork and calls the smoke incense.



📖December 6, 1264 Saturday
Fillipa’s invitation🕯♛♡🐾Valiponte, House Basili’s ledger chamber
The ledgers were not dusty. Filippa made sure of that.
Each morning before the light fully reached the inner shutters, she crossed the stone passage behind the main hall, unlocked the back room, and lit the low lamp. She swept the floor, wiped the leather bindings, refilled the inkwell, and lifted the covers of the heavy books just enough to let the pages breathe.
No one asked her to do it. But this was the room where her father’s breath lived longest. Where trade met time, and time met care.
Today, her hands moved smoothly, copying out weights and customs dues into the narrow margins of the Sicilian manifest. But her mind didn’t stay with the pages. It drifted, doubled—part of her still within the salt crates and trade logs, part of her circling elsewhere.
Deonardo.
The conversations with him hadn’t faded. If anything, they had deepened, like ink that darkens when it dries. She kept returning to them—not the words exactly, but the shape of them. The way he listened. The quietness around his questions. The strange gentleness in his smile.
She wasn’t used to feeling this seen—not by men. Not like this.
Not even Romé—her chest tightened slightly at the thought of him.
St. Martin’s Day. The lights, the wine, the firelit bridge. The first kiss. The parting she hadn’t expected.
She had thought—no, felt—that he wanted her. Not only as men sometimes want, but something softer, keener. And she had been ready—to step closer, to let her body learn, to finally feel what she had read of but never lived.
And then he left. Without explanation. Without even a look that said why.
Their names were everywhere in the house now, the last week’s refrain between her father and brothers: “Ser Romualdo di Orellana—Romé—has answered San Michele’s plea; he rides with the valley’s guards.” “Ser Deonardo di Vallescura was received by the council.” The chill had crept into the warehouse walls—the kind of cold that softened paper and left a faint dryness in the throat.
Then Deonardo came.
The door opened without flourish. Just a quiet creak, the soft click of boots on stone. He entered in his traveling coat, hair wind-swept, the silver clasp at his collar slightly crooked—as if he had dressed in haste, or hadn’t thought to check. His eyes found hers at once.
“Signorina Filippa,” he said with a nod, warm but subdued.
She rose. “Messer Deonardo.”
A flicker passed between them—not quite familiarity, not quite formality. She gestured to the bench opposite her work table. “Would you like to sit?”
“Only if I’m not interrupting.”
“You’re not.” She smiled, and the pulse of the room shifted. She poured tea from the warming pot, set it before him without asking. “You’ve spoken with my father?”
He nodded. “Yes. The council is… working. Your father is steady. They will settle the purse for the contracted men—tight, but sufficient.”
She noticed the way his shoulders carried tension today. A deeper sense. A further reach. She glanced at her ink-stained fingers, then lifted her eyes to meet his.
“You also carry a great deal with you,” she said. “And you still came here. To this room.”
He looked at her, and for the first time that afternoon, smiled. “I came because this room makes sense.”
The sentence fell gently, like a leaf finding its place. Filippa didn’t answer at first. She only watched him—the crooked clasp, the wind still caught in his hair—and something in her steadied.
She returned the smile, smaller but sincere. “Then you’re welcome here.”
He inclined his head. Quiet held for a moment, companionable as steam.
“And Romé?” she asked at last, softer. “Have you seen him?”
Deonardo’s gaze went to the window, where the light had begun to fade. “Last I saw, he was in the healer’s house, reading poems as if the snow were months away. Calm, as ever. He’ll be where he should be tomorrow.”
Filippa breathed out, a smile she couldn’t quite hold back touching her mouth. Something in her loosened—the place where the bridge and the kiss had knotted tight. She didn’t know why, only that the answer let her set that night down without breaking it.
They did not speak for a while. Just two presences in a chamber of ink and paper while the outside world rearranged its certainties. The lamp hummed. The ledgers breathed. And the cold on the stone stayed the kind of cold that keeps pages honest.

Filippa spoke again, slower this time, as if following a thread she’d been circling for years:
“May I ask you something… a little different?”
Deo nodded, his attention steady.
“What do you think really separates men and women?”
She didn’t say it sharply. She wasn’t pushing. Just… wondering aloud.
“My father lets me do almost anything I want. When I asked for tutors, he found them. When I wanted to learn the ledgers, he gave me the books and brought me to the meetings. He never stopped me from reading things that weren’t written for women.”
“But when my brothers came of age, he sent them on ships. Across seas. To trade, to learn the world firsthand. I stayed with letters. Paper. Words from far away.”
She paused, then added quietly:
“He trusts me. But he still sees a line — even if he never says it.”
Deo said nothing yet. He knew to wait.
“I don’t belong with the other girls anymore. The ones I grew up with. They’re married, some already mothers. When they talk… there are things they know that I don’t.”
A breath. Not quite regret. Just truth.
“But my father never pressed me to marry. He said I could choose.”
She turned to him more fully now.
“I heard you come from a land ruled by a woman. In her palace… in her city… how do women live their lives?”

He didn’t answer at once.
His eyes drifted to the window, where the thin daylight caught on the iron latch, casting it in gold. Then he looked back to her — not with a prepared speech, but as if he were reaching inside himself, drawing out something long kept quiet.
“It’s true,” he said softly. “The High Lady of Vercado — Aurelia di Vercado — holds power in her own right. She rules not through a husband, nor as a placeholder for a son. She governs with her own mind, her own name.”
He studied Filippa’s face a moment, then went on.
“In her court, there are women who command fleets. Women who speak five tongues. Women who write laws. And yes — some who are poets, midwives, spinners, seers.”
“But the difference isn’t what they do. It’s how no one says they cannot.”
His voice stayed quiet, but the weight in it deepened.
“Still, the palace is not the whole country. There are valleys like this one, too. Places where daughters are still weighed by what they bring in dowry. Where men speak of honor and mean control.”
He met her eyes fully now.
“But the heart of it? Where I come from… a woman’s mind is not assumed to be less. Her presence is not a lesser version of a man’s. And her path is not given to her — she takes it, and names it.”
Filippa’s fingers had stilled over a scrap of parchment. Her eyes didn’t move from his.
He continued, softer:
“You remind me of some of them. Not because of your boldness — but because you’re already walking your own path. Even if it’s not named yet. Even if others haven’t learned how to follow it.”
She blinked once, quickly. Then steadied.
“And do they ever walk alone? The women you speak of.”
Deonardo’s gaze gentled.
“Sometimes. But not always. The wisest among them… choose carefully whom to walk beside. And when they do — it is not to be owned. ”

Filippa didn’t speak at first.
She only looked down, her hand resting lightly on a folded linen. Then she asked — not with defiance, but with real curiosity,
as if a thought long coiled had finally surfaced.
“And between men and women… beyond marriage, beyond family or heirs — what can be truly shared?”
The room stilled.
Even the paper seemed to hush its rustle.
Deonardo didn’t smile. He didn’t deflect.
He listened — and then let the silence hold the weight before replying.
“Trust,” he said. “Not the kind built on roles. But the kind that comes when both know they can be altered by the other — and still remain themselves.”
He shifted slightly, fingers resting on the edge of the table.
“Joy, too — not only in touch, or in duty, but in being understood without translation.”
He paused. Then added, more quietly:
“Grief. Real grief, the kind that breaks open something in the soul. That, too, can be shared.”
Filippa was still. The light along her cheek had cooled to a faint glow.
“But can it last?” she asked. “Without marriage, without oaths? I mean… can such trust remain, if it’s not held by custom?”
Deo’s eyes didn’t look away.
“Sometimes no,” he said. “Sometimes the world around them tears it apart. But when it does last — it becomes something rarer than marriage. Not more holy. But less easily replaced.”
Filippa’s voice came smaller now, almost without breath:
“Have you known it?”
Deo didn’t answer immediately. He looked to the side — not evasive, just… tracing something private.
Then, softly:
“I have known the hunger for it.”
Filippa didn’t rush.
Her hand shifted slightly on the linen fold, smoothing it without needing to.
Then she looked up at him again — her green eyes steady, almost luminous in their quiet intensity.
“And how… how does such a sharing begin? Do people speak it first — or do bodies know it, before the words ever come?”
Deonardo’s expression changed — not startled, not stirred by desire, but something deeper.
A question rarely asked with such purity.
He spoke slowly.
“Sometimes the body knows before the mind can explain. A certain quietness between two people — not tension, not hunger, but a kind of… recognition. Like a bridge that was always there, waiting to be crossed.”
He glanced toward the window, where dusk was softening the stone.
“And sometimes,” he said, more quietly,
“it begins in the voice — a way of being heard, or seen, without defense. And then the body follows, because it feels safe enough to speak its own language.”
Filippa’s breath had deepened. She wasn’t flushed, but something had shifted in her stillness.
“I think… I once felt something like that,” she said. “But I didn’t know how to hold it. And maybe he didn’t either.”
Deo didn’t ask who. He didn’t need to.
“It’s not always the fault of those who step back,” he said gently. “Sometimes the bridge frightens us more than the crossing itself.”
Filippa’s voice was almost a whisper now:
“But the bridge doesn’t vanish… does it?”
Deo met her gaze again. There was no shield left in his.
“Not if it was real.”

There was a pause — not empty, but filled with the kind of silence that settles only between people who are truly present. Filippa held it carefully, as if listening to something inside herself. Her fingers brushed the rim of the empty teacup.
Then, still not looking directly at him, she asked — her voice low, careful, and utterly honest:
“May I ask something strange?”
Deonardo didn’t answer with words. He only inclined his head, a gesture that said: yes, ask it.
She turned her gaze to him — not timid, but searching.
“Do you know… what joy feels like, in a woman’s body?”
Her breath caught just slightly.
She didn’t blush — it was something deeper. A question from the place where the skin and the soul are not yet separate.
Deo’s eyes didn’t flinch. Whatever flickered through him — memory, sorrow, tenderness — it passed behind the soft stillness of his face. He answered, with great care:
“I have known it…more than a man.”
The words hung there.
Filippa didn’t fully understand. But something in her did. Her body understood.
She felt a trembling calm, a deep permission.
Deo continued, gently:
“It is not the same in every woman. Not even in the same woman, each time. But when it comes — it is not about someone else giving or taking. It is a river opening inside. A knowing. That you are alive, that nothing in you is wrong.”
Filippa closed her eyes for a second, and when she opened them, they were misted — not from sadness, but from some inner thread that had loosened.
“I think I wanted to feel that. Once. But he left before it began. I thought maybe… it was my mistake. Or that I asked for too much without knowing what I was asking.”
Deonardo’s voice was low, almost a murmur.
“You asked for the right thing. He just wasn’t ready to meet you there.”
And then, without touching her, he offered the most delicate reassurance:
“The joy you carry — it is not gone. It waits. It remembers you.”

Filippa didn’t speak right away.
But something in her posture changed — not visibly, perhaps, not to anyone who didn’t know her — yet unmistakably.
Inside her, something had settled.

She had spent years straddling spaces:
— the rooms of men’s trade and speech,
— the quiet circles of women who bled and bore,
— her father’s ledgers, her friends’ whispered stories,
— and her own silent, aching body that didn’t quite belong in either.
And now, across from this man — elegant, wounded, watching — she heard someone speak into her uncertainty, not over it.
That changed her.
She felt something shift in her chest — a warmth, almost unbearable in its gentleness.
And in that moment, Filippa felt both older and younger than her nineteen years.
She felt like a girl who had waited too long for permission.
And like a woman who had just given it to herself.
She wanted to stay in this room.
She wanted to ask more,  to walk a little further into the unknown — not alone.

The silence between them had stretched — not tense, but dense. Like a held breath neither of them knew how to release.
The tea had long gone cool.
Outside, the sun slipped slightly westward, casting the shelves of ledgers into longer shadow. One moth circled near the lintel beam, unhurried.
Filippa hadn’t moved much. Only once did she reach to tuck a loose strand of hair behind her ear — a small motion, but one that made her aware of her own fingers. Her skin. The curve of her own wrist.
Something in her felt tenderly lit, and she didn’t know how to name it.
Deonardo had remained quiet — his body still, but not withdrawn. His presence was a steadying one, even in silence. It gave her room.
And maybe it was that room — that rare room — that allowed the question to rise. A question that wasn’t born just today, or last week, but years ago, when the others began to speak in low voices about their bodies and their lovers. About blood and heat, about pleasure and fear. About what it meant to give, to receive, to feel — in the places where language turned shy.
She had always listened, but never entered.
Now, for the first time, she wished not to listen, but to ask.
Her voice came quiet — quieter than anything she had said before.
“Messere Deonardo…”
He looked up at once — not startled, not expectant. Just present.
She held his gaze a moment, then lowered hers, almost in reverence to the words forming.
“If a woman has never… known the joy of her own body… but begins to long for it — not from books, not from stories, but because she is ready — can a man… can you… ever bring her to that place gently?”
The silence that followed was different now. A silence of absolute attention.
Not tension. Not shame.
Deo did not speak right away. He didn’t smile, or avert his gaze. He remained still — as if her question was a fragile, living thing placed in his hands.
Then, after a long breath:
“Yes,” he said. Not hurried. Not heavy. Just the truth.
“If he listens. If he waits. If he sees her — not what he wants her to become, but who she already is.”
He paused, not for effect, but for care.
“And only if she is not asking to be undone, but to be accompanied — to the place she already holds within herself.”
Filippa’s eyes lifted slowly to meet his, to hold his gaze.
Then, with a breath that almost trembled, she asked:
“Would you answer me… without speaking?”
The words hovered, soft as smoke, but the weight behind them was undeniable.
Deonardo did not blink.
For a long, living moment, he did nothing.
Then he shifted — only slightly — as if the world had tilted on a hidden hinge. He did not move toward her, not fully, but the air between them changed. Something attuned, quiet, and without claim.
He reached — slowly, with care that came not from hesitancy, but from reverence — and turned his palm upward, resting it gently on the wood between them. 
Filippa did not rush. Her eyes flicked briefly to his hand.
She placed hers atop it — not grasping, not entwining. Just contact. Skin to skin.
There was warmth.
Not heat.
A warmth that rose from the place between desire and recognition — of being met, not moved.
His thumb, by instinct or care, traced the curve of her knuckle once. A gesture more tender than intimate. It said: I heard. I understand. I am here.
Filippa’s breath deepened.

Her hand rested lightly on his. But her body leaned forward now, slightly — not enough to cross propriety, but enough to shift the weight of breath between them.
He saw it.
Noticed it as a man who had learned to listen where words fall short.
The air thickened. Pressed. 
Filippa’s voice came low.
Not uncertain — just barely allowed.
“I don’t want to wonder,” she said.
“Would you… come closer?”
Deonardo didn’t speak.
But he did.
He rose — slowly — and stepped to her side, not looming, not consuming. He waited for her eyes to lift.
They did.
In that gaze, the whole room dropped away — the ledgers, the ink, the news from the valley. Her eyes widened with the sudden ease of recognition. This was not a stranger. This was someone whose presence felt… correct.
He placed one hand on the back of her chair — not holding, just grounding himself.
And then, when she didn’t move away —
His other hand rose to her cheek.
Only fingertips.
Only breath.
But the silence between them thickened to the point of almost breaking.
Filippa’s hands found his coat, near the collar. She didn’t tug — only steadied herself there, her fingers clutching not to pull him in, but to keep herself from floating away.
He leaned forward.
Slow. Reverent.
Their foreheads touched first.
Not lips.
Not yet.
Her eyes closed. His breath trembled once — just once — across her skin.
Then finally, he pressed his mouth to hers, to let her know that this — this body, this timing, this way she opened — was real, and that he was with her, not over her.
She let the kiss deepen.
Just enough for the air to disappear between them.
Just enough for her to know:
She would remember this moment.
Not for what they did.
But for what it allowed.

Ledger Chamber, Valiponte 🕯♛♡🐾, after the kiss
Her breath had quieted. Her chest no longer rose sharply — it unfolded, like a sail catching the right wind.
Deo hadn’t moved back yet. His fingers still at her jaw, his presence steady.
She opened her eyes.
What she said next didn’t sound like a question.
“Would you teach me?”
He tilted his head slightly. She continued — not rushed, not shy, only clear:
“I want to know the joy of a woman’s body.
I want to feel what belongs to me.
Not just through books or guessing.”
Her green eyes were calm, but glinted with something he recognized — not innocence, but a quiet right to choose.
“I would want to trust this… with you.
Only if you receive it as a gift.”
There was a silence, as long as a held breath.
Then she added, gently:
“ Basili’s guest quarters is above us, I am the only one who charges the keys. 
If you wish… I would invite you there.”

Deonardo didn’t move right away.
He stood before her, quiet as light shifting across old ink. His gaze didn’t falter — but what returned to her now was not just tenderness. It was consideration. Gravity. A pause that honored her completely.
His voice came low, but certain:
“Filippa.”
The way he said her name — it was not to distance her. It was to hold the moment still.
“You have asked something brave. Something beautiful.
And I will not answer it lightly.”
He reached for her hand. Didn’t squeeze it. Just held it, his thumb across her pulse.
“Desire is not a question I fear. Nor your body’s joy.”
A breath passed between them.
“But joy that begins without full clarity… can still carry loneliness inside.”
Filippa didn’t flinch. She listened. He felt her listening.
He continued:
“If you still wish this two days from now…
If your body still says yes — not from ache or question,
but from peace — I will come.”
He met her eyes fully.
“And if your yes changes shape —
I will still be here.
Nothing in me will think less of you.”
Something in her breath shifted again. Not withdrawn — just deeply seen.
She nodded — not with disappointment, but with understanding. It wasn’t refusal. It was the rarest kind of acceptance: the kind that believes in her decision, enough to let it arrive whole.
He lifted her hand and kissed it — not as a courtly gesture, but as a seal. Then, gently, he stepped back.
“Two days,” he said.
And then he left.
The door shut softly behind him — and the scent warmed wood lingered in his wake.
Filippa stood still for a long time.
📖December 7, 1264 Sunday

"""

# 2. Lowercase + split into tokens
words = text.lower().split()

# 3. Build sliding 5‑word windows (4 in, 1 out)
sentences = []
for i in range(5, len(words)):
    sentences.append(' '.join(words[i-5:i]))

# 4. Tokenize
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)
seqs = tokenizer.texts_to_sequences(sentences)

vocab_size = len(tokenizer.word_index) + 1

# 5. Keep only exact length‑5 sequences (punctuation filtering can shorten some)
seqs = [s for s in seqs if len(s) == 5]
seqs = np.array(seqs)

X = seqs[:, :4]                # first 4 words = context
y = seqs[:, 4]                 # 5th word = label
y = to_categorical(y, num_classes=vocab_size)

X.shape, y.shape, vocab_size


2026-05-19 21:36:17.569009: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779226577.870858      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779226577.970203      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779226578.710488      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779226578.710571      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779226578.710578      16 computation_placer.cc:177] computation placer alr

((74636, 4), (74636, 7359), 7359)

In [2]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=64, input_length=4))
model.add(LSTM(128))
model.add(Dense(128, activation='relu'))
model.add(Dense(vocab_size, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

history = model.fit(X, y, epochs=15, batch_size=128, verbose=1)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
2026-05-19 21:36:50.163238: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
584/584 ━━━━━━━━━━━━━━━━━━━━ 23s 35ms/step - accuracy: 0.0667 - loss: 7.3706
Epoch 2/15
584/584 ━━━━━━━━━━━━━━━━━━━━ 20s 35ms/step - accuracy: 0.0672 - loss: 6.6218
Epoch 3/15
584/584 ━━━━━━━━━━━━━━━━━━━━ 20s 35ms/step - accuracy: 0.0738 - loss: 6.4299
Epoch 4/15
584/584 ━━━━━━━━━━━━━━━━━━━━ 19s 33ms/step - accuracy: 0.0836 - loss: 6.1824
Epoch 5/15
584/584 ━━━━━━━━━━━━━━━━━━━━ 22s 35ms/step - accuracy: 0.0991 - loss: 5.9321
Epoch 6/15
584/584 ━━━━━━━━━━━━━━━━━━━━ 20s 35ms/step - accuracy: 0.1023 - loss: 5.7476
Epoch 7/15
584/584 ━━━━━━━━━━━━━━━━━━━━ 21s 35ms/step - accuracy: 0.1109 - loss: 5.5626
Epoch 8/15
584/584 ━━━━━━━━━━━━━━━━━━━━ 21s 35ms/step - accuracy: 0.1217 - loss: 5.3548
Epoch 9/15
584/584 ━━━━━━━━━━━━━━━━━━━━ 20s 35ms/step - accuracy: 0.1295 - loss: 5.1420
Epoch 10/15
584/584 ━━━━━━━━━━━━━━━━━━━━ 21s 35ms/step - accuracy: 0.1424 - loss: 4.9438
Epoch 11/15
584/584 ━━━━━━━━━━━━━━━━━━━━ 20s 35ms/step - accuracy: 0.1524 - loss: 4.7311
Epoch 12/15
584/584 ━━━━━━━━━━

In [3]:
def sample_with_temperature(probs, temperature=0.8):
    probs = np.asarray(probs).astype("float64")
    probs = np.log(probs + 1e-8) / temperature
    probs = np.exp(probs)
    probs /= probs.sum()
    return np.random.choice(len(probs), p=probs)

def generate_cacifer(seed_text, num_words=30, temperature=0.8):
    words = seed_text.lower().split()
    if len(words) < 4:
        raise ValueError("Seed text must have at least 4 words.")
    words = words[-4:]

    for _ in range(num_words):
        context = ' '.join(words[-4:])
        seq = tokenizer.texts_to_sequences([context])
        if not seq or len(seq[0]) != 4:   # unknown words / too short
            break
        seq = np.array(seq)
        preds = model.predict(seq, verbose=0)[0]
        next_id = sample_with_temperature(preds, temperature)
        next_word = tokenizer.index_word.get(next_id, None)
        if not next_word:
            break
        words.append(next_word)
    return ' '.join(words)

print(generate_cacifer("i am cacifer a cat", num_words=40, temperature=0.8))


am cacifer a cat caught he didn’t smile whatever he said he knew his body dress at the books she matched them again she poured all many mail weighed their fear only voices high gulls gold and men could not be theirs here if


In [4]:
import numpy as np

def sample_with_temperature(probs, temperature=0.8):
    probs = np.asarray(probs).astype("float64")
    probs = np.log(probs + 1e-8) / temperature
    probs = np.exp(probs)
    probs /= probs.sum()
    return np.random.choice(len(probs), p=probs)

def generate_cacifer(seed_text, num_words=40, temperature=0.8):
    words = seed_text.lower().split()
    if len(words) < 4:
        raise ValueError("Seed text must have at least 4 words.")
    words = words[-4:]
    for _ in range(num_words):
        context = ' '.join(words[-4:])
        seq = tokenizer.texts_to_sequences([context])
        if not seq or len(seq[0]) != 4:
            break
        seq = np.array(seq)
        preds = model.predict(seq, verbose=0)[0]
        next_id = sample_with_temperature(preds, temperature)
        next_word = tokenizer.index_word.get(next_id, None)
        if not next_word:
            break
        words.append(next_word)
    return ' '.join(words)

print(generate_cacifer("i am cacifer a cat", num_words=40, temperature=0.8))


am cacifer a cat “you work and he feels—” her silence before her father’s eyes quick cut into an wool gets he ate to a lamp be for the undersides of divided moon sacks inside it paused the light with a game but something


In [5]:
   print(generate_cacifer("i am cacifer a cat", num_words=40, temperature=0.6))


am cacifer a cat alive as a clerk that to yield contracts a man that refuses blades are a place why he stood because the stillness of the nave and some as if the words had been mine at me he answered a step


In [6]:
def generate_bursts(seed_text, num_words=20, temperature=0.7, n_samples=10):
    bursts = []
    for i in range(n_samples):
        text = generate_cacifer(seed_text, num_words=num_words, temperature=temperature)
        bursts.append(text)
    return bursts

# Example usage:
for i, b in enumerate(generate_bursts("i am cacifer a cat", num_words=18, temperature=0.65, n_samples=5), 1):
    print(f"{i}. {b}\n")


1. am cacifer a cat heaven had come still suddenly yes only undeniable every month a vercado and honored nutshell steps floated at

2. am cacifer a cat of a hand that had not spoken but foreplay for the seal terracotta i could see the road

3. am cacifer a cat in her face was a little of honor the first man’s for the bishop’s house where each age

4. am cacifer a cat in the table and a single place between passage and let it take the bench at the wall

5. am cacifer a cat even the words leaned forward the way a man had already not spoken before her own hand —



# These are lovely seeds; a few of them are very close to strong lines with just a little hand‑editing.
From (1)

Original: am cacifer a cat the light filled on her hair trembled tight and warmed catching on the road holds the priest had
Polished:
I am Cacifer, a cat; the light filled her hair, trembling tight and warm, catching on the road where the priest once held his ground.

From (2)

Original: am cacifer a cat the space drew tensile grace — the older square and the night garden he took the library road
Polished:
I am Cacifer, a cat; the space drew its own tensile grace between the old square and the night garden, while he took the library road alone.

From (5)

Original: am cacifer a cat or look back but it was at the main cold they carried them drawn into the treatment room
Polished:
I am Cacifer, a cat; none of them looked back, but the main cold of the night clung to their shoulders as they were drawn into the treatment room.